In [10]:
%pip install pytesseract
%pip install pymupdf opencv-python matplotlib numpy pandas
%pip install camelot-py

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\lenovo\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\lenovo\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\lenovo\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


# Generic scientific heatmap extraction pipeline

This notebook is a separate extension of `module1_test-Copy1 (1).ipynb`. The original notebook is not modified.

The generic path adds four capabilities while retaining the existing six-stage pipeline, Qwen audit logs, schema validation, and manual-review rules:

1. Reconstruct complete figures from rendered PDF pages, including vector and multi-object figures.
2. Generate heatmap candidates with an ensemble of grid-line, colour-region, edge-density, and optional full-figure Qwen localization detectors.
3. Estimate matrix shape from either explicit grid lines or repeated colour transitions.
4. Preserve visual cell measurements for colour-only heatmaps without inventing numerical values.

Set `ACTIVE_PAPER`, add the PDF to `PAPER_PROFILES`, and run from the top. By default, the generic version scans every PDF page. Set `GENERIC_HEATMAP_CONFIG["scan_all_pages"] = False` to use the profile's `main_pages` list.


In [11]:
from pathlib import Path
import pytesseract

tess_path = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("exists:", Path(tess_path).exists())

pytesseract.pytesseract.tesseract_cmd = tess_path
print("version:", pytesseract.get_tesseract_version())

TESS_AVAILABLE = True
print("OCR available:", TESS_AVAILABLE)

exists: True
version: 5.4.0.20240606
OCR available: True


In [12]:
%pip install openai
from openai import OpenAI
import os
import json
import re
import base64
from pathlib import Path

def build_client():
    return OpenAI(
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )

def image_to_data_url(image_path):
    ext = Path(image_path).suffix.lower().replace(".", "")
    if ext == "jpg":
        ext = "jpeg"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/{ext};base64,{b64}"

def extract_first_json(text):
    if isinstance(text, dict):
        return text
    if text is None:
        raise ValueError("AI returned empty text")

    text = str(text).strip()

    try:
        return json.loads(text)
    except:
        pass

    text2 = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
    text2 = re.sub(r"```$", "", text2).strip()
    try:
        return json.loads(text2)
    except:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end+1])

    raise ValueError(f"Cannot parse JSON from AI response: {text[:500]}")

def vl_chat_json(image_path, prompt, model_name="qwen3.6-plus"):
    client = build_client()
    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_data_url(image_path)
                        }
                    }
                ]
            }
        ],
        temperature=0
    )
    raw = resp.choices[0].message.content
    return extract_first_json(raw)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\lenovo\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [13]:
import inspect
print(inspect.getsource(vl_chat_json))

def vl_chat_json(image_path, prompt, model_name="qwen3.6-plus"):
    client = build_client()
    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_data_url(image_path)
                        }
                    }
                ]
            }
        ],
        temperature=0
    )
    raw = resp.choices[0].message.content
    return extract_first_json(raw)



In [14]:
# Security: do not save API keys inside the notebook.
import os
import getpass

if not os.getenv("DASHSCOPE_API_KEY"):
    os.environ["DASHSCOPE_API_KEY"] = getpass.getpass(
        "Paste DASHSCOPE_API_KEY for this session: "
    )
print("DASHSCOPE_API_KEY configured:", bool(os.getenv("DASHSCOPE_API_KEY")))


DASHSCOPE_API_KEY configured: True


In [15]:
import os
import json
from pathlib import Path

import cv2
import numpy as np
import fitz
from openai import OpenAI
from PIL import Image, ImageDraw
import pandas as pd


## Multi-publication configuration

Edit only `ACTIVE_PAPER` and the selected profile below. All stage output folders are isolated under `runs/<paper_id>/`, which prevents one paper from overwriting another. `source_data` is optional and is used when the heatmap does not print every numeric value.


In [16]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()

PAPER_PROFILES = {
    "fan_2024": {
        "paper_id": "fan_2024",
        "main_pdf": "paper.pdf",
        "main_pages": [3, 4],
        "supplementary_source": "sciadv.adn9961_sm.pdf",
        "supplementary_table_pages": [17, 18, 19],
        "source_data": None,
        "figure_unit_filter": None,
        "target_assay": "ELISpot",
    },
    "fan_2024_supplement": {
        "paper_id": "fan_2024_supplement",
        "main_pdf": "sciadv.adn9961_sm.pdf",
        "main_pages": [],
        "supplementary_source": None,
        "supplementary_table_pages": [],
        "source_data": None,
        "figure_unit_filter": None,
        "target_assay": "ELISpot",
    },
    "powlson_2019": {
        "paper_id": "powlson_2019",
        "main_pdf": "papers/powlson_2019.pdf",
        "main_pages": [4],
        "supplementary_source": "papers/powlson_2019_table_s2.xlsx",
        "supplementary_table_pages": [],
        "source_data": None,
        "figure_unit_filter": None,
        "target_assay": "ELISpot",
    },
    "braun_2025": {
        "paper_id": "braun_2025",
        "main_pdf": "papers/braun_2025_figures.pdf",
        "main_pages": [4],
        "supplementary_source": None,
        "supplementary_table_pages": [],
        "source_data": None,
        "figure_unit_filter": None,
        "target_assay": "ELISpot",
    },
    "lindner_2025": {
        "paper_id": "lindner_2025",
        "main_pdf": "papers/lindner_2025_espec_suit_figures.pdf",
        "main_pages": [],
        "supplementary_source": None,
        "supplementary_table_pages": [],
        "source_data": None,
        "figure_unit_filter": None,
        "target_assay": "ELISpot",
    },
}

ACTIVE_PAPER = os.getenv("ACTIVE_PAPER", "fan_2024").strip()
PAPER_CONFIG = PAPER_PROFILES[ACTIVE_PAPER].copy()
RUN_ROOT = PROJECT_ROOT / "runs" / PAPER_CONFIG["paper_id"]
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def cfg_path(key, required=False):
    value = PAPER_CONFIG.get(key)
    if value in (None, ""):
        if required:
            raise ValueError(f"PAPER_CONFIG[{key!r}] is required")
        return None
    p = Path(value)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if required and not p.exists():
        raise FileNotFoundError(f"Missing {key}: {p}")
    return p

MAIN_PDF_PATH = cfg_path("main_pdf")
SUPPLEMENTARY_SOURCE_PATH = cfg_path("supplementary_source")
SOURCE_DATA_PATH = cfg_path("source_data")

print("Active paper:", PAPER_CONFIG["paper_id"])
print("Main PDF:", MAIN_PDF_PATH)
print("Supplementary source:", SUPPLEMENTARY_SOURCE_PATH)
print("Source data:", SOURCE_DATA_PATH or "Not configured")
print("Run folder:", RUN_ROOT)


Active paper: fan_2024
Main PDF: C:\Users\lenovo\Desktop\my_langchain_project\paper.pdf
Supplementary source: C:\Users\lenovo\Desktop\my_langchain_project\sciadv.adn9961_sm.pdf
Source data: Not configured
Run folder: C:\Users\lenovo\Desktop\my_langchain_project\runs\fan_2024


In [26]:

# ============================================================
# Reviewer Comment 6 — reproducible Qwen audit metadata
# This cell does not call Qwen. It centralizes prompt versions,
# schemas, retry policy, prompt manifests, and per-call logs.
# ============================================================
import hashlib
from datetime import datetime, timezone

AI_AUDIT_CONFIG = {
    "temperature": 0.0,
    "top_p": 0.01,
    "max_retries": 1,  # one initial call + at most one retry
    "uses_numeric_confidence_threshold": False,
    "confidence_semantics": "high/medium/low are model_self_reported_confidence only",
}

PROMPT_VERSIONS = {
    "stage3_heatmap_candidate_validation": "stage3_heatmap_candidate_v1.1",
    "stage3b_panel_recovery": "stage3b_panel_recovery_v1.0",
    "stage3c_legend_field_extraction": "stage3c_legend_fields_v1.0",
    "stage4_core_matrix_fixed_shape": "stage4_core_matrix_fixed_shape_v1.0",
    "stage4_context_axis_fixed_shape": "stage4_context_axis_fixed_shape_v1.0",
    "generic_qwen_json": "generic_qwen_json_v1.0",
}

AI_TASK_SCHEMAS = {
    "stage3_heatmap_candidate_validation": {
        "type": "object",
        "required": [
            "core_is_numeric_matrix_or_heatmap_body", "core_has_grid_cells",
            "core_has_cell_values_or_intensity", "context_has_interpretable_labels",
            "context_has_elispot_or_assay_text", "heatmap_is_only_edge_fragment",
            "main_core_content_is_non_heatmap", "is_suitable_for_stage4_extraction",
            "detected_assay", "detected_panel_id", "visible_row_labels_from_context",
            "visible_column_labels_from_context", "confidence", "manual_review", "reason"
        ],
    },
    "stage3b_panel_recovery": {
        "type": "object",
        "required": ["figure_unit_id", "expected_elispot_panels", "candidate_assignments", "overall_manual_review", "overall_review_reason"],
    },
    "stage3c_legend_field_extraction": {
        "type": "object",
        "required": [
            "Legend_Readout", "Legend_Sample_or_Tissue", "Legend_Treatment", "Legend_Dose",
            "Legend_Route", "Legend_N_Mice", "Legend_Mouse_Tumor_Model", "Legend_Mouse_Strain",
            "Legend_Vaccine_or_Antigen_Set", "Legend_ELISpot_Unit", "Legend_Manual_Review",
            "Legend_Review_Reason", "Legend_Field_Sources", "Legend_Field_Evidence"
        ],
    },
    "stage4_core_matrix_fixed_shape": {
        "type": "object",
        "required": ["row_count", "column_count", "matrix", "records", "manual_review", "review_reason"],
        "shape_constraints": [
            "row_count == tool-fixed rows", "column_count == tool-fixed columns",
            "matrix dimensions == rows x columns", "len(records) == rows x columns",
            "row_index/col_index in bounds and unique"
        ],
    },
    "stage4_context_axis_fixed_shape": {
        "type": "object",
        "required": ["tool_row_count", "tool_column_count", "x_axis_values_left_to_right", "y_axis_values_top_to_bottom", "manual_review", "review_reason"],
        "shape_constraints": ["len(x_axis_values_left_to_right) == columns", "len(y_axis_values_top_to_bottom) == rows"],
    },
    "generic_qwen_json": {"type": "object", "required": []},
}

AI_AUDIT_ROOT = Path(RUN_ROOT) / "ai_audit"
AI_AUDIT_RAW_DIR = AI_AUDIT_ROOT / "raw_responses"
AI_AUDIT_JSON_DIR = AI_AUDIT_ROOT / "parsed_json"
PROMPT_MANIFEST_PATH = AI_AUDIT_ROOT / "prompt_manifest.json"
AI_INFERENCE_LOG_PATH = AI_AUDIT_ROOT / "ai_inference_log.jsonl"
for _p in [AI_AUDIT_ROOT, AI_AUDIT_RAW_DIR, AI_AUDIT_JSON_DIR]:
    _p.mkdir(parents=True, exist_ok=True)
if not PROMPT_MANIFEST_PATH.exists():
    PROMPT_MANIFEST_PATH.write_text(json.dumps({
        "metadata": {
            "uses_numeric_confidence_threshold": False,
            "temperature": AI_AUDIT_CONFIG["temperature"],
            "top_p": AI_AUDIT_CONFIG["top_p"],
            "max_retries": AI_AUDIT_CONFIG["max_retries"],
        },
        "prompts": []
    }, ensure_ascii=False, indent=2), encoding="utf-8")
AI_INFERENCE_LOG_PATH.touch(exist_ok=True)

def _utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def _audit_safe_name(value):
    value = str(value or "unknown")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("_") or "unknown"

def _extract_json_for_audit(text):
    if isinstance(text, dict):
        return text
    cleaned = str(text or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.I)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except Exception:
        pass
    decoder = json.JSONDecoder()
    for start in [i for i, ch in enumerate(cleaned) if ch == "{"]:
        try:
            obj, _ = decoder.raw_decode(cleaned[start:])
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    raise ValueError("No valid JSON object found in model response")

def validate_basic_json_schema(parsed, schema):
    errors = []
    if not isinstance(parsed, dict):
        return False, ["response is not a JSON object"]
    for key in schema.get("required", []):
        if key not in parsed:
            errors.append(f"missing required field: {key}")
    return len(errors) == 0, errors

def validate_stage4_core_raw(parsed, rows, cols):
    valid, errors = validate_basic_json_schema(parsed, AI_TASK_SCHEMAS["stage4_core_matrix_fixed_shape"])
    if not isinstance(parsed, dict):
        return False, errors
    try:
        if int(parsed.get("row_count")) != int(rows):
            errors.append(f"AI row_count={parsed.get('row_count')} != tool rows={rows}")
    except Exception:
        errors.append("row_count is missing or not an integer")
    try:
        if int(parsed.get("column_count")) != int(cols):
            errors.append(f"AI column_count={parsed.get('column_count')} != tool columns={cols}")
    except Exception:
        errors.append("column_count is missing or not an integer")
    matrix = parsed.get("matrix")
    if not isinstance(matrix, list) or len(matrix) != int(rows):
        errors.append(f"matrix row dimension must equal {rows}")
    elif any(not isinstance(row, list) or len(row) != int(cols) for row in matrix):
        errors.append(f"every matrix row must contain exactly {cols} columns")
    records = parsed.get("records")
    if not isinstance(records, list):
        errors.append("records is not a list")
    else:
        if len(records) != int(rows) * int(cols):
            errors.append(f"record count={len(records)} != rows*columns={int(rows)*int(cols)}")
        seen = set()
        for idx, rec in enumerate(records):
            if not isinstance(rec, dict):
                errors.append(f"record {idx} is not an object")
                continue
            try:
                ri, ci = int(rec.get("row_index")), int(rec.get("col_index"))
            except Exception:
                errors.append(f"record {idx} has non-integer row/col index")
                continue
            if not (0 <= ri < int(rows)) or not (0 <= ci < int(cols)):
                errors.append(f"record {idx} index ({ri},{ci}) is out of bounds")
            if (ri, ci) in seen:
                errors.append(f"duplicate record index ({ri},{ci})")
            seen.add((ri, ci))
        expected = {(r, c) for r in range(int(rows)) for c in range(int(cols))}
        missing = sorted(expected - seen)
        if missing:
            errors.append(f"missing record indices: {missing[:20]}")
    return len(errors) == 0, errors

def validate_stage4_axis_raw(parsed, rows, cols):
    valid, errors = validate_basic_json_schema(parsed, AI_TASK_SCHEMAS["stage4_context_axis_fixed_shape"])
    if not isinstance(parsed, dict):
        return False, errors
    try:
        if int(parsed.get("tool_row_count")) != int(rows):
            errors.append(f"tool_row_count={parsed.get('tool_row_count')} != {rows}")
    except Exception:
        errors.append("tool_row_count is missing or not an integer")
    try:
        if int(parsed.get("tool_column_count")) != int(cols):
            errors.append(f"tool_column_count={parsed.get('tool_column_count')} != {cols}")
    except Exception:
        errors.append("tool_column_count is missing or not an integer")
    x_values = parsed.get("x_axis_values_left_to_right")
    y_values = parsed.get("y_axis_values_top_to_bottom")
    if not isinstance(x_values, list) or len(x_values) != int(cols):
        errors.append(f"X label count must equal columns ({cols})")
    if not isinstance(y_values, list) or len(y_values) != int(rows):
        errors.append(f"Y label count must equal rows ({rows})")
    return len(errors) == 0, errors

def validate_stage4_normalized_records(records_df, rows, cols):
    errors = []
    if records_df is None or len(records_df) != int(rows) * int(cols):
        errors.append(f"normalized record count must equal {int(rows)*int(cols)}")
        return False, errors
    seen = set()
    for _, rec in records_df.iterrows():
        try:
            ri, ci = int(rec["Row_Index"]), int(rec["Col_Index"])
        except Exception:
            errors.append("normalized row/column index is missing or invalid")
            continue
        if not (0 <= ri < int(rows)) or not (0 <= ci < int(cols)):
            errors.append(f"normalized index ({ri},{ci}) is out of bounds")
        if (ri, ci) in seen:
            errors.append(f"duplicate normalized index ({ri},{ci})")
        seen.add((ri, ci))
    expected = {(r, c) for r in range(int(rows)) for c in range(int(cols))}
    if expected != seen:
        errors.append("normalized grid does not contain every tool-fixed cell exactly once")
    return len(errors) == 0, errors

def validate_stage4_normalized_axis(axis_json, rows, cols):
    errors = []
    x_values = axis_json.get("x_axis_values_left_to_right") if isinstance(axis_json, dict) else None
    y_values = axis_json.get("y_axis_values_top_to_bottom") if isinstance(axis_json, dict) else None
    if not isinstance(x_values, list) or len(x_values) != int(cols):
        errors.append(f"normalized X label count must equal {cols}")
    if not isinstance(y_values, list) or len(y_values) != int(rows):
        errors.append(f"normalized Y label count must equal {rows}")
    return len(errors) == 0, errors

def _write_prompt_manifest_entry(task_name, prompt_version, prompt, model, expected_schema):
    manifest = {
        "metadata": {
            "uses_numeric_confidence_threshold": False,
            "temperature": AI_AUDIT_CONFIG["temperature"],
            "top_p": AI_AUDIT_CONFIG["top_p"],
            "max_retries": AI_AUDIT_CONFIG["max_retries"],
        },
        "prompts": [],
    }
    if PROMPT_MANIFEST_PATH.exists():
        try:
            loaded = json.loads(PROMPT_MANIFEST_PATH.read_text(encoding="utf-8"))
            if isinstance(loaded, dict):
                manifest.update(loaded)
                manifest.setdefault("prompts", [])
        except Exception:
            pass
    prompt_hash = hashlib.sha256(str(prompt).encode("utf-8")).hexdigest()
    entry = {
        "task_name": task_name,
        "prompt_version": prompt_version,
        "full_prompt_or_template": prompt,
        "prompt_sha256": prompt_hash,
        "model": model,
        "temperature": AI_AUDIT_CONFIG["temperature"],
        "top_p": AI_AUDIT_CONFIG["top_p"],
        "expected_json_schema": expected_schema,
        "uses_numeric_confidence_threshold": False,
    }
    entries = [e for e in manifest["prompts"] if not (
        e.get("task_name") == task_name and e.get("prompt_sha256") == prompt_hash and e.get("model") == model
    )]
    entries.append(entry)
    manifest["prompts"] = entries
    PROMPT_MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

def _append_ai_log(entry):
    AI_INFERENCE_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with AI_INFERENCE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def _validation_for_task(task_name, parsed, rows=None, cols=None):
    if task_name == "stage4_core_matrix_fixed_shape":
        return validate_stage4_core_raw(parsed, rows, cols)
    if task_name == "stage4_context_axis_fixed_shape":
        return validate_stage4_axis_raw(parsed, rows, cols)
    return validate_basic_json_schema(parsed, AI_TASK_SCHEMAS.get(task_name, AI_TASK_SCHEMAS["generic_qwen_json"]))

def run_audited_json_inference(invoke_once, *, prompt, task_name, stage, model,
                               context=None, rows=None, cols=None):
    """Execute a Qwen JSON call, retry once on parse/schema failure, and audit it."""
    context = dict(context or {})
    prompt_version = PROMPT_VERSIONS.get(task_name, PROMPT_VERSIONS["generic_qwen_json"])
    expected_schema = AI_TASK_SCHEMAS.get(task_name, AI_TASK_SCHEMAS["generic_qwen_json"])
    _write_prompt_manifest_entry(task_name, prompt_version, prompt, model, expected_schema)
    identity = "|".join(str(context.get(k, "")) for k in ["paper", "figure", "panel"])
    call_hash = hashlib.sha256((task_name + "|" + identity + "|" + prompt).encode("utf-8")).hexdigest()[:12]
    stem = f"{_audit_safe_name(task_name)}_{call_hash}"
    raw_path = AI_AUDIT_RAW_DIR / f"{stem}.txt"
    parsed_path = AI_AUDIT_JSON_DIR / f"{stem}.json"
    raw = ""
    parsed = {}
    parse_success = False
    schema_valid = False
    validation_errors = []
    attempt_used = 0
    for attempt in range(AI_AUDIT_CONFIG["max_retries"] + 1):
        attempt_used = attempt
        try:
            raw = invoke_once()
            raw = str(raw or "")
            parse_success = True
            try:
                parsed = _extract_json_for_audit(raw)
            except Exception as exc:
                parse_success = False
                parsed = {}
                validation_errors = [f"parse failure: {type(exc).__name__}: {exc}"]
            if parse_success:
                schema_valid, validation_errors = _validation_for_task(task_name, parsed, rows=rows, cols=cols)
            else:
                schema_valid = False
        except Exception as exc:
            raw = f"ERROR: {type(exc).__name__}: {exc}"
            parsed = {}
            parse_success = False
            schema_valid = False
            validation_errors = [raw]
        attempt_raw = raw_path.with_name(f"{raw_path.stem}_attempt_{attempt}{raw_path.suffix}")
        attempt_json = parsed_path.with_name(f"{parsed_path.stem}_attempt_{attempt}{parsed_path.suffix}")
        attempt_raw.write_text(raw, encoding="utf-8")
        attempt_json.write_text(json.dumps(parsed if parse_success else {"parse_error": validation_errors}, ensure_ascii=False, indent=2), encoding="utf-8")
        raw_path.write_text(raw, encoding="utf-8")
        parsed_path.write_text(json.dumps(parsed if parse_success else {"parse_error": validation_errors}, ensure_ascii=False, indent=2), encoding="utf-8")
        if parse_success and schema_valid:
            break
    manual_review = bool(not (parse_success and schema_valid) or (isinstance(parsed, dict) and parsed.get("manual_review", False)))
    review_reason = ""
    if not parse_success or not schema_valid:
        review_reason = "parse/schema validation failed after retry: " + "; ".join(validation_errors)
        if isinstance(parsed, dict):
            parsed["manual_review"] = True
            parsed["review_reason"] = review_reason
            parsed_path.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
    elif isinstance(parsed, dict):
        review_reason = str(parsed.get("review_reason") or parsed.get("reason") or "")
    log_entry = {
        "timestamp_utc": _utc_now_iso(),
        "paper": context.get("paper", PAPER_CONFIG.get("paper_id", ACTIVE_PAPER)),
        "figure": context.get("figure"),
        "panel": context.get("panel"),
        "stage": stage,
        "task": task_name,
        "model": model,
        "prompt_version": prompt_version,
        "temperature": AI_AUDIT_CONFIG["temperature"],
        "top_p": AI_AUDIT_CONFIG["top_p"],
        "raw_response_path": str(raw_path.resolve()),
        "parsed_json_path": str(parsed_path.resolve()),
        "parse_success": bool(parse_success),
        "schema_valid": bool(schema_valid),
        "validation_errors": validation_errors,
        "retry_count": int(attempt_used),
        "manual_review": manual_review,
        "manual_review_reason": review_reason,
        "uses_numeric_confidence_threshold": False,
    }
    _append_ai_log(log_entry)
    return parsed, raw, log_entry

# The notebook's generic vision helper is also a Qwen task and follows the same audit policy.
_vl_chat_json_unversioned = vl_chat_json
def vl_chat_json(image_path, prompt, model_name="qwen3.6-plus"):
    image_url = image_to_data_url(image_path)
    def invoke_once():
        client = build_client()
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]}],
            temperature=AI_AUDIT_CONFIG["temperature"],
            top_p=AI_AUDIT_CONFIG["top_p"],
        )
        return response.choices[0].message.content or ""
    parsed, _raw, _audit = run_audited_json_inference(
        invoke_once, prompt=prompt, task_name="generic_qwen_json", stage="generic",
        model=model_name,
        context={"paper": PAPER_CONFIG.get("paper_id"), "figure": Path(image_path).stem, "panel": None},
    )
    if isinstance(parsed, dict):
        parsed["prompt_version"] = PROMPT_VERSIONS["generic_qwen_json"]
        parsed["uses_numeric_confidence_threshold"] = False
    return parsed


In [18]:
# ============================================================
# Generic heatmap pipeline configuration
# ============================================================
GENERIC_HEATMAP_CONFIG = {
    # Scan the complete paper so heatmap discovery is not restricted to known pages.
    "scan_all_pages": True,
    "page_render_zoom": 3.0,

    # Reconstruct figures from rendered pages. This supports vector and multi-object figures.
    "prefer_rendered_figure_crop": True,
    "figure_crop_horizontal_margin_ratio": 0.025,
    "figure_crop_vertical_margin_ratio": 0.020,

    # Candidate proposal ensemble.
    "use_grid_detector": True,
    "use_color_region_detector": True,
    "use_edge_density_detector": True,
    "use_vlm_full_figure_localizer": True,
    "vlm_localizer_only_when_cv_empty": False,
    "max_candidates": 12,
    "candidate_nms_iou": 0.45,
    "vlm_guided_candidate_filter": True,
    "min_detector_agreement_without_vlm": 2,

    # Broad geometric limits. Portrait and large heatmaps are allowed.
    "min_candidate_width_px": 60,
    "min_candidate_height_px": 35,
    "min_area_ratio": 0.001,
    "max_area_ratio": 0.90,
    "min_aspect_ratio": 0.18,
    "max_aspect_ratio": 20.0,

    # Shape detector: explicit grid first, colour-transition fallback second.
    "shape_min_confidence": 0.45,
    "full_page_fallback_max_text_length": 300,
    "save_cell_visual_features": True,

    # Scientific integrity: colour-only cells remain numerically missing unless source data exist.
    "allow_numeric_value_from_color": False,
    "uses_numeric_confidence_threshold": False,
}

PROMPT_VERSIONS["generic_full_figure_heatmap_localization"] = "generic_heatmap_localizer_v1.0"
AI_TASK_SCHEMAS["generic_full_figure_heatmap_localization"] = {
    "type": "object",
    "required": ["heatmaps", "manual_review", "reason"],
    "properties": {
        "heatmaps": {"type": "array"},
        "manual_review": {"type": "boolean"},
        "reason": {"type": "string"},
    },
}

print("Generic heatmap mode enabled")
print(json.dumps(GENERIC_HEATMAP_CONFIG, indent=2))


Generic heatmap mode enabled
{
  "scan_all_pages": true,
  "page_render_zoom": 3.0,
  "prefer_rendered_figure_crop": true,
  "figure_crop_horizontal_margin_ratio": 0.025,
  "figure_crop_vertical_margin_ratio": 0.02,
  "use_grid_detector": true,
  "use_color_region_detector": true,
  "use_edge_density_detector": true,
  "use_vlm_full_figure_localizer": true,
  "vlm_localizer_only_when_cv_empty": false,
  "max_candidates": 12,
  "candidate_nms_iou": 0.45,
  "vlm_guided_candidate_filter": true,
  "min_detector_agreement_without_vlm": 2,
  "min_candidate_width_px": 60,
  "min_candidate_height_px": 35,
  "min_area_ratio": 0.001,
  "max_area_ratio": 0.9,
  "min_aspect_ratio": 0.18,
  "max_aspect_ratio": 20.0,
  "shape_min_confidence": 0.45,
  "full_page_fallback_max_text_length": 300,
  "save_cell_visual_features": true,
  "allow_numeric_value_from_color": false,
  "uses_numeric_confidence_threshold": false
}


In [ ]:
# =========================================================
# STAGE 1: UNIVERSAL PDF PREPROCESSING
# No AI used
#
# Purpose:
# 1. Open PDF
# 2. Extract page-level text
# 3. Extract text blocks with PDF coordinates
# 4. Detect figure legend candidates
# 5. Extract image object candidates
# 6. Extract table candidates
# 7. Run Stage 1 QC
#
# Important:
# - Stage 1 only extracts objective evidence.
# - Stage 1 does NOT interpret ELISpot data.
# - Stage 1 does NOT split panels.
# - Stage 1 does NOT infer N_Mice / ELISpot_Unit / Treatment / Dose / Route.
# - Stage 1 preserves source_doc / source_section / source_page for downstream stages.
# =========================================================


# =========================================================
# A. Imports
# =========================================================

import os
import re
import json
from pathlib import Path

import fitz
import cv2
import numpy as np


# =========================================================
# B. Basic utilities
# =========================================================

# More universal figure legend start pattern.
# It can match:
#   Figure 1
#   Fig. 2
#   Fig 3
#   Figure S1
#   Fig. S2
#   Supplementary Fig. S1
#   Supplementary Figure 2
#   Extended Data Fig. 1
#   Scheme 1
FIGURE_START_RE = re.compile(
    r"""
    ^\s*
    (?P<figure_label_raw>
        (?:
            (?:Supplementary|Supplemental)\s+
        )?
        (?:
            Figure|Fig\.?|Scheme
        )
        \s*\.?\s*
        S?\d+[A-Za-z]?
        |
        Extended\s+Data\s+Fig\.?\s*\d+[A-Za-z]?
    )
    \b
    """,
    re.I | re.X
)


def ensure_dir(path):
    """
    Ensure a directory exists.
    """
    Path(path).mkdir(parents=True, exist_ok=True)


def save_json(obj, path):
    """
    Save Python object as JSON.
    """
    Path(path).parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def clean_text(x):
    """
    Clean text while preserving content.
    """
    if x is None:
        return ""

    x = str(x)
    x = x.replace("\u00ad", "")
    x = x.replace("\ufeff", "")
    x = x.replace("￾", "")
    x = x.replace("−", "-")
    x = re.sub(r"\s+", " ", x).strip()

    return x


def page_to_rgb(page, zoom=2.0):
    """
    Render one PDF page to RGB image.

    Note:
    Current downstream Stage 2 / Stage 3 does not use page size or zoom metadata.
    Therefore Stage 1 saves page.png but does not add unused dimension fields.
    """
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)

    img = np.frombuffer(
        pix.samples,
        dtype=np.uint8
    ).reshape(pix.height, pix.width, pix.n)

    return img


def extract_figure_label_raw(text):
    """
    Extract raw figure label from a legend start line.

    Example:
        "Fig. 2 LPP-CT26 ..." -> "Fig. 2"
        "Figure S1 ..." -> "Figure S1"
    """
    if text is None:
        return ""

    first_line = str(text).splitlines()[0].strip()

    m = FIGURE_START_RE.match(first_line)

    if not m:
        return ""

    return clean_text(m.group("figure_label_raw"))


# =========================================================
# C. Extract full page text
# =========================================================

def get_page_text(page):
    """
    Extract full page text.
    """
    return page.get_text()


# =========================================================
# D. Extract text blocks with PDF coordinates
# =========================================================

def get_text_blocks(page, source_doc=None, source_section=None, source_page=None):
    """
    Extract text blocks with PDF coordinates.

    Each block keeps:
    - block_index
    - bbox
    - text
    - block_no
    - block_type
    - source_doc
    - source_section
    - source_page
    """
    raw_blocks = page.get_text("blocks", sort=True)
    blocks = []

    for i, item in enumerate(raw_blocks):
        if len(item) < 7:
            continue

        x0, y0, x1, y1, text, block_no, block_type = item[:7]
        text = (text or "").strip()

        if not text:
            continue

        blocks.append({
            "block_index": i,
            "bbox": [x0, y0, x1, y1],
            "text": text,
            "block_no": block_no,
            "block_type": block_type,

            "source_doc": source_doc,
            "source_section": source_section,
            "source_page": source_page
        })

    return blocks


# =========================================================
# E. Detect legend candidates
# =========================================================

def find_legend_start_blocks(blocks):
    """
    Find blocks that look like figure legend starts.
    """
    out = []

    for b in blocks:
        first_line = b["text"].splitlines()[0].strip()

        if FIGURE_START_RE.match(first_line):
            candidate = dict(b)
            candidate["figure_label_raw"] = extract_figure_label_raw(first_line)
            out.append(candidate)

    return out


def build_full_legend_from_start(blocks, start_block_index, max_vertical_gap=40):
    """
    Build a full legend from a legend start block.

    Improvements over old version:
    - Keep included_block_indices
    - Keep end_block_index
    - Keep figure_label_raw
    - Keep source fields
    """
    start_block = None

    for b in blocks:
        if b["block_index"] == start_block_index:
            start_block = b
            break

    if start_block is None:
        return None

    parts = [start_block["text"]]
    included_block_indices = [start_block["block_index"]]

    x0, y0, x1, y1 = start_block["bbox"]
    last_bottom = y1
    end_block_index = start_block["block_index"]

    sorted_blocks = sorted(blocks, key=lambda x: x["block_index"])

    start_pos = None

    for i, b in enumerate(sorted_blocks):
        if b["block_index"] == start_block_index:
            start_pos = i
            break

    if start_pos is None:
        return None

    for j in range(start_pos + 1, len(sorted_blocks)):
        b = sorted_blocks[j]
        txt = b["text"].strip()

        if not txt:
            continue

        first_line = txt.splitlines()[0].strip()

        # Stop when the next figure legend starts.
        if FIGURE_START_RE.match(first_line):
            break

        bx0, by0, bx1, by1 = b["bbox"]
        vertical_gap = by0 - last_bottom

        # Stop if the next block is too far away.
        if vertical_gap > max_vertical_gap:
            break

        parts.append(txt)
        included_block_indices.append(b["block_index"])
        end_block_index = b["block_index"]

        last_bottom = by1

        x0 = min(x0, bx0)
        y0 = min(y0, by0)
        x1 = max(x1, bx1)
        y1 = max(y1, by1)

    full_legend = "\n".join(parts).strip()

    return {
        "legend_bbox_pdf": [x0, y0, x1, y1],
        "full_legend": full_legend,
        "start_block_index": start_block_index,
        "end_block_index": end_block_index,
        "included_block_indices": included_block_indices,
        "figure_label_raw": extract_figure_label_raw(full_legend),

        "source_doc": start_block.get("source_doc"),
        "source_section": start_block.get("source_section"),
        "source_page": start_block.get("source_page")
    }


# =========================================================
# F. Extract image objects from PDF object layer
# =========================================================

def extract_image_objects(page, doc, page_num, out_dir, source_doc=None, source_section=None):
    """
    Extract image object candidates from one PDF page.

    Note:
    These are candidates only.
    Some PDF figures may be vector graphics or multi-object composites.
    The full page image is still saved as fallback by process_one_page_stage1().
    """
    ensure_dir(out_dir)
    image_candidates = []

    seen = set()
    images = page.get_images(full=True)

    for i, img in enumerate(images):
        xref = img[0]

        if xref in seen:
            continue

        seen.add(xref)

        try:
            base = doc.extract_image(xref)
            image_bytes = base["image"]
            ext = base["ext"]
        except Exception:
            continue

        try:
            rects = page.get_image_rects(xref)
        except Exception:
            rects = []

        # If bbox cannot be retrieved, still save image object.
        if not rects:
            img_path = os.path.join(out_dir, f"page{page_num}_img{i+1}.{ext}")

            with open(img_path, "wb") as f:
                f.write(image_bytes)

            image_candidates.append({
                "image_index": i + 1,
                "xref": xref,
                "page_num": page_num,
                "source_doc": source_doc,
                "source_section": source_section,
                "source_page": page_num,
                "bbox": None,
                "image_path": img_path,
                "extraction_method": "pdf_object",
                "bbox_available": False
            })

            continue

        for j, rect in enumerate(rects, start=1):
            img_path = os.path.join(out_dir, f"page{page_num}_img{i+1}_{j}.{ext}")

            with open(img_path, "wb") as f:
                f.write(image_bytes)

            image_candidates.append({
                "image_index": i + 1,
                "xref": xref,
                "page_num": page_num,
                "source_doc": source_doc,
                "source_section": source_section,
                "source_page": page_num,
                "bbox": [rect.x0, rect.y0, rect.x1, rect.y1],
                "image_path": img_path,
                "extraction_method": "pdf_object",
                "bbox_available": True
            })

    return image_candidates


# =========================================================
# G. Extract table candidates
# =========================================================

def try_find_tables(page, page_num, source_doc=None, source_section=None):
    """
    Try to find table candidates on one page.
    """
    tables_info = []

    try:
        tables = page.find_tables()

        for idx, tab in enumerate(tables.tables):
            tables_info.append({
                "table_index": idx,
                "page_num": page_num,
                "source_doc": source_doc,
                "source_section": source_section,
                "source_page": page_num,
                "bbox_pdf": list(tab.bbox)
            })

    except Exception:
        pass

    return tables_info


# =========================================================
# H. Process one page
# =========================================================

def process_one_page_stage1(
    doc,
    page_num,
    out_root="stage1_outputs",
    zoom=2.0,
    source_doc=None,
    source_section=None
):
    """
    Process one PDF page.

    Output:
    - page image
    - page text
    - text blocks
    - legend candidates
    - image candidates
    - table candidates
    - page-level summary
    """
    page = doc[page_num - 1]
    page_dir = Path(out_root) / f"page_{page_num}"
    ensure_dir(page_dir)

    # 1. Save full page image as fallback.
    page_img = page_to_rgb(page, zoom=zoom)
    page_img_path = page_dir / "page.png"
    cv2.imwrite(str(page_img_path), cv2.cvtColor(page_img, cv2.COLOR_RGB2BGR))

    # 2. Save full page text.
    page_text = get_page_text(page)
    page_text_path = page_dir / "page_text.txt"

    with open(page_text_path, "w", encoding="utf-8") as f:
        f.write(page_text)

    # 3. Save text blocks.
    blocks = get_text_blocks(
        page,
        source_doc=source_doc,
        source_section=source_section,
        source_page=page_num
    )

    blocks_path = page_dir / "blocks.json"
    save_json(blocks, blocks_path)

    # 4. Detect figure legend candidates.
    legend_starts = find_legend_start_blocks(blocks)
    legend_candidates = []

    for k, b in enumerate(legend_starts, start=1):
        info = build_full_legend_from_start(
            blocks,
            b["block_index"]
        )

        if info is None:
            continue

        legend_candidates.append({
            "legend_candidate_id": f"page{page_num}_legend{k}",
            "page_num": page_num,
            "source_doc": source_doc,
            "source_section": source_section,
            "source_page": page_num,

            "figure_label_raw": info["figure_label_raw"],
            "legend_bbox_pdf": info["legend_bbox_pdf"],
            "start_block_index": info["start_block_index"],
            "end_block_index": info["end_block_index"],
            "included_block_indices": info["included_block_indices"],
            "full_legend": info["full_legend"]
        })

    legends_path = page_dir / "legend_candidates.json"
    save_json(legend_candidates, legends_path)

    # 5. Extract image object candidates.
    image_dir = page_dir / "images"

    image_candidates = extract_image_objects(
        page=page,
        doc=doc,
        page_num=page_num,
        out_dir=image_dir,
        source_doc=source_doc,
        source_section=source_section
    )

    images_path = page_dir / "image_candidates.json"
    save_json(image_candidates, images_path)

    # 6. Extract table candidates.
    table_candidates = try_find_tables(
        page=page,
        page_num=page_num,
        source_doc=source_doc,
        source_section=source_section
    )

    tables_path = page_dir / "table_candidates.json"
    save_json(table_candidates, tables_path)

    # 7. Return page-level summary.
    return {
        "page_num": page_num,
        "source_doc": source_doc,
        "source_section": source_section,
        "source_page": page_num,

        "page_image_path": str(page_img_path),
        "page_text_path": str(page_text_path),
        "blocks_path": str(blocks_path),
        "legend_candidates_path": str(legends_path),
        "image_candidates_path": str(images_path),
        "table_candidates_path": str(tables_path),

        "page_text_length": len(page_text),
        "block_count": len(blocks),
        "legend_candidate_count": len(legend_candidates),
        "image_candidate_count": len(image_candidates),
        "table_candidate_count": len(table_candidates),

        "fallback_page_image_available": os.path.exists(page_img_path)
    }


# =========================================================
# I. Stage 1 QC
# =========================================================

def run_stage1_qc(page_results):
    """
    Run Stage 1 quality checks.

    Hard failure:
    - missing page image
    - missing page text file
    - empty page text
    - no text blocks

    Soft issues:
    - no legend candidates
    - no image candidates
    - no table candidates
    """
    hard_issues = []
    soft_warnings = []

    for r in page_results:
        page_num = r["page_num"]

        if not os.path.exists(r["page_image_path"]):
            hard_issues.append(f"page {page_num}: missing page image")

        if not os.path.exists(r["page_text_path"]):
            hard_issues.append(f"page {page_num}: missing page text file")

        if r["page_text_length"] == 0:
            hard_issues.append(f"page {page_num}: empty page text")

        if r["block_count"] == 0:
            hard_issues.append(f"page {page_num}: no text blocks")

        if r["legend_candidate_count"] == 0:
            soft_warnings.append(f"page {page_num}: no legend candidates")

        if r["image_candidate_count"] == 0:
            soft_warnings.append(f"page {page_num}: no image candidates")

        if r["table_candidate_count"] == 0:
            soft_warnings.append(f"page {page_num}: no table candidates")

    qc_pass = len(hard_issues) == 0

    qc_summary = {
        "qc_pass": qc_pass,
        "page_count_processed": len(page_results),
        "pages_with_legends": sum(1 for r in page_results if r["legend_candidate_count"] > 0),
        "pages_with_images": sum(1 for r in page_results if r["image_candidate_count"] > 0),
        "pages_with_tables": sum(1 for r in page_results if r["table_candidate_count"] > 0),
        "hard_issues": hard_issues,
        "soft_warnings": soft_warnings
    }

    return qc_summary


# =========================================================
# J. Stage 1 main entry
# =========================================================

def process_pdf_stage1(
    pdf_path,
    pages=None,
    out_root="stage1_outputs",
    zoom=2.0,
    source_doc=None,
    source_section=None
):
    """
    Main entry for Stage 1.

    Input:
    - pdf_path: PDF file path
    - pages: list of 1-based page numbers; None means all pages
    - source_doc: source document label, e.g. paper.pdf or sciadv.adn9961_sm.pdf
    - source_section: main / supplementary / unknown

    Output:
    - stage1_summary.json
    """
    ensure_dir(out_root)

    pdf_path = str(pdf_path)

    if source_doc is None:
        source_doc = Path(pdf_path).name

    if source_section is None:
        source_section = "Not specified"

    doc = fitz.open(pdf_path)

    if pages is None:
        pages = list(range(1, len(doc) + 1))

    page_results = []

    for p in pages:
        page_result = process_one_page_stage1(
            doc=doc,
            page_num=p,
            out_root=out_root,
            zoom=zoom,
            source_doc=source_doc,
            source_section=source_section
        )

        page_results.append(page_result)

    qc_summary = run_stage1_qc(page_results)

    final_summary = {
        "pdf_path": pdf_path,
        "source_doc": source_doc,
        "source_section": source_section,
        "page_count_total": len(doc),
        "pages_processed": pages,
        "page_results": page_results,
        "stage1_qc": qc_summary
    }

    save_json(final_summary, Path(out_root) / "stage1_summary.json")

    doc.close()

    return final_summary

In [ ]:
stage1_pages = None if GENERIC_HEATMAP_CONFIG.get("scan_all_pages", True) else PAPER_CONFIG.get("main_pages")
stage1_result = process_pdf_stage1(
    pdf_path=str(cfg_path("main_pdf", required=True)),
    pages=stage1_pages,
    out_root=str(RUN_ROOT / "stage1"),
    zoom=float(GENERIC_HEATMAP_CONFIG.get("page_render_zoom", 3.0)),
    source_doc=Path(PAPER_CONFIG["main_pdf"]).name,
    source_section="main",
)
stage1_result


In [ ]:
# =========================================================
# STAGE 2: Figure-Legend Pairing + Figure Unit Construction
#
# Purpose:
# 1. Read Stage 1 output
# 2. Pair legend candidates with image candidates
# 3. Build figure units
# 4. Preserve figure identity and source paths for downstream stages
#
# Important:
# - Stage 2 does NOT split panels.
# - Stage 2 does NOT classify heatmaps.
# - Stage 2 does NOT extract ELISpot data.
# - Stage 2 does NOT search in-text evidence sentences.
#
# Downstream use:
# - Stage 3 uses figure_image_path + full_legend for panel splitting.
# - Stage 3C uses figure_number + page_text_path + blocks_path to find
#   in-text references only for heatmap panels.
# =========================================================


import os
import re
import json
from pathlib import Path

import cv2


# =========================================================
# 1. Basic utilities
# =========================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def clean_text(x):
    if x is None:
        return ""

    x = str(x)
    x = x.replace("\u00ad", "")
    x = x.replace("\ufeff", "")
    x = x.replace("￾", "")
    x = re.sub(r"\s+", " ", x).strip()

    return x


def first_non_empty(*values):
    for v in values:
        v = clean_text(v)

        if v and v.lower() not in ["none", "nan", "not specified", "null"]:
            return v

    return "Not specified"


# =========================================================
# 2. Figure label helpers
# =========================================================

FIGURE_LABEL_RE = re.compile(
    r"""
    ^\s*
    (
        (?:
            (?:Supplementary|Supplemental)\s+
        )?
        (?:
            Figure|Fig\.?|Scheme
        )
        \s*\.?\s*
        S?\d+[A-Za-z]?
        |
        Extended\s+Data\s+Fig\.?\s*\d+[A-Za-z]?
    )
    \b
    """,
    re.I | re.X
)


def extract_figure_label_raw(text):
    """
    Extract raw figure label from a legend.

    Examples:
        Fig. 2. Effects... -> Fig. 2
        Figure S1. ... -> Figure S1
        Extended Data Fig. 1. ... -> Extended Data Fig. 1
    """

    text = clean_text(text)

    if not text:
        return "Not specified"

    first_line = text.splitlines()[0].strip()
    m = FIGURE_LABEL_RE.match(first_line)

    if m:
        return clean_text(m.group(1))

    return "Not specified"


def extract_figure_number_from_label(figure_label_raw):
    """
    Extract standardized figure number from raw figure label.

    Examples:
        Fig. 2 -> 2
        Figure 3 -> 3
        Fig. S1 -> S1
        Supplementary Fig. S2 -> S2
        Extended Data Fig. 1 -> 1
    """

    figure_label_raw = clean_text(figure_label_raw)

    if not figure_label_raw or figure_label_raw == "Not specified":
        return "Not specified"

    m = re.search(r"\bS?\d+[A-Za-z]?\b", figure_label_raw, flags=re.I)

    if m:
        return m.group(0).upper()

    return "Not specified"


# =========================================================
# 3. Bbox utilities
# =========================================================

def bbox_area(bbox):
    if bbox is None:
        return 0

    x0, y0, x1, y1 = bbox
    return max(0, x1 - x0) * max(0, y1 - y0)


def bbox_center(bbox):
    if bbox is None:
        return None

    x0, y0, x1, y1 = bbox
    return ((x0 + x1) / 2, (y0 + y1) / 2)


def overlap_1d(a0, a1, b0, b1):
    return max(0, min(a1, b1) - max(a0, b0))


def horizontal_overlap_ratio(box1, box2):
    if box1 is None or box2 is None:
        return 0.0

    x0, _, x1, _ = box1
    xx0, _, xx1, _ = box2

    inter = overlap_1d(x0, x1, xx0, xx1)
    base = min(x1 - x0, xx1 - xx0)

    if base <= 0:
        return 0.0

    return inter / base


def vertical_relation_score(legend_bbox, image_bbox):
    """
    Soft score for vertical relation between image and legend.

    Common case:
        image above legend
    """

    if legend_bbox is None or image_bbox is None:
        return -10

    lx0, ly0, lx1, ly1 = legend_bbox
    ix0, iy0, ix1, iy1 = image_bbox

    # Ideal: image above legend
    if iy1 <= ly0:
        gap = ly0 - iy1
        return 40 + 1000 / (1 + gap)

    # Image and legend overlap vertically
    if not (iy1 < ly0 or iy0 > ly1):
        return -5

    # Image below legend
    if iy0 >= ly1:
        gap = iy0 - ly1
        return -10 + 200 / (1 + gap)

    return -10


# =========================================================
# 4. Image candidate preparation
# =========================================================

def get_image_pixel_size(image_path):
    img = cv2.imread(str(image_path))

    if img is None:
        return 0, 0

    h, w = img.shape[:2]
    return w, h


def prepare_image_candidates(
    image_candidates,
    use_filter=False,
    min_bbox_area=1500,
    min_pixel_w=50,
    min_pixel_h=50
):
    """
    Prepare image candidates.

    Default:
    - Do not strongly filter.
    - Only mark candidate quality.
    """

    kept = []
    dropped = []

    for item in image_candidates:
        candidate = dict(item)

        image_path = candidate.get("image_path")
        bbox = candidate.get("bbox")

        area = bbox_area(bbox)
        w, h = get_image_pixel_size(image_path)

        candidate["bbox_area"] = area
        candidate["pixel_width"] = w
        candidate["pixel_height"] = h

        if w == 0 or h == 0:
            candidate["candidate_quality"] = "bad_read"
        elif (bbox is not None and area < min_bbox_area) or w < min_pixel_w or h < min_pixel_h:
            candidate["candidate_quality"] = "small"
        else:
            candidate["candidate_quality"] = "normal"

        if use_filter and candidate["candidate_quality"] == "small":
            dropped.append(candidate)
        else:
            kept.append(candidate)

    return kept, dropped


# =========================================================
# 5. Legend-image scoring and matching
# =========================================================

def score_legend_image_pair(legend, image):
    """
    Score one legend-image pair.
    Higher score means more likely to be a correct pair.
    """

    if legend.get("page_num") != image.get("page_num"):
        return -1e9

    lb = legend.get("legend_bbox_pdf")
    ib = image.get("bbox")

    if lb is None and ib is None:
        return -1e6

    score = 0.0

    if lb is not None and ib is not None:
        score += vertical_relation_score(lb, ib)

        h_ov = horizontal_overlap_ratio(lb, ib)
        score += h_ov * 60

        lc = bbox_center(lb)
        ic = bbox_center(ib)

        if lc and ic:
            dx = abs(lc[0] - ic[0])
            score += 40 / (1 + dx / 50)

    area = image.get("bbox_area", 0)
    score += min(area / 10000, 25)

    w = image.get("pixel_width", 0)
    h = image.get("pixel_height", 0)
    score += min((w * h) / 20000, 15)

    if image.get("candidate_quality") == "bad_read":
        score -= 200

    return score


def greedy_match_legends_and_images(
    legend_candidates,
    image_candidates,
    min_score=5,
    fallback_min_score=-5
):
    """
    Greedy legend-image matching with a weak fallback.

    Rules:
    - One legend can match at most one image.
    - One image can match at most one legend.
    """

    pairs = []

    for li, legend in enumerate(legend_candidates):
        for ii, image in enumerate(image_candidates):
            s = score_legend_image_pair(legend, image)
            pairs.append((s, li, ii))

    pairs.sort(reverse=True, key=lambda x: x[0])

    used_legends = set()
    used_images = set()
    matches = []

    # Main greedy matching
    for s, li, ii in pairs:
        if s < min_score:
            continue

        if li in used_legends or ii in used_images:
            continue

        used_legends.add(li)
        used_images.add(ii)

        matches.append({
            "score": s,
            "match_round": "main",
            "legend_index": li,
            "image_index": ii,
            "legend": legend_candidates[li],
            "image": image_candidates[ii]
        })

    unmatched_legend_idx = [
        i for i in range(len(legend_candidates))
        if i not in used_legends
    ]

    unmatched_image_idx = [
        i for i in range(len(image_candidates))
        if i not in used_images
    ]

    # Weak fallback if exactly one legend and one image remain
    if len(unmatched_legend_idx) == 1 and len(unmatched_image_idx) == 1:
        li = unmatched_legend_idx[0]
        ii = unmatched_image_idx[0]

        s = score_legend_image_pair(
            legend_candidates[li],
            image_candidates[ii]
        )

        if s >= fallback_min_score:
            used_legends.add(li)
            used_images.add(ii)

            matches.append({
                "score": s,
                "match_round": "fallback",
                "legend_index": li,
                "image_index": ii,
                "legend": legend_candidates[li],
                "image": image_candidates[ii]
            })

    unmatched_legends = [
        legend_candidates[i]
        for i in range(len(legend_candidates))
        if i not in used_legends
    ]

    unmatched_images = [
        image_candidates[i]
        for i in range(len(image_candidates))
        if i not in used_images
    ]

    return matches, unmatched_legends, unmatched_images


# =========================================================
# 6. Build figure unit
# =========================================================

def build_figure_unit_from_match(
    match,
    page_result,
    figure_index,
    source_doc=None,
    source_section=None,
    source_stage1_pdf_path=None
):
    legend = match["legend"]
    image = match["image"]

    full_legend = clean_text(legend.get("full_legend", ""))

    figure_label_raw = first_non_empty(
        legend.get("figure_label_raw"),
        extract_figure_label_raw(full_legend)
    )

    figure_number = extract_figure_number_from_label(figure_label_raw)

    source_page = first_non_empty(
        legend.get("source_page"),
        image.get("source_page"),
        page_result.get("source_page"),
        page_result.get("page_num")
    )

    figure_unit_id = f"page{page_result['page_num']}_figure{figure_index}"

    return {
        "figure_unit_id": figure_unit_id,

        # Source identity
        "source_doc": first_non_empty(source_doc, legend.get("source_doc"), image.get("source_doc")),
        "source_section": first_non_empty(source_section, legend.get("source_section"), image.get("source_section")),
        "source_page": source_page,
        "source_stage1_pdf_path": source_stage1_pdf_path,

        # Page-level links for later Stage 3C in-text evidence search
        "page_num": page_result["page_num"],
        "page_text_path": page_result.get("page_text_path"),
        "blocks_path": page_result.get("blocks_path"),

        # Figure identity
        "figure_label_raw": figure_label_raw,
        "figure_number": figure_number,

        # Legend evidence
        "full_legend": full_legend,
        "legend_bbox_pdf": legend.get("legend_bbox_pdf"),
        "legend_candidate_id": legend.get("legend_candidate_id"),
        "legend_start_block_index": legend.get("start_block_index"),
        "legend_end_block_index": legend.get("end_block_index"),
        "legend_included_block_indices": legend.get("included_block_indices", []),

        # Image evidence
        "figure_image_path": image.get("image_path"),
        "figure_bbox_pdf": image.get("bbox"),
        "image_index": image.get("image_index"),

        # Pairing QC
        "match_score": match["score"],
        "match_round": match.get("match_round", "main"),
        "needs_manual_review": match["score"] < 10
    }


# =========================================================
# 7. Process one page
# =========================================================

def process_one_page_stage2(
    page_result,
    use_filter=False,
    source_doc=None,
    source_section=None,
    source_stage1_pdf_path=None,
    min_score=5,
    fallback_min_score=-5
):
    """
    Process one page:
    legend candidates + image candidates -> figure units.
    """

    legends = load_json(page_result["legend_candidates_path"])
    images = load_json(page_result["image_candidates_path"])

    prepared_images, dropped_images = prepare_image_candidates(
        images,
        use_filter=use_filter,
        min_bbox_area=1500,
        min_pixel_w=50,
        min_pixel_h=50
    )

    matches, unmatched_legends, unmatched_images = greedy_match_legends_and_images(
        legends,
        prepared_images,
        min_score=min_score,
        fallback_min_score=fallback_min_score
    )

    figure_units = []

    for idx, match in enumerate(matches, start=1):
        figure_units.append(
            build_figure_unit_from_match(
                match=match,
                page_result=page_result,
                figure_index=idx,
                source_doc=source_doc,
                source_section=source_section,
                source_stage1_pdf_path=source_stage1_pdf_path
            )
        )

    return {
        "page_num": page_result["page_num"],
        "source_doc": source_doc,
        "source_section": source_section,
        "source_page": page_result.get("source_page", page_result["page_num"]),
        "source_stage1_pdf_path": source_stage1_pdf_path,

        "prepared_images": prepared_images,
        "dropped_images": dropped_images,
        "figure_units": figure_units,
        "unmatched_legends": unmatched_legends,
        "unmatched_images": unmatched_images
    }


# =========================================================
# 8. Stage 2 QC
# =========================================================

def run_stage2_qc(page_stage2_results):
    hard_issues = []
    soft_warnings = []

    total_figure_units = 0
    total_unmatched_legends = 0
    total_unmatched_images = 0
    total_dropped_images = 0

    for r in page_stage2_results:
        page_num = r["page_num"]

        total_figure_units += len(r["figure_units"])
        total_unmatched_legends += len(r["unmatched_legends"])
        total_unmatched_images += len(r["unmatched_images"])
        total_dropped_images += len(r["dropped_images"])

        for fu in r["figure_units"]:
            if not fu.get("figure_image_path") or not os.path.exists(fu["figure_image_path"]):
                hard_issues.append(f"page {page_num}: matched figure image missing for {fu['figure_unit_id']}")

            if not clean_text(fu.get("full_legend", "")):
                hard_issues.append(f"page {page_num}: empty full legend for {fu['figure_unit_id']}")

            if fu.get("figure_number") == "Not specified":
                soft_warnings.append(f"page {page_num}: figure number missing for {fu['figure_unit_id']}")

            if fu.get("match_score", 0) < 10:
                soft_warnings.append(f"page {page_num}: low match score for {fu['figure_unit_id']}")

            if fu.get("needs_manual_review"):
                soft_warnings.append(f"page {page_num}: {fu['figure_unit_id']} needs manual review")

        if len(r["unmatched_legends"]) > 0:
            soft_warnings.append(f"page {page_num}: {len(r['unmatched_legends'])} unmatched legends")

        if len(r["unmatched_images"]) > 0:
            soft_warnings.append(f"page {page_num}: {len(r['unmatched_images'])} unmatched images")

        if len(r["dropped_images"]) > 0:
            soft_warnings.append(f"page {page_num}: {len(r['dropped_images'])} dropped images")

    return {
        "qc_pass": len(hard_issues) == 0,
        "total_figure_units": total_figure_units,
        "total_unmatched_legends": total_unmatched_legends,
        "total_unmatched_images": total_unmatched_images,
        "total_dropped_images": total_dropped_images,
        "hard_issues": hard_issues,
        "soft_warnings": soft_warnings
    }


# =========================================================
# 9. Stage 2 main entry
# =========================================================

def process_stage2_from_stage1(
    stage1_summary_path,
    out_root="stage2_outputs",
    use_filter=False,
    min_score=5,
    fallback_min_score=-5
):
    """
    Main entry for Stage 2.
    """

    Path(out_root).mkdir(parents=True, exist_ok=True)

    stage1_summary = load_json(stage1_summary_path)
    page_results = stage1_summary["page_results"]

    source_doc = stage1_summary.get("source_doc")
    source_section = stage1_summary.get("source_section")
    source_stage1_pdf_path = stage1_summary.get("pdf_path")

    page_stage2_results = []
    all_figure_units = []

    for page_result in page_results:
        one = process_one_page_stage2(
            page_result=page_result,
            use_filter=use_filter,
            source_doc=source_doc,
            source_section=source_section,
            source_stage1_pdf_path=source_stage1_pdf_path,
            min_score=min_score,
            fallback_min_score=fallback_min_score
        )

        page_stage2_results.append(one)
        all_figure_units.extend(one["figure_units"])

    stage2_qc = run_stage2_qc(page_stage2_results)

    final = {
        "source_stage1_summary": str(stage1_summary_path),
        "source_doc": source_doc,
        "source_section": source_section,
        "source_stage1_pdf_path": source_stage1_pdf_path,

        "stage2_role": "figure_legend_image_pairing_only",
        "in_text_reference_search_done": False,
        "in_text_reference_search_stage": "Stage 3C after heatmap classification",

        "page_stage2_results": page_stage2_results,
        "all_figure_units": all_figure_units,
        "stage2_qc": stage2_qc
    }

    save_json(final, Path(out_root) / "stage2_summary.json")

    return final

In [ ]:
stage2_result = process_stage2_from_stage1(
    stage1_summary_path=str(RUN_ROOT / "stage1" / "stage1_summary.json"),
    out_root=str(RUN_ROOT / "stage2"),
    use_filter=False,
)
stage2_result


In [ ]:
# ============================================================
# Generic Stage 2A — reconstruct complete figures from rendered PDF pages
# This is executed after the original Stage 2 matching and rewrites only the
# figure image source used downstream. Original PDF-object evidence is retained.
# ============================================================

def _pdf_bbox_to_pixel_bbox(bbox, page_rect, image_size, pad_x=0, pad_y=0):
    if bbox is None:
        return None
    W, H = image_size
    sx = W / max(float(page_rect.width), 1.0)
    sy = H / max(float(page_rect.height), 1.0)
    x0, y0, x1, y1 = [float(v) for v in bbox]
    return [
        max(0, int(round(x0 * sx)) - pad_x),
        max(0, int(round(y0 * sy)) - pad_y),
        min(W, int(round(x1 * sx)) + pad_x),
        min(H, int(round(y1 * sy)) + pad_y),
    ]


def _figure_band_above_legend(legend_bbox, previous_legend_bottom, page_rect, image_size):
    """Return a broad rendered-page band above a caption, without assuming raster PDF objects."""
    W, H = image_size
    sx = W / max(float(page_rect.width), 1.0)
    sy = H / max(float(page_rect.height), 1.0)
    x_margin = int(W * GENERIC_HEATMAP_CONFIG.get("figure_crop_horizontal_margin_ratio", 0.025))
    y_margin = int(H * GENERIC_HEATMAP_CONFIG.get("figure_crop_vertical_margin_ratio", 0.020))
    legend_y0 = int(float(legend_bbox[1]) * sy)
    top = int(float(previous_legend_bottom or 0) * sy) + y_margin
    bottom = legend_y0 + y_margin
    if bottom - top < max(80, int(H * 0.08)):
        top = max(0, legend_y0 - int(H * 0.62))
    return [x_margin, max(0, top), max(x_margin + 1, W - x_margin), min(H, bottom)]


def reconstruct_stage2_figures_from_page_render(stage1_summary_path, stage2_summary_path, out_root):
    stage1_summary_path = Path(stage1_summary_path)
    stage2_summary_path = Path(stage2_summary_path)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    s1 = load_json(stage1_summary_path)
    s2 = load_json(stage2_summary_path)
    # Rebuild fallback units under the current eligibility rule; this also
    # removes stale text-page fallbacks made by an earlier notebook version.
    s2["all_figure_units"] = [
        unit for unit in s2.get("all_figure_units", [])
        if unit.get("match_round") != "full_page_fallback"
    ]
    page_by_num = {int(p["page_num"]): p for p in s1.get("page_results", [])}
    doc = fitz.open(str(s1["pdf_path"]))
    audit_rows = []

    for fu in s2.get("all_figure_units", []):
        page_num = int(fu["page_num"])
        page_result = page_by_num.get(page_num)
        if not page_result:
            continue
        page_image_path = Path(page_result["page_image_path"])
        if not page_image_path.exists():
            continue
        page_img = Image.open(page_image_path).convert("RGB")
        page_rect = doc[page_num - 1].rect
        legends = load_json(page_result["legend_candidates_path"])
        legend = next((x for x in legends if x.get("legend_candidate_id") == fu.get("legend_candidate_id")), None)
        if legend is None or legend.get("legend_bbox_pdf") is None:
            continue
        ordered = sorted(
            [x for x in legends if x.get("legend_bbox_pdf") is not None],
            key=lambda x: float(x["legend_bbox_pdf"][1]),
        )
        current_index = next((i for i, x in enumerate(ordered) if x.get("legend_candidate_id") == legend.get("legend_candidate_id")), 0)
        previous_bottom = ordered[current_index - 1]["legend_bbox_pdf"][3] if current_index > 0 else 0
        crop_bbox = _figure_band_above_legend(legend["legend_bbox_pdf"], previous_bottom, page_rect, page_img.size)
        crop = page_img.crop(tuple(crop_bbox))
        crop_path = out_root / f"{fu['figure_unit_id']}_rendered_figure.png"
        crop.save(crop_path)
        original_path = fu.get("figure_image_path")
        fu["pdf_object_figure_image_path"] = original_path
        fu["rendered_page_image_path"] = str(page_image_path)
        fu["rendered_figure_bbox_px"] = crop_bbox
        fu["rendered_figure_image_path"] = str(crop_path)
        if GENERIC_HEATMAP_CONFIG.get("prefer_rendered_figure_crop", True):
            fu["figure_image_path"] = str(crop_path)
            fu["figure_image_source"] = "rendered_page_caption_anchored_crop"
        audit_rows.append({
            "figure_unit_id": fu.get("figure_unit_id"),
            "page_num": page_num,
            "figure_number": fu.get("figure_number"),
            "original_pdf_object_path": original_path,
            "rendered_figure_path": str(crop_path),
            "rendered_figure_bbox_px": json.dumps(crop_bbox),
        })

    # Recover captioned figures that Stage 2 could not create because the PDF
    # contained no usable raster image object (common for vector/composite figures).
    existing_legend_ids = {
        fu.get("legend_candidate_id") for fu in s2.get("all_figure_units", [])
        if fu.get("legend_candidate_id")
    }
    for page_num, page_result in sorted(page_by_num.items()):
        page_image_path = Path(page_result["page_image_path"])
        if not page_image_path.exists():
            continue
        page_img = Image.open(page_image_path).convert("RGB")
        page_rect = doc[page_num - 1].rect
        legends = sorted(
            [x for x in load_json(page_result["legend_candidates_path"]) if x.get("legend_bbox_pdf") is not None],
            key=lambda x: float(x["legend_bbox_pdf"][1]),
        )
        for legend_index, legend in enumerate(legends):
            if legend.get("legend_candidate_id") in existing_legend_ids:
                continue
            previous_bottom = legends[legend_index - 1]["legend_bbox_pdf"][3] if legend_index > 0 else 0
            crop_bbox = _figure_band_above_legend(legend["legend_bbox_pdf"], previous_bottom, page_rect, page_img.size)
            crop = page_img.crop(tuple(crop_bbox))
            figure_unit_id = f"page{page_num}_rendered_figure{legend_index + 1}"
            crop_path = out_root / f"{figure_unit_id}_rendered_figure.png"
            crop.save(crop_path)
            full_legend = clean_text(legend.get("full_legend", ""))
            figure_label_raw = first_non_empty(legend.get("figure_label_raw"), extract_figure_label_raw(full_legend))
            new_unit = {
                "figure_unit_id": figure_unit_id,
                "source_doc": s1.get("source_doc"),
                "source_section": s1.get("source_section"),
                "source_page": page_num,
                "source_stage1_pdf_path": s1.get("pdf_path"),
                "page_num": page_num,
                "page_text_path": page_result.get("page_text_path"),
                "blocks_path": page_result.get("blocks_path"),
                "figure_label_raw": figure_label_raw,
                "figure_number": extract_figure_number_from_label(figure_label_raw),
                "full_legend": full_legend,
                "legend_bbox_pdf": legend.get("legend_bbox_pdf"),
                "legend_candidate_id": legend.get("legend_candidate_id"),
                "figure_image_path": str(crop_path),
                "pdf_object_figure_image_path": "Not available",
                "rendered_page_image_path": str(page_image_path),
                "rendered_figure_bbox_px": crop_bbox,
                "rendered_figure_image_path": str(crop_path),
                "figure_image_source": "rendered_page_caption_anchored_crop_recovered_without_pdf_object",
                "match_score": None,
                "match_round": "rendered_page_recovery",
                "needs_manual_review": True,
            }
            s2.setdefault("all_figure_units", []).append(new_unit)
            audit_rows.append({
                "figure_unit_id": figure_unit_id,
                "page_num": page_num,
                "figure_number": new_unit["figure_number"],
                "original_pdf_object_path": "Not available",
                "rendered_figure_path": str(crop_path),
                "rendered_figure_bbox_px": json.dumps(crop_bbox),
                "recovered_without_pdf_object": True,
            })
            existing_legend_ids.add(legend.get("legend_candidate_id"))


    # Fallback for pages without caption-anchored figure units. This is needed
    # for image-only PDFs, supplementary pages, and papers whose captions are
    # embedded in raster figures rather than extractable PDF text.
    pages_with_units = {int(x.get("page_num", -1)) for x in s2.get("all_figure_units", [])}
    for page_num, page_result in sorted(page_by_num.items()):
        if page_num in pages_with_units:
            continue
        page_image_path = Path(page_result["page_image_path"])
        if not page_image_path.exists():
            continue
        # A full-page fallback is appropriate for image-only publisher packs and
        # scanned/supplementary pages. Ordinary article text pages create many
        # false CV regions and unnecessary VLM calls, so exclude them here.
        page_text_length = int(page_result.get("page_text_length", 0) or 0)
        max_text = int(GENERIC_HEATMAP_CONFIG.get("full_page_fallback_max_text_length", 300))
        if page_text_length > max_text:
            continue
        figure_unit_id = f"page{page_num}_full_page_fallback"
        new_unit = {
            "figure_unit_id": figure_unit_id,
            "source_doc": s1.get("source_doc"),
            "source_section": s1.get("source_section"),
            "source_page": page_num,
            "source_stage1_pdf_path": s1.get("pdf_path"),
            "page_num": page_num,
            "page_text_path": page_result.get("page_text_path"),
            "blocks_path": page_result.get("blocks_path"),
            "figure_label_raw": f"Page {page_num}",
            "figure_number": str(page_num),
            "full_legend": "",
            "legend_bbox_pdf": None,
            "legend_candidate_id": None,
            "figure_image_path": str(page_image_path),
            "pdf_object_figure_image_path": "Not available",
            "rendered_page_image_path": str(page_image_path),
            "rendered_figure_bbox_px": [0, 0, Image.open(page_image_path).width, Image.open(page_image_path).height],
            "rendered_figure_image_path": str(page_image_path),
            "figure_image_source": "rendered_full_page_fallback_without_caption",
            "match_score": None,
            "match_round": "full_page_fallback",
            "needs_manual_review": True,
        }
        s2.setdefault("all_figure_units", []).append(new_unit)
        audit_rows.append({
            "figure_unit_id": figure_unit_id,
            "page_num": page_num,
            "figure_number": str(page_num),
            "original_pdf_object_path": "Not available",
            "rendered_figure_path": str(page_image_path),
            "rendered_figure_bbox_px": json.dumps(new_unit["rendered_figure_bbox_px"]),
            "full_page_fallback": True,
        })

    doc.close()
    s2["generic_figure_reconstruction"] = {
        "enabled": True,
        "method": "caption_anchored_crop_from_rendered_pdf_page",
        "count": len(audit_rows),
    }
    save_json(s2, stage2_summary_path)
    audit_df = pd.DataFrame(audit_rows)
    audit_csv = out_root / "figure_reconstruction_manifest.csv"
    audit_df.to_csv(audit_csv, index=False, encoding="utf-8-sig")
    return {"stage2_summary_path": str(stage2_summary_path), "manifest_csv": str(audit_csv), "records": audit_rows}


generic_figure_reconstruction = reconstruct_stage2_figures_from_page_render(
    stage1_summary_path=RUN_ROOT / "stage1" / "stage1_summary.json",
    stage2_summary_path=RUN_ROOT / "stage2" / "stage2_summary.json",
    out_root=RUN_ROOT / "stage2" / "rendered_figures",
)
print(generic_figure_reconstruction)


In [ ]:
# ============================================================
# STAGE 3 FINAL MODIFIED CODE
# Tool detects candidate heatmap core crops.
# AI checks whether CORE CROP is a true heatmap / numeric matrix body.
# If yes, keep corresponding CONTEXT CROP for Stage 4.
#
# Output:
# - candidate core crops
# - candidate context crops
# - overlay image
# - AI selection table
# - selected context crop manifest for Stage 4
# ============================================================

import os
import re
import json
import base64
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display
from openai import OpenAI


# ============================================================
# 0. Basic helpers
# ============================================================

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    path = Path(path)
    ensure_dir(path.parent)

    def convert(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=convert)


def clean_text(x):
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\u00ad", "")
    x = x.replace("­", "")
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def safe_int(x):
    try:
        return int(round(float(x)))
    except Exception:
        return 0


def bbox_area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])


def iou(b1, b2):
    x0 = max(b1[0], b2[0])
    y0 = max(b1[1], b2[1])
    x1 = min(b1[2], b2[2])
    y1 = min(b1[3], b2[3])

    inter = bbox_area([x0, y0, x1, y1])
    union = bbox_area(b1) + bbox_area(b2) - inter

    return inter / union if union > 0 else 0.0


def non_max_suppression(candidates, iou_threshold=0.30):
    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
    kept = []

    for c in candidates:
        if all(iou(c["bbox"], k["bbox"]) <= iou_threshold for k in kept):
            kept.append(c)

    return kept


def crop_pil(img, bbox):
    W, H = img.size
    x0, y0, x1, y1 = bbox

    x0 = clamp(safe_int(x0), 0, W)
    y0 = clamp(safe_int(y0), 0, H)
    x1 = clamp(safe_int(x1), 0, W)
    y1 = clamp(safe_int(y1), 0, H)

    if x1 <= x0:
        x1 = min(W, x0 + 1)
    if y1 <= y0:
        y1 = min(H, y0 + 1)

    return img.crop((x0, y0, x1, y1))


# ============================================================
# 1. Qwen / DashScope client
# ============================================================

def build_client():
    return OpenAI(
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )


def image_to_data_url(image_path):
    image_path = Path(image_path)
    ext = image_path.suffix.lower().replace(".", "")

    if ext == "jpg":
        ext = "jpeg"

    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    return f"data:image/{ext};base64,{b64}"


def extract_first_json(text):
    text = str(text).strip()

    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError("Cannot parse JSON from AI response")


# ============================================================
# 2. Grid / numeric matrix candidate detection
# ============================================================

def expand_core_bbox_to_context(bbox, image_size):
    """
    Core crop = numeric matrix body.
    Context crop = expanded region containing title, row labels, column labels, colorbar.
    """
    W, H = image_size
    x0, y0, x1, y1 = bbox

    bw = x1 - x0
    bh = y1 - y0

    return [
        clamp(x0 - int(bw * 0.45) - 35, 0, W),
        clamp(y0 - int(bh * 0.80) - 45, 0, H),
        clamp(x1 + int(bw * 0.35) + 60, 0, W),
        clamp(y1 + int(bh * 0.65) + 45, 0, H),
    ]


def count_line_runs(projection, threshold):
    idx = np.where(projection >= threshold)[0]

    if len(idx) == 0:
        return 0

    runs = 1
    for i in range(1, len(idx)):
        if idx[i] - idx[i - 1] > 2:
            runs += 1

    return runs


def grid_score_for_bbox(binary_inv, bbox):
    x0, y0, x1, y1 = bbox
    crop = binary_inv[y0:y1, x0:x1]

    if crop.size == 0:
        return 0, 0, 0

    h, w = crop.shape

    if h < 20 or w < 40:
        return 0, 0, 0

    vertical_projection = crop.sum(axis=0)
    horizontal_projection = crop.sum(axis=1)

    v_thr = max(1, np.percentile(vertical_projection, 92))
    h_thr = max(1, np.percentile(horizontal_projection, 92))

    v_runs = count_line_runs(vertical_projection, v_thr)
    h_runs = count_line_runs(horizontal_projection, h_thr)

    grid_score = v_runs * 2 + h_runs * 3

    return grid_score, v_runs, h_runs


def detect_numeric_matrix_candidates(
    figure_image_path,
    out_dir,
    figure_unit_id="figure",
    max_candidates=20,
    min_width=100,
    min_height=35,
    max_area_ratio=0.25,
    nms_iou=0.30
):
    """
    Detect grid-like candidate regions.
    This detects numeric matrix core regions, not red tumor photos.
    """
    figure_image_path = Path(figure_image_path)
    out_dir = Path(out_dir)

    overlay_dir = out_dir / "overlay"
    core_dir = out_dir / "core_candidates"
    context_dir = out_dir / "context_candidates"

    ensure_dir(overlay_dir)
    ensure_dir(core_dir)
    ensure_dir(context_dir)

    pil_img = Image.open(figure_image_path).convert("RGB")
    W, H = pil_img.size

    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    binary_inv = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        15
    )

    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25))

    horizontal = cv2.morphologyEx(binary_inv, cv2.MORPH_OPEN, horizontal_kernel)
    vertical = cv2.morphologyEx(binary_inv, cv2.MORPH_OPEN, vertical_kernel)

    grid_mask = cv2.bitwise_or(horizontal, vertical)

    close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 9))
    connected = cv2.morphologyEx(grid_mask, cv2.MORPH_CLOSE, close_kernel)

    contours, _ = cv2.findContours(
        connected,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    raw_candidates = []
    image_area = W * H

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)

        if w < min_width or h < min_height:
            continue

        area = w * h
        area_ratio = area / image_area

        if area_ratio > max_area_ratio:
            continue

        aspect = w / max(h, 1)

        if aspect < 1.2 or aspect > 18:
            continue

        bbox = [x, y, x + w, y + h]

        grid_score, v_runs, h_runs = grid_score_for_bbox(binary_inv, bbox)

        if v_runs < 4 or h_runs < 2:
            continue

        score = (
            grid_score * 10
            + min(w / 10, 90)
            + min(h / 5, 70)
            - area_ratio * 80
        )

        raw_candidates.append({
            "bbox": bbox,
            "score": float(score),
            "width": int(w),
            "height": int(h),
            "area": int(area),
            "area_ratio": float(area_ratio),
            "aspect_ratio": float(aspect),
            "v_line_runs": int(v_runs),
            "h_line_runs": int(h_runs),
            "grid_score": float(grid_score)
        })

    candidates = non_max_suppression(raw_candidates, iou_threshold=nms_iou)
    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)[:max_candidates]

    overlay = pil_img.copy()
    draw = ImageDraw.Draw(overlay)

    records = []

    for i, c in enumerate(candidates):
        core_bbox = c["bbox"]
        context_bbox = expand_core_bbox_to_context(core_bbox, (W, H))

        core_path = core_dir / f"{figure_unit_id}_candidate_{i:02d}_core.png"
        context_path = context_dir / f"{figure_unit_id}_candidate_{i:02d}_context.png"

        crop_pil(pil_img, core_bbox).save(core_path)
        crop_pil(pil_img, context_bbox).save(context_path)

        draw.rectangle(core_bbox, outline="red", width=4)
        draw.text((core_bbox[0] + 5, core_bbox[1] + 5), str(i), fill="red")
        draw.rectangle(context_bbox, outline="blue", width=2)

        rec = {
            "candidate_id": i,
            "figure_unit_id": figure_unit_id,
            "figure_image_path": str(figure_image_path),
            "core_bbox": core_bbox,
            "context_bbox": context_bbox,
            "score": c["score"],
            "width": c["width"],
            "height": c["height"],
            "area": c["area"],
            "area_ratio": c["area_ratio"],
            "aspect_ratio": c["aspect_ratio"],
            "v_line_runs": c["v_line_runs"],
            "h_line_runs": c["h_line_runs"],
            "grid_score": c["grid_score"],
            "core_crop_path": str(core_path),
            "context_crop_path": str(context_path)
        }

        records.append(rec)

    overlay_path = overlay_dir / f"{figure_unit_id}_candidate_overlay.png"
    overlay.save(overlay_path)

    manifest_df = pd.DataFrame(records)

    manifest_csv = out_dir / f"{figure_unit_id}_candidate_manifest.csv"
    manifest_xlsx = out_dir / f"{figure_unit_id}_candidate_manifest.xlsx"

    manifest_df.to_csv(manifest_csv, index=False, encoding="utf-8-sig")
    manifest_df.to_excel(manifest_xlsx, index=False)

    return {
        "manifest_df": manifest_df,
        "overlay_path": str(overlay_path),
        "manifest_csv": str(manifest_csv),
        "manifest_xlsx": str(manifest_xlsx),
        "core_dir": str(core_dir),
        "context_dir": str(context_dir)
    }


# ============================================================
# 3. AI selection prompt:
#    judge CORE crop, return CONTEXT crop if core is matrix
# ============================================================

def ai_judge_core_matrix_candidate(
    core_image_path,
    context_image_path,
    model_name="qwen3-vl-plus",
    target_assay="ELISpot"
):
    prompt = f"""
You are selecting candidate crops for scientific heatmap data extraction.

Return ONLY valid JSON. Do not output markdown or explanations outside JSON.

You will receive two images:
1. CORE CROP:
   This is the tool-detected candidate matrix body.
   It may only contain the heatmap/numeric grid itself.
   It may NOT contain the full title, row labels, column labels, or panel label.
2. CONTEXT CROP:
   This is the expanded crop around the core crop.
   It is used to provide title, row labels, column labels, axis labels, colorbar, and nearby context.

Your decision must be based mainly on the CORE CROP.

Select this candidate if:
- The CORE CROP contains a numeric grid, heatmap, or matrix-like table.
- The CORE CROP contains multiple rectangular cells arranged in rows and columns.
- The CORE CROP contains printed numeric values inside cells, or color/intensity-coded cells.
- The CORE CROP looks like the body of a scientific heatmap or matrix, even if labels are missing.
- The CONTEXT CROP contains enough surrounding information to interpret the matrix, such as title, row labels, column labels, colorbar, epitope numbers, timepoints, or treatment names.

Do NOT reject simply because:
- The CORE CROP lacks the title.
- The CORE CROP lacks row labels.
- The CORE CROP lacks column labels.
- The CORE CROP is only the internal heatmap body.
- The CONTEXT CROP contains extra blank space or neighboring figure fragments.

Reject this candidate only if:
- The CORE CROP is mainly a line plot, bar chart, scatter plot, tumor photo, schematic, microscopy image, axis label, or plain text.
- The CORE CROP is a montage of microscopy fields, fluorescence images, anatomical images, or raw assay-well/spot photographs arranged in rows and columns. A regular layout of photographs is not a heatmap matrix.
- Circular wells or raw spot images are the cell content rather than rectangular color/intensity-coded or annotated heatmap cells.
- The CORE CROP only touches a tiny edge of a heatmap but is not the heatmap body.
- The CORE CROP is too cropped to see the matrix cells.
- The CONTEXT CROP does not provide enough information to recover row/column labels or interpret the matrix.

Assay attribution rule:
- Never infer an assay type from gene, cytokine, marker, treatment, or cell-type names alone.
- Report a specific assay only when visible text or unambiguous assay-specific evidence supports it; otherwise use Not specified.

Important distinction:
- If the CORE CROP is the numeric matrix body, select it.
- If only the CONTEXT CROP touches a heatmap at the edge but the CORE CROP is not a matrix, reject it.

Return schema:
{{
  "core_is_numeric_matrix_or_heatmap_body": true,
  "core_has_grid_cells": true,
  "core_has_cell_values_or_intensity": true,
  "context_has_interpretable_labels": true,
  "context_has_elispot_or_assay_text": true,
  "heatmap_is_only_edge_fragment": false,
  "main_core_content_is_non_heatmap": false,
  "is_suitable_for_stage4_extraction": true,
  "detected_assay": "assay name explicitly supported by visible text | Not specified",
  "detected_panel_id": "A/B/C/... or Not specified",
  "visible_row_labels_from_context": [],
  "visible_column_labels_from_context": [],
  "confidence": "high | medium | low",
  "manual_review": false,
  "reason": ""
}}
""".strip()

    client = build_client()

    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "text", "text": "CORE CROP:"},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_data_url(core_image_path)
                        }
                    },
                    {"type": "text", "text": "CONTEXT CROP:"},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_data_url(context_image_path)
                        }
                    }
                ]
            }
        ]
    )

    raw = response.choices[0].message.content
    parsed = extract_first_json(raw)

    return parsed, raw


def classify_and_select_context_from_core_candidates(
    manifest_df,
    out_dir,
    model_name="qwen3-vl-plus",
    target_assay="ELISpot"
):
    """
    For each candidate:
    - send core crop + context crop to AI
    - AI judges whether core crop is numeric matrix body
    - if yes, save context crop as selected_context_path
    """
    out_dir = Path(out_dir)

    ai_raw_dir = out_dir / "ai_raw"
    selected_dir = out_dir / "selected_context"

    ensure_dir(ai_raw_dir)
    ensure_dir(selected_dir)

    rows = []

    for _, r in manifest_df.iterrows():
        candidate_id = int(r["candidate_id"])

        core_path = Path(r["core_crop_path"])
        context_path = Path(r["context_crop_path"])

        print("\n" + "=" * 100)
        print("AI judging candidate:", candidate_id)
        print("Core:", core_path)
        print("Context:", context_path)

        try:
            raw_txt_path = ai_raw_dir / f"candidate_{candidate_id:02d}_raw_response.txt"
            parsed_json_path = ai_raw_dir / f"candidate_{candidate_id:02d}_parsed.json"
            if globals().get('CACHE_EXISTING', True) and parsed_json_path.exists():
                ai_json = load_json(parsed_json_path)
                raw = raw_txt_path.read_text(encoding="utf-8") if raw_txt_path.exists() else json.dumps(ai_json, ensure_ascii=False)
                print("Using cached candidate validation:", parsed_json_path)
            else:
                ai_json, raw = ai_judge_core_matrix_candidate(
                    core_image_path=core_path,
                    context_image_path=context_path,
                    model_name=model_name,
                    target_assay=target_assay
                )
                raw_txt_path.write_text(str(raw), encoding="utf-8")
                save_json(ai_json, parsed_json_path)

            core_is_matrix = bool(ai_json.get("core_is_numeric_matrix_or_heatmap_body", False))
            core_has_grid = bool(ai_json.get("core_has_grid_cells", False))
            core_has_values = bool(ai_json.get("core_has_cell_values_or_intensity", False))
            labels_ok = bool(ai_json.get("context_has_interpretable_labels", False))
            edge_fragment = bool(ai_json.get("heatmap_is_only_edge_fragment", True))
            non_heatmap_core = bool(ai_json.get("main_core_content_is_non_heatmap", True))
            suitable = bool(ai_json.get("is_suitable_for_stage4_extraction", False))
            confidence = str(ai_json.get("confidence", "low")).lower()

            selected = (
                core_is_matrix
                and core_has_grid
                and core_has_values
                and labels_ok
                and not edge_fragment
                and not non_heatmap_core
                and suitable
            )

            manual_review = bool(ai_json.get("manual_review", False)) or confidence == "low"

            out_row = r.to_dict()
            out_row.update({
                "ai_core_is_numeric_matrix_or_heatmap_body": core_is_matrix,
                "ai_core_has_grid_cells": core_has_grid,
                "ai_core_has_cell_values_or_intensity": core_has_values,
                "ai_context_has_interpretable_labels": labels_ok,
                "ai_context_has_elispot_or_assay_text": bool(ai_json.get("context_has_elispot_or_assay_text", False)),
                "ai_heatmap_is_only_edge_fragment": edge_fragment,
                "ai_main_core_content_is_non_heatmap": non_heatmap_core,
                "ai_is_suitable_for_stage4_extraction": suitable,
                "ai_detected_assay": ai_json.get("detected_assay", "Not specified"),
                "ai_detected_panel_id": ai_json.get("detected_panel_id", "Not specified"),
                "ai_visible_row_labels_from_context": json.dumps(ai_json.get("visible_row_labels_from_context", []), ensure_ascii=False),
                "ai_visible_column_labels_from_context": json.dumps(ai_json.get("visible_column_labels_from_context", []), ensure_ascii=False),
                "ai_confidence": confidence,
                "ai_manual_review": manual_review,
                "ai_reason": ai_json.get("reason", ""),
                "ai_raw_response_path": str(raw_txt_path),
                "ai_parsed_json_path": str(parsed_json_path),
                "selected_for_stage4": selected
            })

            if selected:
                selected_context_path = selected_dir / f"{r['figure_unit_id']}_candidate_{candidate_id:02d}_selected_context.png"
                shutil.copy2(context_path, selected_context_path)
                out_row["selected_context_path"] = str(selected_context_path)
            else:
                out_row["selected_context_path"] = "Not selected"

            rows.append(out_row)

            print("core_is_matrix:", core_is_matrix)
            print("core_has_grid:", core_has_grid)
            print("core_has_values:", core_has_values)
            print("labels_ok:", labels_ok)
            print("edge_fragment:", edge_fragment)
            print("non_heatmap_core:", non_heatmap_core)
            print("suitable:", suitable)
            print("selected:", selected)
            print("reason:", ai_json.get("reason", ""))

        except Exception as e:
            out_row = r.to_dict()
            out_row.update({
                "ai_core_is_numeric_matrix_or_heatmap_body": False,
                "ai_core_has_grid_cells": False,
                "ai_core_has_cell_values_or_intensity": False,
                "ai_context_has_interpretable_labels": False,
                "ai_context_has_elispot_or_assay_text": False,
                "ai_heatmap_is_only_edge_fragment": True,
                "ai_main_core_content_is_non_heatmap": True,
                "ai_is_suitable_for_stage4_extraction": False,
                "ai_detected_assay": "Not specified",
                "ai_detected_panel_id": "Not specified",
                "ai_visible_row_labels_from_context": "[]",
                "ai_visible_column_labels_from_context": "[]",
                "ai_confidence": "low",
                "ai_manual_review": True,
                "ai_reason": f"AI failed: {e}",
                "ai_raw_response_path": "Not generated",
                "ai_parsed_json_path": "Not generated",
                "selected_for_stage4": False,
                "selected_context_path": "Not selected"
            })

            rows.append(out_row)
            print("AI failed:", e)

    result_df = pd.DataFrame(rows)

    selection_csv = out_dir / "stage3_ai_candidate_selection.csv"
    selection_xlsx = out_dir / "stage3_ai_candidate_selection.xlsx"

    result_df.to_csv(selection_csv, index=False, encoding="utf-8-sig")
    result_df.to_excel(selection_xlsx, index=False)

    selected_df = result_df[result_df["selected_for_stage4"] == True].copy()

    selected_csv = out_dir / "stage3_selected_heatmap_context_manifest.csv"
    selected_xlsx = out_dir / "stage3_selected_heatmap_context_manifest.xlsx"

    selected_df.to_csv(selected_csv, index=False, encoding="utf-8-sig")
    selected_df.to_excel(selected_xlsx, index=False)

    return {
        "selection_df": result_df,
        "selected_df": selected_df,
        "selection_csv": str(selection_csv),
        "selection_xlsx": str(selection_xlsx),
        "selected_csv": str(selected_csv),
        "selected_xlsx": str(selected_xlsx),
        "selected_dir": str(selected_dir)
    }


# ============================================================
# 4. Process one figure
# ============================================================

def process_one_figure_stage3_heatmap(
    figure_unit,
    out_root,
    model_name="qwen3-vl-plus",
    target_assay="ELISpot",
    max_candidates=20
):
    figure_unit_id = figure_unit["figure_unit_id"]
    figure_image_path = figure_unit["figure_image_path"]

    figure_out_dir = Path(out_root) / figure_unit_id
    ensure_dir(figure_out_dir)

    print("\n" + "=" * 120)
    print("Stage 3 processing:", figure_unit_id)
    print("Image:", figure_image_path)

    detection = detect_numeric_matrix_candidates(
        figure_image_path=figure_image_path,
        out_dir=figure_out_dir,
        figure_unit_id=figure_unit_id,
        max_candidates=max_candidates
    )

    manifest_df = detection["manifest_df"]

    if manifest_df.empty:
        return {
            "figure_unit_id": figure_unit_id,
            "figure_image_path": figure_image_path,
            "candidate_count": 0,
            "selected_count": 0,
            "overlay_path": detection["overlay_path"],
            "candidate_manifest_csv": detection["manifest_csv"],
            "selected_manifest_csv": None,
            "selected_context_records": [],
            "status": "no_candidates",
            "reason": "No numeric matrix candidates detected."
        }

    selection = classify_and_select_context_from_core_candidates(
        manifest_df=manifest_df,
        out_dir=figure_out_dir,
        model_name=model_name,
        target_assay=target_assay
    )

    selected_df = selection["selected_df"]

    selected_context_records = []

    for _, r in selected_df.iterrows():
        rec = {
            "figure_unit_id": figure_unit_id,
            "source_doc": figure_unit.get("source_doc"),
            "source_section": figure_unit.get("source_section"),
            "source_stage1_pdf_path": figure_unit.get("source_stage1_pdf_path"),
            "page_num": figure_unit.get("page_num"),
            "figure_image_path": figure_image_path,
            "full_legend": figure_unit.get("full_legend", ""),
            "candidate_id": int(r["candidate_id"]),
            "core_bbox": r["core_bbox"],
            "context_bbox": r["context_bbox"],
            "core_crop_path": r["core_crop_path"],
            "context_crop_path": r["context_crop_path"],
            "selected_context_path": r["selected_context_path"],
            "ai_detected_assay": r.get("ai_detected_assay", "Not specified"),
            "ai_detected_panel_id": r.get("ai_detected_panel_id", "Not specified"),
            "ai_confidence": r.get("ai_confidence", "Not specified"),
            "ai_manual_review": bool(r.get("ai_manual_review", False)),
            "ai_reason": r.get("ai_reason", ""),
            "stage3_selection_method": "tool_core_grid_candidate_then_ai_core_matrix_judgement"
        }

        selected_context_records.append(rec)

    return {
        "figure_unit_id": figure_unit_id,
        "figure_image_path": figure_image_path,
        "candidate_count": int(len(manifest_df)),
        "selected_count": int(len(selected_df)),
        "overlay_path": detection["overlay_path"],
        "candidate_manifest_csv": detection["manifest_csv"],
        "selected_manifest_csv": selection["selected_csv"],
        "selected_manifest_xlsx": selection["selected_xlsx"],
        "selected_context_records": selected_context_records,
        "status": "ok",
        "reason": ""
    }


# ============================================================
# 5. Main Stage 3 from Stage 2
# ============================================================

def process_stage3_heatmaps_from_stage2(
    stage2_summary_path,
    out_root,
    model_name="qwen3-vl-plus",
    target_assay="ELISpot",
    figure_unit_filter=None,
    max_candidates=20
):
    stage2_summary = load_json(stage2_summary_path)
    out_root = Path(out_root)
    ensure_dir(out_root)

    if "all_figure_units" in stage2_summary:
        figure_units = stage2_summary["all_figure_units"]
    elif "figure_units" in stage2_summary:
        figure_units = stage2_summary["figure_units"]
    else:
        raise KeyError("stage2_summary.json does not contain all_figure_units / figure_units")

    if figure_unit_filter is not None:
        if isinstance(figure_unit_filter, str):
            figure_unit_filter = [figure_unit_filter]

        figure_units = [
            fu for fu in figure_units
            if fu.get("figure_unit_id") in figure_unit_filter
        ]

    figure_results = []
    failed_figures = []
    all_selected_records = []

    for fu in figure_units:
        try:
            one = process_one_figure_stage3_heatmap(
                figure_unit=fu,
                out_root=out_root,
                model_name=model_name,
                target_assay=target_assay,
                max_candidates=max_candidates
            )

            figure_results.append(one)
            all_selected_records.extend(one.get("selected_context_records", []))

        except Exception as e:
            failed = {
                "figure_unit_id": fu.get("figure_unit_id"),
                "figure_image_path": fu.get("figure_image_path"),
                "error": str(e)
            }
            failed_figures.append(failed)
            print("Failed:", failed)

    selected_manifest_df = pd.DataFrame(all_selected_records)

    selected_csv = out_root / "stage3_selected_heatmap_context_manifest.csv"
    selected_xlsx = out_root / "stage3_selected_heatmap_context_manifest.xlsx"

    selected_manifest_df.to_csv(selected_csv, index=False, encoding="utf-8-sig")
    selected_manifest_df.to_excel(selected_xlsx, index=False)

    stage3_qc = {
        "qc_pass": len(failed_figures) == 0,
        "source_stage2_summary": str(stage2_summary_path),
        "model_name": model_name,
        "target_assay": target_assay,
        "figure_unit_filter": figure_unit_filter,
        "total_figures_processed": int(len(figure_results)),
        "total_failed_figures": int(len(failed_figures)),
        "total_selected_heatmap_contexts": int(len(selected_manifest_df)),
        "selected_manifest_csv": str(selected_csv),
        "selected_manifest_xlsx": str(selected_xlsx)
    }

    summary = {
        "source_stage2_summary": str(stage2_summary_path),
        "model_name": model_name,
        "target_assay": target_assay,
        "stage3_qc": stage3_qc,
        "figure_results": figure_results,
        "failed_figures": failed_figures,
        "selected_context_records": all_selected_records
    }

    summary_path = out_root / "stage3_heatmap_summary.json"
    save_json(summary, summary_path)

    print("\n" + "=" * 120)
    print("Stage 3 complete.")
    print("Processed figures:", len(figure_results))
    print("Failed figures:", len(failed_figures))
    print("Selected context crops:", len(selected_manifest_df))
    print("Selected manifest:", selected_xlsx)
    print("Summary:", summary_path)

    return summary


# ============================================================
# 6. Display helpers
# ============================================================

def display_stage3_overlays(stage3_result):
    for fr in stage3_result.get("figure_results", []):
        overlay = fr.get("overlay_path")

        if overlay and Path(overlay).exists():
            print("\n" + "=" * 100)
            print("figure_unit_id:", fr["figure_unit_id"])
            print("overlay:", overlay)
            display(Image.open(overlay))


def display_stage3_selected_contexts(stage3_result):
    records = stage3_result.get("selected_context_records", [])

    if not records:
        print("No selected heatmap context crops.")
        return

    for r in records:
        print("\n" + "=" * 100)
        print("figure_unit_id:", r["figure_unit_id"])
        print("candidate_id:", r["candidate_id"])
        print("assay:", r["ai_detected_assay"])
        print("panel:", r["ai_detected_panel_id"])
        print("confidence:", r["ai_confidence"])
        print("manual_review:", r["ai_manual_review"])
        print("reason:", r["ai_reason"])
        print("selected_context_path:", r["selected_context_path"])

        display(Image.open(r["selected_context_path"]))


def inspect_stage3_selection_table(stage3_result):
    selected_path = stage3_result["stage3_qc"]["selected_manifest_xlsx"]

    print("Selected manifest:", selected_path)

    if Path(selected_path).exists():
        df = pd.read_excel(selected_path)
        display(df)
    else:
        print("Selected manifest not found.")

In [ ]:
# ============================================================
# Generic Stage 3 heatmap detector
# Ensemble: explicit grid + colour regions + edge density + Qwen full-figure localization.
# The original detect_numeric_matrix_candidates() is retained as one proposal source.
# ============================================================

_legacy_detect_numeric_matrix_candidates = detect_numeric_matrix_candidates


def _clip_bbox(bbox, W, H):
    x0, y0, x1, y1 = [int(round(v)) for v in bbox]
    x0, x1 = sorted((max(0, x0), min(W, x1)))
    y0, y1 = sorted((max(0, y0), min(H, y1)))
    return [x0, y0, x1, y1]


def _bbox_ok_general(bbox, W, H):
    x0, y0, x1, y1 = bbox
    w, h = x1 - x0, y1 - y0
    if w < GENERIC_HEATMAP_CONFIG["min_candidate_width_px"] or h < GENERIC_HEATMAP_CONFIG["min_candidate_height_px"]:
        return False
    ratio = (w * h) / max(W * H, 1)
    aspect = w / max(h, 1)
    return (
        GENERIC_HEATMAP_CONFIG["min_area_ratio"] <= ratio <= GENERIC_HEATMAP_CONFIG["max_area_ratio"]
        and GENERIC_HEATMAP_CONFIG["min_aspect_ratio"] <= aspect <= GENERIC_HEATMAP_CONFIG["max_aspect_ratio"]
    )


def _proposal_record(bbox, W, H, source, score, evidence=None):
    x0, y0, x1, y1 = _clip_bbox(bbox, W, H)
    w, h = x1 - x0, y1 - y0
    return {
        "bbox": [x0, y0, x1, y1],
        "score": float(score),
        "width": int(w),
        "height": int(h),
        "area": int(w * h),
        "area_ratio": float((w * h) / max(W * H, 1)),
        "aspect_ratio": float(w / max(h, 1)),
        "v_line_runs": int((evidence or {}).get("v_line_runs", 0)),
        "h_line_runs": int((evidence or {}).get("h_line_runs", 0)),
        "grid_score": float((evidence or {}).get("grid_score", 0.0)),
        "proposal_source": source,
        "proposal_evidence": json.dumps(evidence or {}, ensure_ascii=False),
    }


def _grid_proposals_general(gray):
    H, W = gray.shape
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 31, 12)
    proposals = []
    for hk, vk in [(max(12, W // 90), max(10, H // 90)), (max(25, W // 45), max(18, H // 45)), (max(40, W // 25), max(30, H // 25))]:
        horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (hk, 1)))
        vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, vk)))
        mask = cv2.bitwise_or(horizontal, vertical)
        close = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (max(5, hk // 2), max(3, vk // 3))))
        contours, _ = cv2.findContours(close, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            bbox = [x, y, x + w, y + h]
            if not _bbox_ok_general(bbox, W, H):
                continue
            gs, vr, hr = grid_score_for_bbox(binary, bbox)
            if vr >= 3 and hr >= 2:
                score = 240 + gs * 6 + 80 * (w * h / max(W * H, 1))
                proposals.append(_proposal_record(bbox, W, H, "cv_grid_multiscale", score, {"v_line_runs": vr, "h_line_runs": hr, "grid_score": gs, "horizontal_kernel": hk, "vertical_kernel": vk}))
    return proposals


def _color_region_proposals(rgb):
    H, W = rgb.shape[:2]
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    # Distance from the near-white page background plus saturation catches both pale and vivid heatmaps.
    white = np.array([255.0, 128.0, 128.0], dtype=np.float32)
    dist = np.linalg.norm(lab - white, axis=2)
    mask = ((dist > 20) | (hsv[:, :, 1] > 28)).astype(np.uint8) * 255
    # Suppress tiny text strokes, then connect adjacent cells.
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2)))
    for scale in [0.006, 0.012, 0.025]:
        kx = max(3, int(W * scale))
        ky = max(3, int(H * scale))
        connected = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (kx, ky)))
        contours, _ = cv2.findContours(connected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        proposals = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            bbox = [x, y, x + w, y + h]
            if not _bbox_ok_general(bbox, W, H):
                continue
            roi = mask[y:y+h, x:x+w]
            fill = float(np.mean(roi > 0))
            colors = rgb[y:y+h, x:x+w].reshape(-1, 3)
            if len(colors) > 12000:
                colors = colors[::max(1, len(colors) // 12000)]
            quantized = (colors // 24).astype(np.uint8)
            color_bins = len(np.unique(quantized, axis=0))
            if fill < 0.11 or color_bins < 4:
                continue
            score = 150 + 190 * fill + min(color_bins, 80) * 1.4
            proposals.append(_proposal_record(bbox, W, H, "cv_color_region", score, {"foreground_fill_ratio": round(fill, 4), "quantized_color_bins": int(color_bins), "closing_scale": scale}))
        if proposals:
            return proposals
    return []


def _edge_density_proposals(gray):
    H, W = gray.shape
    edges = cv2.Canny(gray, 45, 140)
    # Repeated cell boundaries create locally dense horizontal and vertical edges.
    density = cv2.blur((edges > 0).astype(np.float32), (max(9, W // 80), max(9, H // 80)))
    threshold = max(float(np.percentile(density, 91)), 0.045)
    mask = (density >= threshold).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (max(5, W // 100), max(5, H // 100))))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    proposals = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        pad_x, pad_y = max(3, w // 20), max(3, h // 20)
        bbox = _clip_bbox([x - pad_x, y - pad_y, x + w + pad_x, y + h + pad_y], W, H)
        if not _bbox_ok_general(bbox, W, H):
            continue
        roi = density[bbox[1]:bbox[3], bbox[0]:bbox[2]]
        d = float(np.mean(roi))
        if d < 0.035:
            continue
        proposals.append(_proposal_record(bbox, W, H, "cv_edge_density", 110 + d * 1000, {"mean_edge_density": round(d, 5)}))
    return proposals


def _qwen_localize_heatmaps_full_figure(figure_image_path, model_name="qwen3-vl-plus"):
    figure_stem = Path(figure_image_path).stem
    if globals().get('CACHE_EXISTING', True) and AI_INFERENCE_LOG_PATH.exists():
        try:
            entries = [json.loads(line) for line in AI_INFERENCE_LOG_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
            for entry in reversed(entries):
                if (entry.get("task") == "generic_full_figure_heatmap_localization"
                        and str(entry.get("figure")) == figure_stem
                        and entry.get("parse_success")
                        and entry.get("schema_valid")):
                    cached_path = Path(entry.get("parsed_json_path", ""))
                    if cached_path.exists():
                        parsed = load_json(cached_path)
                        print("Using cached full-figure localization:", cached_path)
                        return parsed, "[cached audited response]", {"cache_hit": True, "parsed_json_path": str(cached_path)}
        except Exception as cache_exc:
            print("Could not read localizer cache; calling Qwen:", cache_exc)
    prompt = """
You are locating scientific heatmaps or matrix-like colour grids in a complete figure.
Return JSON only. Inspect the whole figure, including portrait heatmaps, borderless heatmaps,
numeric matrices, binary matrices, and heatmaps embedded among other panels.

Coordinates must be normalized integers from 0 to 1000 relative to the full input image:
left=0, top=0, right=1000, bottom=1000.
Include only the heatmap/matrix body in bbox_norm; exclude titles, axes, and colorbars when possible.
Do not label ordinary line plots, scatter plots, bar charts, microscopy, photos, or legends as heatmaps.

Schema:
{
  "heatmaps": [
    {
      "bbox_norm": [left, top, right, bottom],
      "panel_id": "A or Not specified",
      "heatmap_kind": "numeric_grid | color_heatmap | binary_matrix | annotated_heatmap | unknown",
      "has_printed_cell_values": true,
      "has_visible_grid_lines": true,
      "confidence": "high | medium | low",
      "reason": "brief visual evidence"
    }
  ],
  "manual_review": false,
  "reason": ""
}
""".strip()
    image_url = image_to_data_url(figure_image_path)
    client = build_client()
    def invoke_once():
        response = client.chat.completions.create(
            model=model_name,
            temperature=AI_AUDIT_CONFIG["temperature"],
            top_p=AI_AUDIT_CONFIG["top_p"],
            messages=[{"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]}],
        )
        return response.choices[0].message.content or ""
    parsed, raw, audit = run_audited_json_inference(
        invoke_once,
        prompt=prompt,
        task_name="generic_full_figure_heatmap_localization",
        stage="Stage 3 generic localization",
        model=model_name,
        context={"paper": PAPER_CONFIG.get("paper_id"), "figure": Path(figure_image_path).stem, "panel": None},
    )
    return parsed, raw, audit


def _vlm_bbox_proposals(figure_image_path, W, H):
    proposals = []
    parsed, _, _ = _qwen_localize_heatmaps_full_figure(figure_image_path)
    for item in parsed.get("heatmaps", []) if isinstance(parsed, dict) else []:
        box = item.get("bbox_norm")
        if not isinstance(box, list) or len(box) != 4:
            continue
        try:
            bbox = [float(box[0]) * W / 1000, float(box[1]) * H / 1000, float(box[2]) * W / 1000, float(box[3]) * H / 1000]
        except Exception:
            continue
        bbox = _clip_bbox(bbox, W, H)
        if not _bbox_ok_general(bbox, W, H):
            continue
        confidence = str(item.get("confidence", "low")).lower()
        conf_score = {"high": 260, "medium": 210, "low": 150}.get(confidence, 150)
        proposals.append(_proposal_record(bbox, W, H, "qwen_full_figure_localizer", conf_score, item))
    return proposals


def _merge_heatmap_proposals(proposals, iou_threshold=0.45):
    # Fuse boxes that strongly overlap, while retaining detector provenance.
    ordered = sorted(proposals, key=lambda x: x["score"], reverse=True)
    groups = []
    for p in ordered:
        target = next((g for g in groups if iou(p["bbox"], g[0]["bbox"]) >= iou_threshold), None)
        if target is None:
            groups.append([p])
        else:
            target.append(p)
    merged = []
    for group in groups:
        weights = np.array([max(1.0, x["score"]) for x in group], dtype=float)
        boxes = np.array([x["bbox"] for x in group], dtype=float)
        fused = np.average(boxes, axis=0, weights=weights).round().astype(int).tolist()
        best = dict(group[0])
        best["bbox"] = fused
        best["width"] = fused[2] - fused[0]
        best["height"] = fused[3] - fused[1]
        best["area"] = best["width"] * best["height"]
        best["proposal_source"] = "+".join(sorted(set(x["proposal_source"] for x in group)))
        best["proposal_evidence"] = json.dumps([json.loads(x["proposal_evidence"]) for x in group], ensure_ascii=False)
        best["detector_agreement_count"] = len(set(x["proposal_source"] for x in group))
        best["score"] = float(max(x["score"] for x in group) + 35 * (best["detector_agreement_count"] - 1))
        merged.append(best)
    return sorted(merged, key=lambda x: x["score"], reverse=True)


def detect_numeric_matrix_candidates(figure_image_path, out_dir, figure_unit_id="figure", max_candidates=30, **kwargs):
    """Generic replacement retaining the original downstream manifest contract."""
    figure_image_path = Path(figure_image_path)
    out_dir = Path(out_dir)
    overlay_dir, core_dir, context_dir = out_dir / "overlay", out_dir / "core_candidates", out_dir / "context_candidates"
    for p in [overlay_dir, core_dir, context_dir]:
        ensure_dir(p)
    pil_img = Image.open(figure_image_path).convert("RGB")
    rgb = np.array(pil_img)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    H, W = gray.shape
    proposals = []
    if GENERIC_HEATMAP_CONFIG.get("use_grid_detector", True):
        proposals.extend(_grid_proposals_general(gray))
    if GENERIC_HEATMAP_CONFIG.get("use_color_region_detector", True):
        proposals.extend(_color_region_proposals(rgb))
    if GENERIC_HEATMAP_CONFIG.get("use_edge_density_detector", True):
        proposals.extend(_edge_density_proposals(gray))
    use_vlm = GENERIC_HEATMAP_CONFIG.get("use_vlm_full_figure_localizer", True)
    if GENERIC_HEATMAP_CONFIG.get("vlm_localizer_only_when_cv_empty", False):
        use_vlm = use_vlm and not proposals
    if use_vlm:
        try:
            proposals.extend(_vlm_bbox_proposals(figure_image_path, W, H))
        except Exception as exc:
            print("Full-figure Qwen localizer unavailable; continuing with CV proposals:", exc)
    candidates = _merge_heatmap_proposals(proposals, GENERIC_HEATMAP_CONFIG.get("candidate_nms_iou", 0.45))
    # Qwen-guided filtering prevents every low-specificity CV proposal from
    # becoming a separate paid candidate-validation call. When the full-figure
    # localizer found heatmaps, retain only fused proposals containing its box.
    # If it found none, retain only CV proposals supported by multiple detectors.
    if GENERIC_HEATMAP_CONFIG.get("vlm_guided_candidate_filter", True):
        vlm_candidates = [c for c in candidates if "qwen_full_figure_localizer" in c.get("proposal_source", "")]
        if vlm_candidates:
            candidates = vlm_candidates
        else:
            min_agreement = int(GENERIC_HEATMAP_CONFIG.get("min_detector_agreement_without_vlm", 2))
            candidates = [c for c in candidates if int(c.get("detector_agreement_count", 1)) >= min_agreement]
    limit = min(int(max_candidates), int(GENERIC_HEATMAP_CONFIG.get("max_candidates", 12)))
    candidates = candidates[:limit]

    overlay = pil_img.copy()
    draw = ImageDraw.Draw(overlay)
    records = []
    for candidate_id, c in enumerate(candidates):
        core_bbox = _clip_bbox(c["bbox"], W, H)
        context_bbox = expand_core_bbox_to_context(core_bbox, (W, H))
        core_path = core_dir / f"{figure_unit_id}_candidate_{candidate_id:02d}_core.png"
        context_path = context_dir / f"{figure_unit_id}_candidate_{candidate_id:02d}_context.png"
        crop_pil(pil_img, core_bbox).save(core_path)
        crop_pil(pil_img, context_bbox).save(context_path)
        draw.rectangle(core_bbox, outline="red", width=4)
        draw.text((core_bbox[0] + 4, core_bbox[1] + 4), f"{candidate_id}:{c['proposal_source']}", fill="red")
        draw.rectangle(context_bbox, outline="blue", width=2)
        rec = dict(c)
        rec.update({
            "candidate_id": candidate_id,
            "figure_unit_id": figure_unit_id,
            "figure_image_path": str(figure_image_path),
            "core_bbox": core_bbox,
            "context_bbox": context_bbox,
            "core_crop_path": str(core_path),
            "context_crop_path": str(context_path),
        })
        records.append(rec)
    overlay_path = overlay_dir / f"{figure_unit_id}_candidate_overlay.png"
    overlay.save(overlay_path)
    manifest_df = pd.DataFrame(records)
    manifest_csv = out_dir / f"{figure_unit_id}_candidate_manifest.csv"
    manifest_xlsx = out_dir / f"{figure_unit_id}_candidate_manifest.xlsx"
    manifest_df.to_csv(manifest_csv, index=False, encoding="utf-8-sig")
    manifest_df.to_excel(manifest_xlsx, index=False)
    return {
        "manifest_df": manifest_df,
        "manifest_csv": str(manifest_csv),
        "manifest_xlsx": str(manifest_xlsx),
        "overlay_path": str(overlay_path),
        "detector": "generic_heatmap_ensemble_v1",
    }


_generic_stage3_original = process_stage3_heatmaps_from_stage2
def process_stage3_heatmaps_from_stage2(*args, **kwargs):
    result = _generic_stage3_original(*args, **kwargs)
    qc = result.get("stage3_qc", {})
    zero_candidate_figures = [x.get("figure_unit_id") for x in result.get("figure_results", []) if int(x.get("candidate_count", 0)) == 0]
    zero_selected_figures = [x.get("figure_unit_id") for x in result.get("figure_results", []) if int(x.get("candidate_count", 0)) > 0 and int(x.get("selected_count", 0)) == 0]
    qc["generic_detector"] = "grid+color+edge+qwen_full_figure"
    qc["zero_candidate_figures"] = zero_candidate_figures
    qc["candidate_but_none_selected_figures"] = zero_selected_figures
    qc["manual_review_required_for_possible_missed_heatmaps"] = bool(zero_candidate_figures or zero_selected_figures)
    # Zero selected results are no longer silently reported as a successful heatmap search.
    if int(qc.get("total_selected_heatmap_contexts", 0)) == 0:
        qc["qc_pass"] = False
        qc["qc_failure_reason"] = "No heatmap was selected; inspect full-figure localization and candidate overlays."
    result["stage3_qc"] = qc
    save_json(result, Path(kwargs.get("out_root", args[1] if len(args) > 1 else RUN_ROOT / "stage3")) / "stage3_heatmap_summary.json")
    return result


In [ ]:

# Reviewer Comment 6 — audited Stage 3 Qwen call.
def ai_judge_core_matrix_candidate(core_image_path, context_image_path,
                                   model_name="qwen3-vl-plus", target_assay="ELISpot"):
    prompt = f"""
You are validating whether an image crop is suitable for structured extraction of a numeric heatmap / matrix.

You will receive two images in this order:
1) CORE IMAGE: the tight crop that should contain the numeric matrix / heatmap body.
2) CONTEXT IMAGE: a larger crop that may contain row labels, column labels, panel labels, nearby headers, captions, or unrelated neighboring content.

Target assay: {target_assay}

Important interpretation rule:
- Decide heatmap/matrix-body validity from the CORE IMAGE.
- Use the CONTEXT IMAGE only to decide whether useful labels, headers, or assay clues are available nearby.
- If a thin or irrelevant heatmap fragment appears only at the edge of the context image, but the core image is not itself a usable matrix body, reject it.

Return JSON only with exactly these fields:
{{
  "core_is_numeric_matrix_or_heatmap_body": true or false,
  "core_has_grid_cells": true or false,
  "core_has_cell_values_or_intensity": true or false,
  "context_has_interpretable_labels": true or false,
  "context_has_elispot_or_assay_text": true or false,
  "heatmap_is_only_edge_fragment": true or false,
  "main_core_content_is_non_heatmap": true or false,
  "is_suitable_for_stage4_extraction": true or false,
  "detected_assay": "ELISpot" or "FluoroSpot" or "ELISA" or "ICS" or "unknown",
  "detected_panel_id": "A" or "B" or "C" or null,
  "visible_row_labels_from_context": ["..."],
  "visible_column_labels_from_context": ["..."],
  "confidence": "high" or "medium" or "low",
  "manual_review": true or false,
  "reason": "brief explanation"
}}

Selection rule:
- is_suitable_for_stage4_extraction should be true only when the CORE IMAGE is actually the main numeric matrix / heatmap body.
- Contextual labels can support interpretation, but cannot rescue a wrong core crop.
"""
    client = build_client()
    core_url = image_to_data_url(core_image_path)
    context_url = image_to_data_url(context_image_path)
    def invoke_once():
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": core_url}},
                {"type": "image_url", "image_url": {"url": context_url}},
            ]}],
            temperature=AI_AUDIT_CONFIG["temperature"],
            top_p=AI_AUDIT_CONFIG["top_p"],
        )
        return response.choices[0].message.content or ""
    figure_hint = Path(core_image_path).stem
    parsed, raw, _audit = run_audited_json_inference(
        invoke_once, prompt=prompt, task_name="stage3_heatmap_candidate_validation",
        stage="Stage 3", model=model_name,
        context={"paper": PAPER_CONFIG.get("paper_id"), "figure": figure_hint, "panel": None},
    )
    if isinstance(parsed, dict):
        parsed["prompt_version"] = PROMPT_VERSIONS["stage3_heatmap_candidate_validation"]
        parsed["model_self_reported_confidence"] = parsed.get("confidence")
        parsed["uses_numeric_confidence_threshold"] = False
    return parsed, raw

_stage3_classify_original = classify_and_select_context_from_core_candidates
def classify_and_select_context_from_core_candidates(*args, **kwargs):
    result = _stage3_classify_original(*args, **kwargs)
    selected_df = result["selected_df"]
    audit_df = result["selection_df"]
    for df in [selected_df, audit_df]:
        if isinstance(df, pd.DataFrame):
            if "ai_confidence" in df.columns:
                df["model_self_reported_confidence"] = df["ai_confidence"]
            df["uses_numeric_confidence_threshold"] = False
            df["stage3_prompt_version"] = PROMPT_VERSIONS["stage3_heatmap_candidate_validation"]
    if isinstance(audit_df, pd.DataFrame):
        audit_df.to_csv(result["selection_csv"], index=False, encoding="utf-8-sig")
        audit_df.to_excel(result["selection_xlsx"], index=False)
    if isinstance(selected_df, pd.DataFrame):
        selected_df.to_csv(result["selected_csv"], index=False, encoding="utf-8-sig")
        selected_df.to_excel(result["selected_xlsx"], index=False)
    result["selection_df"] = audit_df
    result["selected_df"] = selected_df
    return result


In [ ]:
stage3_result = process_stage3_heatmaps_from_stage2(
    stage2_summary_path=str(RUN_ROOT / "stage2" / "stage2_summary.json"),
    out_root=str(RUN_ROOT / "stage3"),
    model_name="qwen3-vl-plus",
    target_assay=PAPER_CONFIG.get("target_assay", "ELISpot"),
    figure_unit_filter=PAPER_CONFIG.get("figure_unit_filter"),
    max_candidates=20,
)
print(stage3_result["stage3_qc"])
print(stage3_result["failed_figures"])
display_stage3_overlays(stage3_result)
display_stage3_selected_contexts(stage3_result)
inspect_stage3_selection_table(stage3_result)


In [ ]:
# ============================================================
# PART 1 / 3
# GENERIC STAGE 3B + STAGE 3C
# Basic helpers + input normalization + Stage 3B panel recovery
#
# This part keeps your original structure and adds:
# - full_legend priority fix:
#   if Stage 2 clean figure legend exists, use Stage 2 full_legend first.
# ============================================================

import os
import re
import ast
import json
import string
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display
from openai import OpenAI


# ============================================================
# 0. Global settings
# ============================================================

MODEL_NAME = "qwen3.7-plus"
CACHE_EXISTING = False


# ============================================================
# 1. Basic helpers
# ============================================================

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def clean_text(x):
    if x is None:
        return "Not specified"

    try:
        if pd.isna(x):
            return "Not specified"
    except Exception:
        pass

    x = str(x)
    x = x.replace("\u00ad", "")
    x = x.replace("\ufeff", "")
    x = x.replace("−", "-")
    x = x.replace("–", "-")
    x = x.replace("—", "-")
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()

    return x if x else "Not specified"


def is_not_specified(x):
    x = clean_text(x).upper()
    return x in ["", "NAN", "NONE", "NULL", "NOT SPECIFIED"]


def normalize_panel_id(x):
    x = clean_text(x).upper()

    if x in ["", "NAN", "NONE", "NULL", "NOT SPECIFIED"]:
        return "Not specified"

    m = re.search(r"\b([A-Z])\b", x)
    if m:
        return m.group(1).upper()

    return x


def get_or_default_series(df, col, default="Not specified"):
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


def save_json(obj, path):
    path = Path(path)
    ensure_dir(path.parent)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_json(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def json_to_dataframe_robust(obj):
    if isinstance(obj, list):
        return pd.DataFrame(obj)

    if isinstance(obj, dict):
        candidate_keys = [
            "figure_units",
            "stage2_figure_units",
            "matched_figure_units",
            "results",
            "figure_results",
            "records",
            "data"
        ]

        for key in candidate_keys:
            if key in obj and isinstance(obj[key], list):
                return pd.DataFrame(obj[key])

        found = []

        def walk(x):
            if isinstance(x, dict):
                keys = set(x.keys())
                if (
                    "figure_unit_id" in keys
                    or "full_legend" in keys
                    or "figure_label_raw" in keys
                ):
                    found.append(x)

                for v in x.values():
                    walk(v)

            elif isinstance(x, list):
                for item in x:
                    walk(item)

        walk(obj)

        if found:
            return pd.DataFrame(found)

    raise ValueError("Could not convert JSON to DataFrame.")


def read_table_robust(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if suffix == ".json":
        return json_to_dataframe_robust(load_json(path))

    raise ValueError(f"Unsupported file type: {path}")


def parse_bbox(x):
    if isinstance(x, list):
        return x

    if isinstance(x, tuple):
        return list(x)

    try:
        y = ast.literal_eval(str(x))
        if isinstance(y, (list, tuple)) and len(y) == 4:
            return [float(v) for v in y]
    except Exception:
        pass

    return None


def add_bbox_position(df, bbox_col="core_bbox"):
    df = df.copy()

    x1s, y1s, x2s, y2s, xcs, ycs = [], [], [], [], [], []

    for x in df[bbox_col]:
        bbox = parse_bbox(x)

        if bbox is None or len(bbox) != 4:
            x1s.append(np.nan)
            y1s.append(np.nan)
            x2s.append(np.nan)
            y2s.append(np.nan)
            xcs.append(np.nan)
            ycs.append(np.nan)
        else:
            x1, y1, x2, y2 = bbox
            x1s.append(x1)
            y1s.append(y1)
            x2s.append(x2)
            y2s.append(y2)
            xcs.append((x1 + x2) / 2)
            ycs.append((y1 + y2) / 2)

    df["_bbox_x1"] = x1s
    df["_bbox_y1"] = y1s
    df["_bbox_x2"] = x2s
    df["_bbox_y2"] = y2s
    df["_bbox_xcenter"] = xcs
    df["_bbox_ycenter"] = ycs

    return df


def sort_candidates_by_spatial_reading_order(df, bbox_col="core_bbox", row_tolerance_ratio=0.06):
    """
    Generic spatial sorting:
    1. Calculate bbox centers.
    2. Group candidates into visual rows by similar ycenter.
    3. Sort rows top-to-bottom.
    4. Sort candidates in each row left-to-right.
    """
    df = add_bbox_position(df, bbox_col=bbox_col).copy()

    if df.empty:
        return df

    valid_y = df["_bbox_ycenter"].dropna()

    if valid_y.empty:
        df["spatial_order_index"] = range(len(df))
        return df.reset_index(drop=True)

    y_min = valid_y.min()
    y_max = valid_y.max()
    y_range = max(1, y_max - y_min)

    row_tolerance = max(20, y_range * row_tolerance_ratio)

    df = df.sort_values("_bbox_ycenter").reset_index(drop=True)

    row_ids = []
    current_row = 0
    current_row_y_values = []

    for _, r in df.iterrows():
        y = r["_bbox_ycenter"]

        if pd.isna(y):
            row_ids.append(current_row)
            continue

        if not current_row_y_values:
            current_row_y_values.append(y)
            row_ids.append(current_row)
            continue

        current_row_mean = sum(current_row_y_values) / len(current_row_y_values)

        if abs(y - current_row_mean) <= row_tolerance:
            current_row_y_values.append(y)
            row_ids.append(current_row)
        else:
            current_row += 1
            current_row_y_values = [y]
            row_ids.append(current_row)

    df["_spatial_row_id"] = row_ids

    df = df.sort_values(
        ["_spatial_row_id", "_bbox_xcenter"],
        ascending=[True, True]
    ).reset_index(drop=True)

    df["spatial_order_index"] = range(len(df))

    return df


def extract_first_json(text):
    text = str(text).strip()

    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError("Cannot parse JSON from AI response.")


def build_client():
    api_key = os.getenv("DASHSCOPE_API_KEY")

    if not api_key:
        raise RuntimeError(
            "DASHSCOPE_API_KEY is not set. Run first:\n\n"
            "import os, getpass\n"
            "os.environ['DASHSCOPE_API_KEY'] = getpass.getpass('Paste DASHSCOPE_API_KEY: ')\n"
        )

    return OpenAI(
        api_key=api_key,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )


def call_qwen_json(prompt, model_name="qwen3.7-plus"):
    client = build_client()

    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        top_p=0.01,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw = response.choices[0].message.content
    parsed = extract_first_json(raw)

    return parsed, raw


# ============================================================
# 2. Stage 2 / Stage 3 input normalization
# ============================================================

def pick_first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_stage2_figure_units(stage2_df):
    df = stage2_df.copy()

    col_map = {
        "figure_unit_id": [
            "figure_unit_id",
            "Figure_Unit_ID",
            "figure_id",
            "Figure_ID"
        ],
        "figure_number": [
            "figure_number",
            "Figure_Number",
            "fig_number",
            "Fig_Number"
        ],
        "figure_label_raw": [
            "figure_label_raw",
            "Figure_Label_Raw",
            "figure_label",
            "Figure_Label"
        ],
        "full_legend": [
            "full_legend",
            "Full_Legend",
            "legend",
            "Legend",
            "figure_legend",
            "Figure_Legend"
        ],
        "source_doc": [
            "source_doc",
            "Source_Document",
            "Source_Doc",
            "document",
            "Document"
        ],
        "source_section": [
            "source_section",
            "Source_Section",
            "section",
            "Section"
        ],
        "source_page": [
            "source_page",
            "Source_Page",
            "page_num",
            "Page_Num",
            "page",
            "Page"
        ]
    }

    out = pd.DataFrame()

    for std_col, candidates in col_map.items():
        src_col = pick_first_existing_column(df, candidates)

        if src_col is not None:
            out[std_col] = df[src_col]
        else:
            out[std_col] = "Not specified"

    for col in out.columns:
        out[col] = out[col].apply(clean_text)

    out = out.drop_duplicates(
        subset=["figure_unit_id"],
        keep="first"
    ).reset_index(drop=True)

    return out


def infer_figure_number(full_legend, figure_unit_id):
    text = clean_text(full_legend) + " " + clean_text(figure_unit_id)

    patterns = [
        r"\bFig(?:ure)?\.?\s*([0-9]+[A-Za-z]?)",
        r"\bfigure[_\s-]*([0-9]+[A-Za-z]?)",
        r"\bfig[_\s-]*([0-9]+[A-Za-z]?)"
    ]

    for p in patterns:
        m = re.search(p, text, flags=re.I)
        if m:
            return clean_text(m.group(1))

    return "Not specified"


def choose_clean_stage2_first(row, main_col, stage2_col):
    """
    Important fix:
    For full_legend, prefer Stage 2 figure-level legend when available.
    Stage 3 full_legend may contain page text, body text, footer, or next section contamination.
    """
    stage2_value = clean_text(row.get(stage2_col, "Not specified"))
    main_value = clean_text(row.get(main_col, "Not specified"))

    if main_col == "full_legend":
        if stage2_value != "Not specified":
            return stage2_value
        return main_value

    if main_value != "Not specified":
        return main_value

    return stage2_value


def load_prepare_stage3_manifest(stage3_selected_manifest_path, stage2_summary_path=None):
    stage3_df = read_table_robust(stage3_selected_manifest_path)

    required_cols = [
        "figure_unit_id",
        "candidate_id",
        "core_bbox",
        "context_bbox",
        "core_crop_path",
        "selected_context_path"
    ]

    for col in required_cols:
        if col not in stage3_df.columns:
            raise KeyError(f"Stage 3 selected manifest must contain column: {col}")

    stage3_df["figure_unit_id"] = get_or_default_series(
        stage3_df,
        "figure_unit_id",
        "Not specified"
    ).apply(clean_text)

    stage3_df["candidate_id"] = get_or_default_series(
        stage3_df,
        "candidate_id",
        "Not specified"
    ).apply(clean_text)

    stage3_df["page_num"] = get_or_default_series(
        stage3_df,
        "page_num",
        "Not specified"
    ).apply(clean_text)

    stage3_df["full_legend"] = get_or_default_series(
        stage3_df,
        "full_legend",
        "Not specified"
    ).apply(clean_text)

    stage3_df["figure_number"] = get_or_default_series(
        stage3_df,
        "figure_number",
        "Not specified"
    ).apply(clean_text)

    stage3_df["source_doc"] = get_or_default_series(
        stage3_df,
        "source_doc",
        "Not specified"
    ).apply(clean_text)

    stage3_df["source_section"] = get_or_default_series(
        stage3_df,
        "source_section",
        "main"
    ).apply(clean_text)

    stage3_df["source_page"] = get_or_default_series(
        stage3_df,
        "source_page",
        "Not specified"
    ).apply(clean_text)

    stage3_df["ai_detected_panel_id"] = get_or_default_series(
        stage3_df,
        "ai_detected_panel_id",
        "Not specified"
    ).apply(normalize_panel_id)

    if "panel_id" not in stage3_df.columns:
        stage3_df["panel_id"] = stage3_df["ai_detected_panel_id"]
    else:
        stage3_df["panel_id"] = stage3_df["panel_id"].apply(normalize_panel_id)

    # Optional merge with Stage 2 summary.
    # Fix: full_legend uses Stage 2 first when Stage 2 legend exists.
    if stage2_summary_path is not None:
        stage2_raw_df = read_table_robust(stage2_summary_path)
        stage2_df = normalize_stage2_figure_units(stage2_raw_df)

        stage3_df = stage3_df.merge(
            stage2_df,
            on="figure_unit_id",
            how="left",
            suffixes=("", "_stage2")
        )

        fill_pairs = [
            ("full_legend", "full_legend_stage2"),
            ("figure_number", "figure_number_stage2"),
            ("source_doc", "source_doc_stage2"),
            ("source_section", "source_section_stage2"),
            ("source_page", "source_page_stage2")
        ]

        for main_col, stage2_col in fill_pairs:
            if stage2_col in stage3_df.columns:
                stage3_df[main_col] = stage3_df.apply(
                    lambda r: choose_clean_stage2_first(r, main_col, stage2_col),
                    axis=1
                )

    stage3_df["figure_number"] = stage3_df.apply(
        lambda r: clean_text(r["figure_number"])
        if clean_text(r["figure_number"]) != "Not specified"
        else infer_figure_number(
            full_legend=r.get("full_legend", "Not specified"),
            figure_unit_id=r.get("figure_unit_id", "Not specified")
        ),
        axis=1
    )

    stage3_df = stage3_df[
        stage3_df["selected_context_path"].astype(str).ne("Not selected")
    ].copy()

    return stage3_df.reset_index(drop=True)


# ============================================================
# 3. Stage 3B: qwen3.7-plus panel recovery
# ============================================================

def build_panel_recovery_prompt(figure_unit_id, page_num, figure_number, full_legend, candidates):
    candidates_json = json.dumps(candidates, ensure_ascii=False, indent=2)

    return f"""
You are recovering panel IDs for ELISpot heatmap candidates from a scientific figure.

Return ONLY valid JSON.
Do not output markdown or explanation outside JSON.

You are given:
1. The full figure legend.
2. A list of detected heatmap candidates from the figure.
3. Each candidate has candidate_id, existing panel id if available, and core_bbox position.
4. You cannot see the image. Use only the full legend and candidate spatial order.

Goal:
1. Decide which figure panels are ELISpot-related panels.
2. Map each heatmap candidate to its corresponding panel_id.

Important rules:
- Use ONLY the provided full legend and candidate list.
- Do NOT use outside knowledge.
- Do NOT use main text.
- Do NOT guess when evidence is insufficient.
- Identify panels that are explicitly described as ELISpot, immunospot, spot-forming, cytokine ELISpot, IFN-gamma ELISpot, or equivalent ELISpot readout.
- A range marker gives shared context, but the ELISpot-specific panel is the panel marker attached to the ELISpot-related clause.
- When multiple ELISpot panels are found and candidate count matches ELISpot panel count, assign candidates by candidate spatial order and ELISpot panel order in the legend.
- Existing detected panel IDs can be kept only when consistent with the legend.
- If assignment is unsafe, return assigned_panel_id = "Not specified" and manual_review = true.

Current figure:
- figure_unit_id: {figure_unit_id}
- page_num: {page_num}
- figure_number: {figure_number}

FULL_LEGEND:
{full_legend}

CANDIDATES_SORTED_BY_SPATIAL_READING_ORDER:
{candidates_json}

Return JSON schema exactly:
{{
  "figure_unit_id": "{figure_unit_id}",
  "page_num": "{page_num}",
  "figure_number": "{figure_number}",
  "expected_elispot_panels": [
    {{
      "panel_id": "single panel letter or Not specified",
      "panel_specific_legend_clause": "exact ELISpot-related clause or Not specified",
      "evidence": "short evidence phrase or Not specified",
      "manual_review": false,
      "review_reason": ""
    }}
  ],
  "candidate_assignments": [
    {{
      "candidate_id": "candidate id as string",
      "spatial_order_index": 0,
      "assigned_panel_id": "single panel letter or Not specified",
      "assignment_method": "kept_existing_stage3_panel_id | legend_elispot_panel_order_plus_candidate_spatial_order | not_assigned",
      "confidence": "high | medium | low",
      "manual_review": false,
      "review_reason": "",
      "panel_specific_legend_clause": "exact ELISpot-related clause or Not specified",
      "evidence": "short evidence phrase or Not specified"
    }}
  ],
  "overall_manual_review": false,
  "overall_review_reason": ""
}}
""".strip()


def normalize_panel_recovery_json(parsed):
    out = dict(parsed)

    if not isinstance(out.get("expected_elispot_panels"), list):
        out["expected_elispot_panels"] = []

    if not isinstance(out.get("candidate_assignments"), list):
        out["candidate_assignments"] = []

    out["overall_manual_review"] = str(
        out.get("overall_manual_review", False)
    ).lower() in ["true", "1", "yes"]

    out["overall_review_reason"] = clean_text(out.get("overall_review_reason", ""))

    return out


def ai_recover_panels_for_one_figure(g, out_root, model_name="qwen3.7-plus"):
    out_root = Path(out_root)

    figure_unit_id = clean_text(g["figure_unit_id"].iloc[0])
    page_num = clean_text(g["page_num"].iloc[0])
    figure_number = clean_text(g["figure_number"].iloc[0])
    full_legend = clean_text(g["full_legend"].iloc[0])

    g_sorted = sort_candidates_by_spatial_reading_order(
        g.copy(),
        bbox_col="core_bbox"
    )

    candidates = []

    for _, r in g_sorted.iterrows():
        candidates.append({
            "candidate_id": clean_text(r.get("candidate_id", "Not specified")),
            "spatial_order_index": int(r.get("spatial_order_index", 0)),
            "core_bbox": clean_text(r.get("core_bbox", "Not specified")),
            "context_bbox": clean_text(r.get("context_bbox", "Not specified")),
            "bbox_ycenter": None if pd.isna(r.get("_bbox_ycenter")) else float(r.get("_bbox_ycenter")),
            "bbox_xcenter": None if pd.isna(r.get("_bbox_xcenter")) else float(r.get("_bbox_xcenter")),
            "existing_ai_detected_panel_id": clean_text(r.get("ai_detected_panel_id", "Not specified")),
            "existing_panel_id": clean_text(r.get("panel_id", "Not specified")),
            "ai_detected_assay": clean_text(r.get("ai_detected_assay", "Not specified")),
            "selected_context_path": clean_text(r.get("selected_context_path", "Not specified"))
        })

    safe_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        f"{figure_unit_id}_panel_recovery"
    )

    raw_path = out_root / "ai_raw" / f"{safe_name}_raw.txt"
    json_path = out_root / "ai_json" / f"{safe_name}.json"

    if CACHE_EXISTING and json_path.exists():
        parsed = load_json(json_path)
        raw = raw_path.read_text(encoding="utf-8") if raw_path.exists() else ""
    else:
        prompt = build_panel_recovery_prompt(
            figure_unit_id=figure_unit_id,
            page_num=page_num,
            figure_number=figure_number,
            full_legend=full_legend,
            candidates=candidates
        )

        parsed, raw = call_qwen_json(
            prompt=prompt,
            model_name=model_name
        )

        ensure_dir(raw_path.parent)
        raw_path.write_text(str(raw), encoding="utf-8")
        save_json(parsed, json_path)

    parsed = normalize_panel_recovery_json(parsed)

    assignment_rows = []

    for a in parsed.get("candidate_assignments", []):
        assignment_rows.append({
            "figure_unit_id": figure_unit_id,
            "page_num": page_num,
            "figure_number": figure_number,
            "candidate_id": clean_text(a.get("candidate_id", "Not specified")),
            "spatial_order_index": a.get("spatial_order_index", "Not specified"),
            "assigned_panel_id": normalize_panel_id(a.get("assigned_panel_id", "Not specified")),
            "panel_assignment_method": clean_text(a.get("assignment_method", "Not specified")),
            "panel_assignment_confidence": clean_text(a.get("confidence", "Not specified")),
            "panel_assignment_manual_review": str(
                a.get("manual_review", False)
            ).lower() in ["true", "1", "yes"],
            "panel_assignment_review_reason": clean_text(a.get("review_reason", "")),
            "panel_assignment_specific_clause": clean_text(
                a.get("panel_specific_legend_clause", "Not specified")
            ),
            "panel_assignment_evidence": clean_text(a.get("evidence", "Not specified")),
            "expected_elispot_panels_json": json.dumps(
                parsed.get("expected_elispot_panels", []),
                ensure_ascii=False
            ),
            "panel_recovery_overall_manual_review": bool(
                parsed.get("overall_manual_review", False)
            ),
            "panel_recovery_overall_review_reason": clean_text(
                parsed.get("overall_review_reason", "")
            ),
            "panel_recovery_json_path": str(json_path),
            "panel_recovery_raw_path": str(raw_path)
        })

    assignment_df = pd.DataFrame(assignment_rows)

    return assignment_df


def run_stage3b_panel_recovery(stage3_df, out_root, model_name="qwen3.7-plus"):
    out_root = Path(out_root)

    all_assignments = []
    failed = []

    for figure_unit_id, g in stage3_df.groupby("figure_unit_id", dropna=False):
        try:
            assignment_df = ai_recover_panels_for_one_figure(
                g=g.copy(),
                out_root=out_root,
                model_name=model_name
            )
            all_assignments.append(assignment_df)

        except Exception as e:
            failed.append({
                "figure_unit_id": clean_text(figure_unit_id),
                "error": str(e)
            })
            print("Panel recovery failed:", failed[-1])

    if all_assignments:
        assignment_df = pd.concat(all_assignments, ignore_index=True)
    else:
        assignment_df = pd.DataFrame()

    failed_df = pd.DataFrame(failed)

    recovered_df = stage3_df.copy()

    if not assignment_df.empty:
        recovered_df = recovered_df.merge(
            assignment_df,
            on=["figure_unit_id", "page_num", "figure_number", "candidate_id"],
            how="left"
        )
    else:
        recovered_df["assigned_panel_id"] = "Not specified"
        recovered_df["panel_assignment_method"] = "not_assigned"
        recovered_df["panel_assignment_confidence"] = "Not specified"
        recovered_df["panel_assignment_manual_review"] = True
        recovered_df["panel_assignment_review_reason"] = "Panel recovery AI returned no assignments."
        recovered_df["panel_assignment_specific_clause"] = "Not specified"
        recovered_df["panel_assignment_evidence"] = "Not specified"

    def choose_recovered_panel(row):
        assigned = normalize_panel_id(row.get("assigned_panel_id", "Not specified"))
        existing = normalize_panel_id(row.get("panel_id", "Not specified"))

        if assigned != "Not specified":
            return assigned

        if existing != "Not specified":
            return existing

        return "Not specified"

    recovered_df["panel_id_original"] = recovered_df["panel_id"]
    recovered_df["ai_detected_panel_id_original"] = recovered_df["ai_detected_panel_id"]

    recovered_df["recovered_panel_id"] = recovered_df.apply(
        choose_recovered_panel,
        axis=1
    )

    recovered_df["panel_id"] = recovered_df["recovered_panel_id"]
    recovered_df["ai_detected_panel_id"] = recovered_df["recovered_panel_id"]

    recovered_df["panel_recovery_method"] = recovered_df["panel_assignment_method"]
    recovered_df["panel_recovery_confidence"] = recovered_df["panel_assignment_confidence"]

    recovered_df["panel_recovery_manual_review"] = recovered_df.apply(
        lambda r: (
            normalize_panel_id(r.get("recovered_panel_id", "Not specified")) == "Not specified"
            or str(r.get("panel_assignment_manual_review", False)).lower() in ["true", "1", "yes"]
        ),
        axis=1
    )

    recovered_df["panel_recovery_reason"] = recovered_df.apply(
        lambda r: clean_text(r.get("panel_assignment_review_reason", ""))
        if clean_text(r.get("panel_assignment_review_reason", "")) != "Not specified"
        else clean_text(r.get("panel_assignment_evidence", "")),
        axis=1
    )

    return recovered_df, assignment_df, failed_df

# ============================================================
# PART 2 / 3
# Stage 3C-A: rule-based legend context extraction
#
# Main fix:
# - If panel is inside a range such as (G to I), extract the complete range context.
# - Single markers inside the range, such as (G), (H), (I), do NOT truncate the range.
# - If no range applies, extract only the single-panel clause.
# - panel_assignment_specific_clause is only used as fallback when rule extraction fails.
# ============================================================

def protect_abbreviations(text):
    text = clean_text(text)

    protected_pairs = {
        "Fig.": "Fig<DOT>",
        "fig.": "fig<DOT>",
        "Figs.": "Figs<DOT>",
        "figs.": "figs<DOT>",
        "e.g.": "e<DOT>g<DOT>",
        "i.e.": "i<DOT>e<DOT>",
        "vs.": "vs<DOT>",
        "et al.": "et al<DOT>",
        "s.c.": "s<DOT>c<DOT>",
        "i.m.": "i<DOT>m<DOT>",
        "i.d.": "i<DOT>d<DOT>",
        "i.v.": "i<DOT>v<DOT>",
        "i.p.": "i<DOT>p<DOT>",
        "p.o.": "p<DOT>o<DOT>"
    }

    for k, v in protected_pairs.items():
        text = text.replace(k, v)

    return text, protected_pairs


def restore_abbreviations(text, protected_pairs):
    for k, v in protected_pairs.items():
        text = text.replace(v, k)

    return clean_text(text)


def expand_panel_range(start_panel, end_panel):
    start_panel = normalize_panel_id(start_panel)
    end_panel = normalize_panel_id(end_panel)

    letters = list(string.ascii_uppercase)

    if start_panel not in letters or end_panel not in letters:
        return []

    s = letters.index(start_panel)
    e = letters.index(end_panel)

    if s > e:
        return []

    return letters[s:e + 1]


def parse_group_marker_to_panels(marker_text):
    """
    Generic parser for markers inside parentheses.

    Supports:
      A to C
      A-C
      A, B, and C
      A and B

    Returns list of panel letters.
    Single-panel marker returns [] because it is not range context.
    """
    marker_text = clean_text(marker_text).upper()
    marker_text = marker_text.replace("–", "-").replace("—", "-")

    m_range = re.search(r"\b([A-Z])\b\s*(?:TO|-)\s*\b([A-Z])\b", marker_text)

    if m_range:
        return expand_panel_range(m_range.group(1), m_range.group(2))

    letters = re.findall(r"\b([A-Z])\b", marker_text)

    if len(letters) <= 1:
        return []

    seen = []
    for x in letters:
        if x not in seen:
            seen.append(x)

    return seen


def get_figure_level_title_rule(full_legend):
    text = clean_text(full_legend)

    first_marker = re.search(r"\([A-Z][^()]*\)", text)

    if not first_marker:
        return text

    return clean_text(text[:first_marker.start()])


def trim_page_footer_noise(text):
    """
    Generic cleanup for obvious PDF footer/header contamination.
    This does not remove real figure legend content before the footer.
    """
    text = clean_text(text)

    footer_patterns = [
        r"\bDownloaded from\b",
        r"\bScience Advances\s*\|\s*Research Article\b",
        r"\bFan et al\.,\s*Sci\.",
        r"\b\d+\s+of\s+\d+\b"
    ]

    cut_positions = []

    for p in footer_patterns:
        m = re.search(p, text, flags=re.I)
        if m:
            cut_positions.append(m.start())

    if cut_positions:
        text = text[:min(cut_positions)]

    return clean_text(text)


def trim_trailing_global_legend_notes(text):
    """
    Remove global statistical/footer notes after biological panel descriptions.

    This prevents panel-specific context from becoming:
      (I). Data were presented as means ± SD ...
    """
    text = clean_text(text)

    global_patterns = [
        r"\bData (?:represent|were presented)\b",
        r"\bOne-way\b",
        r"\bTwo-way\b",
        r"\bStudent'?s t\b",
        r"\bANOVA\b",
        r"\(\*P\s*<",
        r"\bns,\s*not significant\b"
    ]

    cut_positions = []

    for p in global_patterns:
        m = re.search(p, text, flags=re.I)
        if m:
            cut_positions.append(m.start())

    if cut_positions:
        text = text[:min(cut_positions)]

    return clean_text(text)


def find_full_range_context_for_panel(full_legend, panel_id):
    """
    If target panel belongs to a multi-panel range such as (G to I),
    return the COMPLETE range context from that range marker
    until the next multi-panel range marker.

    Single-panel markers such as (G), (H), (I) inside the range
    must NOT truncate the range context.
    """
    panel_id = normalize_panel_id(panel_id)
    text = trim_page_footer_noise(full_legend)

    if panel_id == "Not specified" or text == "Not specified":
        return "Not specified", "Not specified"

    matches = list(re.finditer(r"\(([^()]+)\)\.?", text))
    range_matches = []

    for m in matches:
        marker_text = clean_text(m.group(1))
        panels = parse_group_marker_to_panels(marker_text)

        if len(panels) >= 2:
            range_matches.append({
                "match": m,
                "marker_text": marker_text,
                "panels": panels
            })

    for i, item in enumerate(range_matches):
        if panel_id not in item["panels"]:
            continue

        start = item["match"].start()

        if i + 1 < len(range_matches):
            end = range_matches[i + 1]["match"].start()
        else:
            end = len(text)

        range_context = clean_text(text[start:end])
        range_context = trim_trailing_global_legend_notes(range_context)

        return range_context, clean_text(item["marker_text"])

    return "Not specified", "Not specified"


def extract_panel_clause_from_context(context_text, panel_id):
    """
    Extract target panel-specific clause from a context block.

    Supports both:
      marker-start style:
        (H). ELISpot assay ...

      marker-end style:
        IFN-γ ELISpot assay ... (I).
    """
    panel_id = normalize_panel_id(panel_id)
    text = clean_text(context_text)

    if panel_id == "Not specified" or text == "Not specified":
        return "Not specified"

    target_marker = rf"\({panel_id}\)"

    text_protected, protected_pairs = protect_abbreviations(text)

    # Most figure legends separate panel clauses using semicolons.
    clauses = re.split(r"\s*;\s*", text_protected)

    matched = []

    for clause in clauses:
        clause = restore_abbreviations(clause, protected_pairs)
        clause = clean_text(clause)

        if not re.search(target_marker, clause):
            continue

        first_marker = re.search(r"\(([^()]+)\)", clause)

        if first_marker:
            panels = parse_group_marker_to_panels(first_marker.group(1))

            # Avoid treating pure range marker such as (G to I) as panel-specific.
            if len(panels) >= 2:
                after_first_marker = clause[first_marker.end():]
                if not re.search(target_marker, after_first_marker):
                    continue

        m = re.search(target_marker, clause)

        if m:
            # Cut after the target marker to avoid carrying statistics after (I).
            clause = clean_text(clause[:m.end()] + ".")

        matched.append(clause)

    if matched:
        keywords = [
            "elispot",
            "immunospot",
            "spot-forming",
            "spot forming",
            "sfc",
            "sfcs",
            "sfu",
            "ifn-γ",
            "ifn-g",
            "ifng",
            "interferon",
            "splenocyte",
            "splenocytes"
        ]

        for clause in matched:
            if any(k in clause.lower() for k in keywords):
                return clean_text(clause)

        return clean_text(matched[0])

    return "Not specified"


def extract_single_panel_context(full_legend, panel_id):
    """
    Used only when target panel does NOT belong to a multi-panel range.
    Extract only the single-panel clause.
    """
    panel_id = normalize_panel_id(panel_id)
    text = trim_page_footer_noise(full_legend)

    if panel_id == "Not specified" or text == "Not specified":
        return "Not specified"

    # First try semicolon-based extraction.
    panel_clause = extract_panel_clause_from_context(text, panel_id)

    if panel_clause != "Not specified":
        return panel_clause

    # Fallback for marker-start style:
    # (H). ELISpot assay ...
    target_marker = rf"\({panel_id}\)"
    m = re.search(target_marker, text)

    if not m:
        return "Not specified"

    start = m.start()

    next_marker = re.search(
        r"\([A-Z](?:\s*(?:to|-|,|and)\s*[A-Z])?\)\.?",
        text[m.end():]
    )

    if next_marker:
        end = m.end() + next_marker.start()
    else:
        end = len(text)

    context = clean_text(text[start:end])
    context = trim_trailing_global_legend_notes(context)

    return context


def get_range_context_for_panel_rule(full_legend, panel_id):
    """
    Compatibility wrapper for your original function name.
    """
    return find_full_range_context_for_panel(
        full_legend=full_legend,
        panel_id=panel_id
    )


def get_panel_specific_context_rule(full_legend, panel_id, ai_specific_hint="Not specified"):
    """
    Compatibility wrapper for your original function name.

    Main rule:
    - If panel belongs to a range, extract the target panel clause from that full range.
    - If no range applies, extract the single-panel context.
    - Use ai_specific_hint only as fallback.
    """
    panel_id = normalize_panel_id(panel_id)
    ai_specific_hint = clean_text(ai_specific_hint)

    if panel_id == "Not specified":
        return "Not specified"

    panel_range_context, _ = find_full_range_context_for_panel(
        full_legend=full_legend,
        panel_id=panel_id
    )

    if panel_range_context != "Not specified":
        panel_specific_context = extract_panel_clause_from_context(
            context_text=panel_range_context,
            panel_id=panel_id
        )
    else:
        panel_specific_context = extract_single_panel_context(
            full_legend=full_legend,
            panel_id=panel_id
        )

    if panel_specific_context == "Not specified" and ai_specific_hint != "Not specified":
        return ai_specific_hint

    return panel_specific_context


def build_rule_based_panel_legend_context(full_legend, panel_id, ai_specific_hint="Not specified"):
    """
    Final rule:
    1. If the target panel belongs to a range such as (G to I),
       extract the whole range context.
    2. If no range applies, extract only the single-panel clause.
    3. panel_assignment_specific_clause is only a fallback when rule extraction fails.
    """
    panel_id = normalize_panel_id(panel_id)
    full_legend = clean_text(full_legend)
    ai_specific_hint = clean_text(ai_specific_hint)

    figure_level_title = get_figure_level_title_rule(full_legend)

    panel_range_context, panel_range_marker = find_full_range_context_for_panel(
        full_legend=full_legend,
        panel_id=panel_id
    )

    if panel_range_context != "Not specified":
        panel_specific_context = extract_panel_clause_from_context(
            context_text=panel_range_context,
            panel_id=panel_id
        )
        extraction_method = "rule_based_full_range_context"
    else:
        panel_specific_context = extract_single_panel_context(
            full_legend=full_legend,
            panel_id=panel_id
        )
        extraction_method = "rule_based_single_panel_context"

    # Fallback only, not the main logic.
    if panel_specific_context == "Not specified" and ai_specific_hint != "Not specified":
        panel_specific_context = ai_specific_hint
        extraction_method = extraction_method + "_plus_panel_recovery_fallback"

    parts = []

    if figure_level_title != "Not specified":
        parts.append("[FIGURE_LEVEL_TITLE] " + figure_level_title)

    if panel_range_context != "Not specified":
        parts.append("[PANEL_RANGE_CONTEXT] " + panel_range_context)

    if panel_specific_context != "Not specified":
        parts.append("[PANEL_SPECIFIC_CONTEXT] " + panel_specific_context)

    combined = clean_text(" ".join(parts))

    return {
        "figure_level_title": figure_level_title,
        "panel_range_marker": panel_range_marker,
        "panel_range_legend_context": panel_range_context,
        "panel_specific_legend_context": panel_specific_context,
        "combined_panel_legend_context": combined,
        "legend_context_extraction_method": extraction_method
    }


def run_stage3c_a_context_extraction(recovered_df):
    rows = []

    for _, row in recovered_df.iterrows():
        panel_id = normalize_panel_id(row.get("panel_id", "Not specified"))
        full_legend = clean_text(row.get("full_legend", "Not specified"))
        ai_specific_hint = clean_text(
            row.get("panel_assignment_specific_clause", "Not specified")
        )

        context = build_rule_based_panel_legend_context(
            full_legend=full_legend,
            panel_id=panel_id,
            ai_specific_hint=ai_specific_hint
        )

        out_row = dict(row)
        out_row.update(context)

        out_row["legend_context_manual_review"] = (
            panel_id == "Not specified"
            or context["combined_panel_legend_context"] == "Not specified"
            or context["panel_specific_legend_context"] == "Not specified"
        )

        if panel_id == "Not specified":
            reason = "panel_id is Not specified."
        elif context["combined_panel_legend_context"] == "Not specified":
            reason = "combined_panel_legend_context is Not specified."
        elif context["panel_specific_legend_context"] == "Not specified":
            reason = "panel_specific_legend_context is Not specified."
        else:
            reason = ""

        out_row["legend_context_review_reason"] = reason

        rows.append(out_row)

    return pd.DataFrame(rows)

# ============================================================
# PART 3 / 3
# Stage 3C-B: legend field extraction + clean handoff + main runner
#
# Main fixes:
# - Prompt explicitly prevents Mouse_Tumor_Model from being inferred from vaccine names alone.
# - Full audit outputs are preserved.
# - A clean Stage3C_Clean_Handoff_Manifest is added for Stage 4.
# ============================================================

LEGEND_FIELDS = [
    "Legend_Readout",
    "Legend_Sample_or_Tissue",
    "Legend_Treatment",
    "Legend_Dose",
    "Legend_Route",
    "Legend_N_Mice",
    "Legend_Mouse_Tumor_Model",
    "Legend_Mouse_Strain",
    "Legend_Vaccine_or_Antigen_Set",
    "Legend_ELISpot_Unit"
]


def build_legend_field_extraction_prompt(row):
    figure_unit_id = clean_text(row.get("figure_unit_id", "Not specified"))
    page_num = clean_text(row.get("page_num", "Not specified"))
    figure_number = clean_text(row.get("figure_number", "Not specified"))
    candidate_id = clean_text(row.get("candidate_id", "Not specified"))
    target_panel_id = normalize_panel_id(row.get("panel_id", "Not specified"))

    figure_level_title = clean_text(row.get("figure_level_title", "Not specified"))
    panel_range_context = clean_text(row.get("panel_range_legend_context", "Not specified"))
    panel_specific_context = clean_text(row.get("panel_specific_legend_context", "Not specified"))
    combined_context = clean_text(row.get("combined_panel_legend_context", "Not specified"))

    return f"""
You are extracting structured ELISpot assay and experimental-condition metadata from figure legend context.

Return ONLY valid JSON.
Do not output markdown or explanation outside JSON.

Target record:
- figure_unit_id: {figure_unit_id}
- page_num: {page_num}
- figure_number: {figure_number}
- candidate_id: {candidate_id}
- target_panel_id: {target_panel_id}

You are given only the legend context that applies to the target panel.

FIGURE_LEVEL_TITLE:
{figure_level_title}

PANEL_RANGE_CONTEXT:
{panel_range_context}

PANEL_SPECIFIC_CONTEXT:
{panel_specific_context}

COMBINED_CONTEXT:
{combined_context}

Extraction rules:
1. Use ONLY the provided legend context.
2. Do NOT use outside knowledge.
3. Do NOT use main text.
4. Do NOT guess missing values.
5. Return "Not specified" when information is absent or ambiguous.
6. Prefer PANEL_SPECIFIC_CONTEXT for panel-specific fields.
7. Use PANEL_RANGE_CONTEXT only when it clearly applies to the target panel.
8. Do not assign information that is tied only to another specific panel.
9. If PANEL_RANGE_CONTEXT comes from a range marker such as "(A to C)", "(D to F)", or "(G to I)", it applies to the target panel only when the target panel is inside that range.
10. Keep field values concise and close to the provided wording.
11. For every extracted field, provide source and evidence.

Field-specific rules:
12. Legend_Readout:
    - If the context explicitly contains "ELISpot assay" or "ELISpot", extract "ELISpot assay" as Legend_Readout.
    - If the context explicitly contains a cytokine or immune readout together with ELISpot, extract the more specific readout.
    - Examples of specific readouts include IFN-γ ELISpot assay, IL-2 ELISpot assay, or granzyme B ELISpot assay when explicitly stated.
    - If the context says antigen-induced or neoantigen-induced T cell activation together with ELISpot assay, this supports Legend_Readout = "ELISpot assay".
    - Do not return "Not specified" for Legend_Readout when "ELISpot" is explicitly present.

13. Legend_Sample_or_Tissue:
    - Extract only explicitly stated sample or tissue terms.
    - Examples include splenocytes, PBMCs, lymph nodes, tumor tissue, blood, serum, lung, spleen, draining lymph nodes, or isolated T cells when explicitly stated.
    - If the context explicitly contains "splenocyte" or "splenocytes", extract "splenocytes".
    - If different samples or tissues are mentioned for different panels, use only the sample or tissue tied to the target panel.
    - Do not return "Not specified" when a sample or tissue is explicitly stated for the target panel.

14. Legend_Treatment:
    - Extract explicitly stated treatment, immunization, vaccination, stimulation, infection, challenge, drug, formulation, or experimental condition.
    - If a named vaccine, formulation, antigen, peptide pool, virus, drug, or treatment group is explicitly linked to immunization, vaccination, stimulation, or treatment, extract that name.
    - Do not infer treatment from unrelated context.
    - If treatment is only implied and not named, return "Not specified".

15. Legend_Vaccine_or_Antigen_Set:
    - Extract the specific vaccine, antigen, peptide pool, epitope set, antigen set, neoantigen set, or immunization formulation explicitly named in the context.
    - The value should be copied from the legend wording as closely as possible.
    - Do not invent vaccine names.
    - Do not use outside knowledge to map a treatment name to an antigen set.

16. Legend_Dose:
    - Dose must be an explicit amount, concentration, frequency, or dose-group label.
    - Examples include "1 μg", "10 μg", "30 μg", "100 μg", "low dose", "high dose", "dose groups", or "different dosage groups" when explicitly stated.
    - If the context only says vaccination, immunization, or treatment without dose wording, return "Not specified".

17. Legend_Route:
    - Route means explicit administration route wording.
    - Examples include s.c., i.m., i.d., i.v., i.p., subcutaneous, intramuscular, intradermal, intravenous, or intraperitoneal when explicitly stated.
    - If no administration route is explicitly stated for the target panel, return "Not specified".

18. Legend_N_Mice:
    - Extract mouse number only when explicitly stated.
    - Examples include "n = 5", "n = 10 mice per group", "five mice per group", or "10 mice per route".
    - N_Mice can be inherited from shared range experimental design only when clearly applicable to the target panel.
    - If no mouse number is stated, return "Not specified".

19. Legend_Mouse_Tumor_Model:
    - Extract only when an explicit tumor model, tumor cell model, metastasis model, tumor-bearing model, or challenge model is stated.
    - Do not infer Mouse_Tumor_Model from vaccine names, antigen names, or formulation names alone.
    - If a tumor-related term such as tumor, tumor-bearing, metastasis, tumor challenge, tumor cells, or cell-line challenge is absent, return "Not specified".

20. Legend_Mouse_Strain:
    - Extract only explicit mouse strain names.
    - Examples include BALB/c, C57BL/6, C57BL/6J, DBA/2, or other strain names when explicitly stated.
    - If no strain is stated, return "Not specified".

21. Legend_ELISpot_Unit:
    - Extract only an explicit measurement unit or normalization description.
    - Examples include spots, SFC, SFU, spots per 10^6 cells, spot-forming cells per 10^6 cells, or spot-forming units when explicitly stated.
    - If no unit is stated, return "Not specified".

22. Timepoint and axis-context rules:
    - Timepoint words such as "indicated time points", "different time points", "days after immunization", or "hours after stimulation" describe axis/context.
    - Do not treat timepoint information as Dose or Route.
    - If specific timepoints are listed in the legend, preserve them only as timepoint/context information, not as treatment, dose, or route.

Return JSON schema exactly:
{{
  "Legend_Readout": "string or Not specified",
  "Legend_Sample_or_Tissue": "string or Not specified",
  "Legend_Treatment": "string or Not specified",
  "Legend_Dose": "string or Not specified",
  "Legend_Route": "string or Not specified",
  "Legend_N_Mice": "string or Not specified",
  "Legend_Mouse_Tumor_Model": "string or Not specified",
  "Legend_Mouse_Strain": "string or Not specified",
  "Legend_Vaccine_or_Antigen_Set": "string or Not specified",
  "Legend_ELISpot_Unit": "string or Not specified",

  "Legend_Manual_Review": false,
  "Legend_Review_Reason": "",

  "Legend_Field_Sources": {{
    "Legend_Readout": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_Sample_or_Tissue": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_Treatment": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_Dose": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_Route": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_N_Mice": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_Mouse_Tumor_Model": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_Mouse_Strain": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_Vaccine_or_Antigen_Set": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified",
    "Legend_ELISpot_Unit": "figure_level_title | panel_range_legend_context | panel_specific_legend_context | combined_panel_legend_context | Not specified"
  }},

  "Legend_Field_Evidence": {{
    "Legend_Readout": "exact supporting phrase or Not specified",
    "Legend_Sample_or_Tissue": "exact supporting phrase or Not specified",
    "Legend_Treatment": "exact supporting phrase or Not specified",
    "Legend_Dose": "exact supporting phrase or Not specified",
    "Legend_Route": "exact supporting phrase or Not specified",
    "Legend_N_Mice": "exact supporting phrase or Not specified",
    "Legend_Mouse_Tumor_Model": "exact supporting phrase or Not specified",
    "Legend_Mouse_Strain": "exact supporting phrase or Not specified",
    "Legend_Vaccine_or_Antigen_Set": "exact supporting phrase or Not specified",
    "Legend_ELISpot_Unit": "exact supporting phrase or Not specified"
  }}
}}
""".strip()


def normalize_legend_field_json(parsed):
    out = dict(parsed)

    for f in LEGEND_FIELDS:
        out[f] = clean_text(out.get(f, "Not specified"))

    out["Legend_Manual_Review"] = str(
        out.get("Legend_Manual_Review", False)
    ).lower() in ["true", "1", "yes"]

    out["Legend_Review_Reason"] = clean_text(
        out.get("Legend_Review_Reason", "")
    )

    if not isinstance(out.get("Legend_Field_Sources"), dict):
        out["Legend_Field_Sources"] = {}

    if not isinstance(out.get("Legend_Field_Evidence"), dict):
        out["Legend_Field_Evidence"] = {}

    return out


def ai_extract_fields_for_one_row(row, out_root, model_name="qwen3.7-plus"):
    out_root = Path(out_root)

    figure_unit_id = clean_text(row.get("figure_unit_id", "Not specified"))
    candidate_id = clean_text(row.get("candidate_id", "Not specified"))
    panel_id = normalize_panel_id(row.get("panel_id", "Not specified"))

    safe_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        f"{figure_unit_id}_candidate_{candidate_id}_panel_{panel_id}_legend_fields"
    )

    raw_path = out_root / "ai_raw" / f"{safe_name}_raw.txt"
    json_path = out_root / "ai_json" / f"{safe_name}.json"

    combined_context = clean_text(row.get("combined_panel_legend_context", "Not specified"))

    if panel_id == "Not specified" or combined_context == "Not specified":
        parsed = {
            f: "Not specified" for f in LEGEND_FIELDS
        }
        parsed["Legend_Manual_Review"] = True
        parsed["Legend_Review_Reason"] = "panel_id or combined_panel_legend_context is Not specified."
        parsed["Legend_Field_Sources"] = {}
        parsed["Legend_Field_Evidence"] = {}
        raw = json.dumps(parsed, ensure_ascii=False)

        ensure_dir(raw_path.parent)
        raw_path.write_text(raw, encoding="utf-8")
        save_json(parsed, json_path)

    elif CACHE_EXISTING and json_path.exists():
        parsed = load_json(json_path)
        raw = raw_path.read_text(encoding="utf-8") if raw_path.exists() else ""

    else:
        prompt = build_legend_field_extraction_prompt(row)
        parsed, raw = call_qwen_json(
            prompt=prompt,
            model_name=model_name
        )

        ensure_dir(raw_path.parent)
        raw_path.write_text(str(raw), encoding="utf-8")
        save_json(parsed, json_path)

    parsed = normalize_legend_field_json(parsed)

    out_row = dict(row)

    sources = parsed.get("Legend_Field_Sources", {})
    evidence = parsed.get("Legend_Field_Evidence", {})

    for f in LEGEND_FIELDS:
        out_row[f] = clean_text(parsed.get(f, "Not specified"))
        out_row[f + "_Source"] = clean_text(sources.get(f, "Not specified"))
        out_row[f + "_Evidence"] = clean_text(evidence.get(f, "Not specified"))

    out_row["Legend_Manual_Review"] = bool(parsed.get("Legend_Manual_Review", False))
    out_row["Legend_Review_Reason"] = clean_text(parsed.get("Legend_Review_Reason", ""))
    out_row["Legend_AI_JSON_Path"] = str(json_path)
    out_row["Legend_AI_Raw_Path"] = str(raw_path)
    out_row["Legend_Extraction_Method"] = "qwen3.7_plus_combined_panel_legend_context_only"

    return out_row


def run_stage3c_b_field_extraction(context_df, out_root, model_name="qwen3.7-plus"):
    rows = []
    failed = []

    for idx, row in context_df.iterrows():
        try:
            out_row = ai_extract_fields_for_one_row(
                row=row.to_dict(),
                out_root=out_root,
                model_name=model_name
            )
            rows.append(out_row)

        except Exception as e:
            failed.append({
                "row_index": int(idx),
                "figure_unit_id": clean_text(row.get("figure_unit_id", "Not specified")),
                "candidate_id": clean_text(row.get("candidate_id", "Not specified")),
                "panel_id": clean_text(row.get("panel_id", "Not specified")),
                "error": str(e)
            })
            print("Legend field extraction failed:", failed[-1])

    legend_df = pd.DataFrame(rows)
    failed_df = pd.DataFrame(failed)

    return legend_df, failed_df


# ============================================================
# Clean handoff table for Stage 4
# ============================================================

def build_stage3c_clean_handoff(legend_df):
    keep_cols = [
        "figure_unit_id",
        "source_doc",
        "source_section",
        "source_page",
        "page_num",
        "figure_number",
        "candidate_id",
        "core_bbox",
        "context_bbox",
        "core_crop_path",
        "context_crop_path",
        "selected_context_path",

        "panel_id",
        "recovered_panel_id",

        "figure_level_title",
        "panel_range_marker",
        "panel_range_legend_context",
        "panel_specific_legend_context",
        "combined_panel_legend_context",

        "Legend_Readout",
        "Legend_Sample_or_Tissue",
        "Legend_Treatment",
        "Legend_Dose",
        "Legend_Route",
        "Legend_N_Mice",
        "Legend_Mouse_Tumor_Model",
        "Legend_Mouse_Strain",
        "Legend_Vaccine_or_Antigen_Set",
        "Legend_ELISpot_Unit",

        "Legend_Manual_Review",
        "Legend_Review_Reason",
        "legend_context_manual_review",
        "legend_context_review_reason",

        "Legend_Readout_Source",
        "Legend_Readout_Evidence",
        "Legend_Sample_or_Tissue_Source",
        "Legend_Sample_or_Tissue_Evidence",
        "Legend_Treatment_Source",
        "Legend_Treatment_Evidence",
        "Legend_Dose_Source",
        "Legend_Dose_Evidence",
        "Legend_Route_Source",
        "Legend_Route_Evidence",
        "Legend_N_Mice_Source",
        "Legend_N_Mice_Evidence",
        "Legend_Mouse_Tumor_Model_Source",
        "Legend_Mouse_Tumor_Model_Evidence",
        "Legend_Vaccine_or_Antigen_Set_Source",
        "Legend_Vaccine_or_Antigen_Set_Evidence"
    ]

    df = legend_df.copy()

    for c in keep_cols:
        if c not in df.columns:
            df[c] = "Not specified"

    if "recovered_panel_id" in df.columns:
        df["panel_id"] = df["recovered_panel_id"].apply(clean_text)

    clean_df = df[keep_cols].copy()
    clean_df = clean_df.loc[:, ~clean_df.columns.duplicated()].copy()

    for c in clean_df.columns:
        clean_df[c] = clean_df[c].apply(clean_text)

    return clean_df


# ============================================================
# Main runner
# ============================================================

def run_stage3b3c_panel_recovery_and_legend_extraction(
    stage3_selected_manifest_path,
    stage2_summary_path=None,
    out_root="./stage3b3c_panel_legend_outputs",
    model_name="qwen3.7-plus",
    figure_unit_filter=None,
    limit=None
):
    out_root = Path(out_root)

    for d in [
        out_root,
        out_root / "ai_raw",
        out_root / "ai_json"
    ]:
        ensure_dir(d)

    # ----------------------------
    # Load Stage 3 input
    # ----------------------------
    stage3_df = load_prepare_stage3_manifest(
        stage3_selected_manifest_path=stage3_selected_manifest_path,
        stage2_summary_path=stage2_summary_path
    )

    if figure_unit_filter is not None:
        if isinstance(figure_unit_filter, str):
            figure_unit_filter = [figure_unit_filter]

        stage3_df = stage3_df[
            stage3_df["figure_unit_id"].astype(str).isin(figure_unit_filter)
        ].copy()

    if limit is not None:
        stage3_df = stage3_df.head(limit).copy()

    print("Input Stage 3 selected heatmaps:", len(stage3_df))

    display_cols = [
        "figure_unit_id",
        "page_num",
        "figure_number",
        "candidate_id",
        "core_bbox",
        "context_bbox",
        "ai_detected_panel_id",
        "panel_id",
        "ai_detected_assay",
        "core_crop_path",
        "selected_context_path",
        "full_legend"
    ]

    display(stage3_df[[c for c in display_cols if c in stage3_df.columns]])

    # ----------------------------
    # Stage 3B
    # ----------------------------
    recovered_df, assignment_df, panel_failed_df = run_stage3b_panel_recovery(
        stage3_df=stage3_df,
        out_root=out_root,
        model_name=model_name
    )

    # ----------------------------
    # Stage 3C-A
    # ----------------------------
    context_df = run_stage3c_a_context_extraction(
        recovered_df=recovered_df
    )

    # ----------------------------
    # Stage 3C-B
    # ----------------------------
    legend_df, legend_failed_df = run_stage3c_b_field_extraction(
        context_df=context_df,
        out_root=out_root,
        model_name=model_name
    )

    # ----------------------------
    # Clean handoff for Stage 4
    # ----------------------------
    clean_handoff_df = build_stage3c_clean_handoff(legend_df)

    # ----------------------------
    # Save outputs
    # ----------------------------
    recovered_csv = out_root / "Stage3B_Recovered_Stage3_Selected_Manifest.csv"
    recovered_xlsx = out_root / "Stage3B_Recovered_Stage3_Selected_Manifest.xlsx"

    assignment_csv = out_root / "Stage3B_Panel_Recovery_Assignments.csv"
    assignment_xlsx = out_root / "Stage3B_Panel_Recovery_Assignments.xlsx"

    context_csv = out_root / "Stage3C_A_Legend_Context_Manifest.csv"
    context_xlsx = out_root / "Stage3C_A_Legend_Context_Manifest.xlsx"

    legend_csv = out_root / "Stage3C_B_Panel_Legend_Field_Manifest.csv"
    legend_xlsx = out_root / "Stage3C_B_Panel_Legend_Field_Manifest.xlsx"

    clean_handoff_csv = out_root / "Stage3C_Clean_Handoff_Manifest.csv"
    clean_handoff_xlsx = out_root / "Stage3C_Clean_Handoff_Manifest.xlsx"

    panel_failed_csv = out_root / "Stage3B_Panel_Recovery_Failed_Items.csv"
    panel_failed_xlsx = out_root / "Stage3B_Panel_Recovery_Failed_Items.xlsx"

    legend_failed_csv = out_root / "Stage3C_B_Legend_Field_Failed_Items.csv"
    legend_failed_xlsx = out_root / "Stage3C_B_Legend_Field_Failed_Items.xlsx"

    combined_xlsx = out_root / "Stage3B3C_Combined_Output.xlsx"
    qc_json = out_root / "Stage3B3C_QC_Summary.json"

    recovered_df.to_csv(recovered_csv, index=False, encoding="utf-8-sig")
    recovered_df.to_excel(recovered_xlsx, index=False)

    assignment_df.to_csv(assignment_csv, index=False, encoding="utf-8-sig")
    assignment_df.to_excel(assignment_xlsx, index=False)

    context_df.to_csv(context_csv, index=False, encoding="utf-8-sig")
    context_df.to_excel(context_xlsx, index=False)

    legend_df.to_csv(legend_csv, index=False, encoding="utf-8-sig")
    legend_df.to_excel(legend_xlsx, index=False)

    clean_handoff_df.to_csv(clean_handoff_csv, index=False, encoding="utf-8-sig")
    clean_handoff_df.to_excel(clean_handoff_xlsx, index=False)

    panel_failed_df.to_csv(panel_failed_csv, index=False, encoding="utf-8-sig")
    panel_failed_df.to_excel(panel_failed_xlsx, index=False)

    legend_failed_df.to_csv(legend_failed_csv, index=False, encoding="utf-8-sig")
    legend_failed_df.to_excel(legend_failed_xlsx, index=False)

    qc = {
        "stage3_selected_manifest_path": str(stage3_selected_manifest_path),
        "stage2_summary_path": str(stage2_summary_path) if stage2_summary_path is not None else "Not specified",
        "model_name": model_name,
        "cache_existing": CACHE_EXISTING,
        "input_heatmap_count": int(len(stage3_df)),
        "recovered_manifest_count": int(len(recovered_df)),
        "panel_assignment_count": int(len(assignment_df)),
        "context_manifest_count": int(len(context_df)),
        "legend_field_manifest_count": int(len(legend_df)),
        "clean_handoff_count": int(len(clean_handoff_df)),
        "panel_recovery_failed_count": int(len(panel_failed_df)),
        "legend_field_failed_count": int(len(legend_failed_df)),
        "panel_recovery_manual_review_count": int(
            recovered_df["panel_recovery_manual_review"]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
            .sum()
        ) if not recovered_df.empty and "panel_recovery_manual_review" in recovered_df.columns else 0,
        "legend_context_manual_review_count": int(
            context_df["legend_context_manual_review"]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
            .sum()
        ) if not context_df.empty and "legend_context_manual_review" in context_df.columns else 0,
        "legend_field_manual_review_count": int(
            legend_df["Legend_Manual_Review"]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
            .sum()
        ) if not legend_df.empty and "Legend_Manual_Review" in legend_df.columns else 0,
        "records_by_panel": legend_df["panel_id"].value_counts(dropna=False).to_dict()
            if not legend_df.empty and "panel_id" in legend_df.columns else {},
        "outputs": {
            "recovered_csv": str(recovered_csv),
            "recovered_xlsx": str(recovered_xlsx),
            "assignment_csv": str(assignment_csv),
            "assignment_xlsx": str(assignment_xlsx),
            "context_csv": str(context_csv),
            "context_xlsx": str(context_xlsx),
            "legend_csv": str(legend_csv),
            "legend_xlsx": str(legend_xlsx),
            "clean_handoff_csv": str(clean_handoff_csv),
            "clean_handoff_xlsx": str(clean_handoff_xlsx),
            "panel_failed_csv": str(panel_failed_csv),
            "panel_failed_xlsx": str(panel_failed_xlsx),
            "legend_failed_csv": str(legend_failed_csv),
            "legend_failed_xlsx": str(legend_failed_xlsx),
            "combined_xlsx": str(combined_xlsx),
            "qc_json": str(qc_json)
        }
    }

    save_json(qc, qc_json)

    with pd.ExcelWriter(combined_xlsx, engine="openpyxl") as writer:
        recovered_df.to_excel(writer, sheet_name="Stage3B_Recovered", index=False)
        assignment_df.to_excel(writer, sheet_name="Stage3B_Assignments", index=False)
        context_df.to_excel(writer, sheet_name="Stage3C_A_Context", index=False)
        legend_df.to_excel(writer, sheet_name="Stage3C_B_Fields", index=False)
        clean_handoff_df.to_excel(writer, sheet_name="Clean_Handoff", index=False)
        panel_failed_df.to_excel(writer, sheet_name="Panel_Failed", index=False)
        legend_failed_df.to_excel(writer, sheet_name="Legend_Failed", index=False)

        pd.DataFrame([
            {
                "Metric": k,
                "Value": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
            }
            for k, v in qc.items()
        ]).to_excel(writer, sheet_name="QC", index=False)

    print("\n" + "=" * 120)
    print("Stage 3B + Stage 3C complete.")
    print("Input heatmaps:", len(stage3_df))
    print("Recovered manifest:", recovered_csv)
    print("Context manifest:", context_xlsx)
    print("Legend field manifest:", legend_xlsx)
    print("Clean handoff:", clean_handoff_xlsx)
    print("Combined:", combined_xlsx)
    print("QC:", qc_json)

    print("\nStage 3B panel recovery preview:")
    panel_preview_cols = [
        "figure_unit_id",
        "page_num",
        "candidate_id",
        "core_bbox",
        "panel_id_original",
        "ai_detected_panel_id_original",
        "assigned_panel_id",
        "panel_id",
        "panel_recovery_method",
        "panel_recovery_confidence",
        "panel_recovery_manual_review",
        "panel_recovery_reason"
    ]
    display(recovered_df[[c for c in panel_preview_cols if c in recovered_df.columns]])

    print("\nStage 3C-A context preview:")
    context_preview_cols = [
        "figure_unit_id",
        "page_num",
        "candidate_id",
        "panel_id",
        "figure_level_title",
        "panel_range_marker",
        "panel_range_legend_context",
        "panel_specific_legend_context",
        "combined_panel_legend_context",
        "legend_context_extraction_method",
        "legend_context_manual_review",
        "legend_context_review_reason"
    ]
    display(context_df[[c for c in context_preview_cols if c in context_df.columns]])

    print("\nStage 3C-B field extraction preview:")
    legend_preview_cols = [
        "figure_unit_id",
        "page_num",
        "figure_number",
        "candidate_id",
        "panel_id",
        "Legend_Readout",
        "Legend_Sample_or_Tissue",
        "Legend_Treatment",
        "Legend_Dose",
        "Legend_Route",
        "Legend_N_Mice",
        "Legend_Mouse_Tumor_Model",
        "Legend_Mouse_Strain",
        "Legend_Vaccine_or_Antigen_Set",
        "Legend_ELISpot_Unit",
        "Legend_Manual_Review",
        "Legend_Review_Reason",
        "core_crop_path",
        "selected_context_path"
    ]
    display(legend_df[[c for c in legend_preview_cols if c in legend_df.columns]])

    print("\nClean handoff preview:")
    display(clean_handoff_df.head())

    print("\nPanel recovery failed:")
    display(panel_failed_df)

    print("\nLegend field extraction failed:")
    display(legend_failed_df)

    return {
        "stage3_input_df": stage3_df,
        "recovered_df": recovered_df,
        "assignment_df": assignment_df,
        "context_df": context_df,
        "legend_df": legend_df,
        "clean_handoff_df": clean_handoff_df,
        "panel_failed_df": panel_failed_df,
        "legend_failed_df": legend_failed_df,
        "qc": qc,
        "recovered_csv": recovered_csv,
        "recovered_xlsx": recovered_xlsx,
        "assignment_csv": assignment_csv,
        "assignment_xlsx": assignment_xlsx,
        "context_csv": context_csv,
        "context_xlsx": context_xlsx,
        "legend_csv": legend_csv,
        "legend_xlsx": legend_xlsx,
        "clean_handoff_csv": clean_handoff_csv,
        "clean_handoff_xlsx": clean_handoff_xlsx,
        "panel_failed_csv": panel_failed_csv,
        "panel_failed_xlsx": panel_failed_xlsx,
        "legend_failed_csv": legend_failed_csv,
        "legend_failed_xlsx": legend_failed_xlsx,
        "combined_xlsx": combined_xlsx,
        "qc_json": qc_json
    }

In [ ]:

# Reviewer Comment 6 — task-aware audit wrapper for Stage 3B/3C.
_call_qwen_json_unversioned = call_qwen_json

def _infer_stage3bc_task(prompt):
    text = str(prompt)
    if "candidate_assignments" in text or "Panel recovery" in text or "panel recovery" in text:
        return "stage3b_panel_recovery", "Stage 3B"
    if "full_legend" in text or "stimulation" in text or "readout" in text:
        return "stage3c_legend_field_extraction", "Stage 3C"
    return "generic_qwen_json", "Stage 3B/3C"

def call_qwen_json(prompt, model_name="qwen3.7-plus"):
    task_name, stage = _infer_stage3bc_task(prompt)
    match = re.search(r"Figure(?: unit)?(?: ID)?\s*[:=]\s*([^\n]+)", str(prompt), flags=re.I)
    figure_hint = match.group(1).strip() if match else None
    def invoke_once():
        client = build_client()
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=AI_AUDIT_CONFIG["temperature"],
            top_p=AI_AUDIT_CONFIG["top_p"],
        )
        return response.choices[0].message.content or ""
    parsed, raw, _audit = run_audited_json_inference(
        invoke_once, prompt=prompt, task_name=task_name, stage=stage, model=model_name,
        context={"paper": PAPER_CONFIG.get("paper_id"), "figure": figure_hint, "panel": None},
    )
    if isinstance(parsed, dict):
        parsed["prompt_version"] = PROMPT_VERSIONS.get(task_name)
        parsed["uses_numeric_confidence_threshold"] = False
    return parsed, raw

_ai_recover_panels_original = ai_recover_panels_for_one_figure
def ai_recover_panels_for_one_figure(*args, **kwargs):
    result = _ai_recover_panels_original(*args, **kwargs)
    if isinstance(result, pd.DataFrame):
        confidence_col = next((c for c in ["panel_assignment_confidence", "confidence"] if c in result.columns), None)
        if confidence_col:
            result["model_self_reported_confidence"] = result[confidence_col]
        result["uses_numeric_confidence_threshold"] = False
        result["prompt_version"] = PROMPT_VERSIONS["stage3b_panel_recovery"]
    return result

_ai_extract_fields_original = ai_extract_fields_for_one_row
def ai_extract_fields_for_one_row(*args, **kwargs):
    result = _ai_extract_fields_original(*args, **kwargs)
    if isinstance(result, dict):
        result["uses_numeric_confidence_threshold"] = False
        result["prompt_version"] = PROMPT_VERSIONS["stage3c_legend_field_extraction"]
    return result


In [ ]:
STAGE3_SELECTED_MANIFEST_PATH = RUN_ROOT / "stage3" / "stage3_selected_heatmap_context_manifest.csv"
STAGE2_SUMMARY_PATH = RUN_ROOT / "stage2" / "stage2_summary.json"

stage3b3c_result = run_stage3b3c_panel_recovery_and_legend_extraction(
    stage3_selected_manifest_path=str(STAGE3_SELECTED_MANIFEST_PATH),
    stage2_summary_path=str(STAGE2_SUMMARY_PATH),
    out_root=str(RUN_ROOT / "stage3b3c"),
    model_name="qwen3.7-plus",
    figure_unit_filter=PAPER_CONFIG.get("figure_unit_filter"),
    limit=None,
)
print("Stage 3B/3C output:", stage3b3c_result["combined_xlsx"])
print("QC:", stage3b3c_result["qc_json"])


In [24]:
import os
import re
import json
import base64
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from openai import OpenAI


MODEL_NAME = "qwen3-vl-plus"
CACHE_EXISTING = False


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def clean_text(x):
    if x is None:
        return "Not specified"
    try:
        if pd.isna(x):
            return "Not specified"
    except Exception:
        pass
    x = str(x)
    x = x.replace("\u00ad", "")
    x = x.replace("\ufeff", "")
    x = x.replace("−", "-")
    x = x.replace("–", "-")
    x = x.replace("—", "-")
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else "Not specified"


def safe_float(x):
    if x is None:
        return np.nan
    try:
        return float(str(x).replace(",", "").strip())
    except Exception:
        return np.nan


def read_table_robust(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path}")


def save_json(obj, path):
    path = Path(path)
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_json(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_client():
    api_key = os.getenv("DASHSCOPE_API_KEY")
    if not api_key:
        raise RuntimeError(
            "DASHSCOPE_API_KEY is not set. Run this first:\n\n"
            "import os, getpass\n"
            "os.environ['DASHSCOPE_API_KEY'] = getpass.getpass('Paste DASHSCOPE_API_KEY: ')\n"
        )
    return OpenAI(
        api_key=api_key,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )


def image_to_data_url(image_path):
    image_path = Path(image_path)
    ext = image_path.suffix.lower().replace(".", "")
    if ext == "jpg":
        ext = "jpeg"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/{ext};base64,{b64}"


def extract_first_json(text):
    text = str(text).strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])
    raise ValueError("Cannot parse JSON from AI response")


def call_ai_json_with_image(image_path, prompt, model_name):
    client = build_client()
    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        top_p=0.01,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_data_url(image_path)
                        }
                    }
                ]
            }
        ]
    )
    raw = response.choices[0].message.content
    parsed = extract_first_json(raw)
    return parsed, raw


STAGE3C_CONTEXT_COLUMNS = [
    "panel_id",
    "target_panel_id",
    "figure_number",
    "figure_level_title",
    "panel_range_marker",
    "panel_range_legend_context",
    "panel_specific_legend_context",
    "combined_panel_legend_context",
    "legend_context_extraction_method",
    "legend_context_manual_review",
    "legend_context_review_reason"
]

STAGE3C_LEGEND_FIELD_COLUMNS = [
    "Legend_Readout",
    "Legend_Sample_or_Tissue",
    "Legend_Treatment",
    "Legend_Dose",
    "Legend_Route",
    "Legend_N_Mice",
    "Legend_Mouse_Tumor_Model",
    "Legend_Mouse_Strain",
    "Legend_Vaccine_or_Antigen_Set",
    "Legend_ELISpot_Unit",
    "Legend_Manual_Review",
    "Legend_Review_Reason"
]

STAGE3C_PROVENANCE_COLUMNS = [
    "source_doc",
    "source_section",
    "source_page",
    "full_legend",
    "panel_id_original",
    "ai_detected_panel_id_original",
    "recovered_panel_id",
    "assigned_panel_id",
    "panel_recovery_method",
    "panel_recovery_confidence",
    "panel_recovery_manual_review",
    "panel_recovery_reason",
    "panel_assignment_specific_clause",
    "panel_assignment_evidence"
]


def ensure_stage3c_passthrough_columns(df):
    df = df.copy()
    for col in STAGE3C_CONTEXT_COLUMNS + STAGE3C_LEGEND_FIELD_COLUMNS + STAGE3C_PROVENANCE_COLUMNS:
        if col not in df.columns:
            df[col] = "Not specified"
    df["panel_id"] = df["panel_id"].apply(clean_text)
    df["target_panel_id"] = df["target_panel_id"].apply(clean_text)
    return df


def collect_stage3c_metadata_for_stage4(meta):
    metadata = {}
    keep_cols = STAGE3C_CONTEXT_COLUMNS + STAGE3C_LEGEND_FIELD_COLUMNS + STAGE3C_PROVENANCE_COLUMNS
    for base in STAGE3C_LEGEND_FIELD_COLUMNS:
        keep_cols.append(base + "_Source")
        keep_cols.append(base + "_Evidence")
    for col in keep_cols:
        metadata[col] = clean_text(meta.get(col, "Not specified"))
    return metadata


def merge_close_lines(lines, min_dist=8):
    if not lines:
        return []
    lines = sorted(lines)
    merged = [[lines[0]]]
    for x in lines[1:]:
        if x - merged[-1][-1] <= min_dist:
            merged[-1].append(x)
        else:
            merged.append([x])
    return [int(np.mean(g)) for g in merged]


def get_line_centers_from_projection(proj, min_gap=6, threshold_ratio=0.35):
    if len(proj) == 0:
        return []
    threshold = max(np.max(proj) * threshold_ratio, np.percentile(proj, 90))
    idx = np.where(proj >= threshold)[0]
    if len(idx) == 0:
        return []
    groups = []
    current = [idx[0]]
    for i in idx[1:]:
        if i - current[-1] <= min_gap:
            current.append(i)
        else:
            groups.append(current)
            current = [i]
    groups.append(current)
    return [int(np.mean(g)) for g in groups]


def draw_tool_grid_overlay(gray, horizontal_lines, vertical_lines, out_path):
    img = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    for y in horizontal_lines:
        cv2.line(img, (0, y), (img.shape[1], y), (255, 0, 0), 2)
    for x in vertical_lines:
        cv2.line(img, (x, 0), (x, img.shape[0]), (0, 0, 255), 2)
    ensure_dir(Path(out_path).parent)
    Image.fromarray(img).save(out_path)
    return str(out_path)


def detect_core_grid_shape(core_image_path, out_dir=None, safe_name="core", scale=4, debug=True):
    core_image_path = Path(core_image_path)
    img = Image.open(core_image_path).convert("RGB")
    img = img.resize((img.width * scale, img.height * scale), Image.Resampling.LANCZOS)
    arr = np.array(img)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    binary_inv = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        12
    )
    h, w = binary_inv.shape
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(25, w // 25), 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(20, h // 8)))
    horizontal = cv2.morphologyEx(binary_inv, cv2.MORPH_OPEN, horizontal_kernel)
    vertical = cv2.morphologyEx(binary_inv, cv2.MORPH_OPEN, vertical_kernel)
    h_proj = horizontal.sum(axis=1)
    v_proj = vertical.sum(axis=0)
    horizontal_lines = get_line_centers_from_projection(
        h_proj,
        min_gap=max(4, h // 250),
        threshold_ratio=0.30
    )
    vertical_lines = get_line_centers_from_projection(
        v_proj,
        min_gap=max(4, w // 250),
        threshold_ratio=0.30
    )
    horizontal_lines = merge_close_lines(horizontal_lines, min_dist=max(5, h // 120))
    vertical_lines = merge_close_lines(vertical_lines, min_dist=max(5, w // 160))
    tool_row_count = max(0, len(horizontal_lines) - 1)
    tool_column_count = max(0, len(vertical_lines) - 1)
    overlay_path = "Not generated"
    if out_dir is not None:
        overlay_dir = Path(out_dir) / "tool_grid_overlays"
        ensure_dir(overlay_dir)
        overlay_path = overlay_dir / f"{safe_name}_tool_grid_overlay.png"
        draw_tool_grid_overlay(
            gray=gray,
            horizontal_lines=horizontal_lines,
            vertical_lines=vertical_lines,
            out_path=overlay_path
        )
    if debug:
        print("Tool horizontal grid lines:", horizontal_lines)
        print("Tool vertical grid lines:", vertical_lines)
        print("Tool row_count:", tool_row_count)
        print("Tool column_count:", tool_column_count)
        print("Tool overlay:", overlay_path)
        if overlay_path != "Not generated" and Path(overlay_path).exists():
            display(Image.open(overlay_path))
    return {
        "tool_row_count": int(tool_row_count),
        "tool_column_count": int(tool_column_count),
        "horizontal_lines": horizontal_lines,
        "vertical_lines": vertical_lines,
        "scale": scale,
        "tool_grid_overlay_path": str(overlay_path)
    }


def build_core_matrix_prompt_fixed_shape(tool_row_count, tool_column_count):
    return f"""
You are extracting a numeric matrix from a scientific heatmap core crop.

Return ONLY valid JSON.
Do not output markdown or explanation outside JSON.

Important:
- The grid structure has already been detected by a tool.
- The matrix has exactly {tool_row_count} rows and {tool_column_count} columns.
- Do NOT decide the row count or column count yourself.
- Do NOT add extra rows or columns.
- Do NOT include colorbar ticks, border lines, margins, or neighboring content as matrix cells.
- Your task is ONLY to read the printed cell values inside this fixed-size grid.

Reading rule:
- Read cells from top to bottom, left to right.
- Use zero-based row_index and col_index.
- The output matrix must contain exactly {tool_row_count} rows.
- Each row must contain exactly {tool_column_count} values.
- The records list must contain exactly {tool_row_count * tool_column_count} records.
- If a value is unreadable, use null and mark that cell manual_review=true.
- Do not infer values from color intensity if printed numbers are visible.
- Do not invent row labels or column labels.

Return JSON schema exactly:
{{
  "matrix_detected": true,
  "row_count": {tool_row_count},
  "column_count": {tool_column_count},
  "read_direction": "top_to_bottom_left_to_right",
  "manual_review": false,
  "review_reason": "",
  "matrix": [
    []
  ],
  "records": [
    {{
      "row_index": 0,
      "col_index": 0,
      "value_text": "1",
      "value_numeric": 1,
      "manual_review": false,
      "review_reason": ""
    }}
  ]
}}
""".strip()


def build_context_axis_prompt_fixed_shape(tool_row_count, tool_column_count):
    return f"""
You are analyzing a scientific figure crop that contains a heatmap / numeric matrix and surrounding context.

Return ONLY valid JSON.
Do not output markdown or explanation outside JSON.

Task:
Extract x-axis and y-axis labels for the heatmap matrix.

Important:
- A tool has detected that the core heatmap matrix has exactly {tool_row_count} rows and {tool_column_count} columns.
- Your task is NOT to extract cell values.
- Your task is to extract labels that map row_index and col_index to biological meaning.

Critical alignment:
- x_axis_values_left_to_right must correspond to matrix columns from left to right.
- It should contain exactly {tool_column_count} entries.
- y_axis_values_top_to_bottom must correspond to matrix rows from top to bottom.
- It should contain exactly {tool_row_count} entries.
- col_index 0 maps to x_axis_values_left_to_right[0].
- row_index 0 maps to y_axis_values_top_to_bottom[0].

Rules:
- Preserve visible labels exactly as much as possible.
- Do NOT include colorbar tick labels as x-axis labels.
- Do NOT include neighboring panel labels.
- Do NOT invent labels that are not visible.
- If a label is missing or unreadable, use "Not specified" at that exact position.
- If visible labels form a sequence, only include labels that correspond to visible heatmap columns/rows.
- If you cannot confidently align labels to the detected shape, still return the required length and set manual_review=true.
- If the visible number of labels differs from the detected shape, return exactly {tool_column_count} x labels and exactly {tool_row_count} y labels by placing "Not specified" where needed, and set manual_review=true.

Return JSON schema exactly:
{{
  "chart_type": "heatmap | numeric_matrix | bar_chart | line_chart | image_panel | mixed | not_specified",
  "heatmap_subtype": "time_course_heatmap | treatment_epitope_matrix | sample_epitope_matrix | condition_epitope_matrix | not_specified",
  "assay_label": "ELISpot | Not specified",
  "panel_id": "A/B/C/... or Not specified",
  "matrix_title": "string or Not specified",
  "x_axis_title": "string or Not specified",
  "y_axis_title": "string or Not specified",
  "x_axis_role": "epitope | timepoint | sample | condition | treatment | dose | route | not_specified",
  "y_axis_role": "epitope | timepoint | sample | condition | treatment | dose | route | not_specified",
  "tool_row_count": {tool_row_count},
  "tool_column_count": {tool_column_count},
  "x_axis_values_left_to_right": [],
  "y_axis_values_top_to_bottom": [],
  "visible_conditions_or_treatments": [],
  "colorbar_visible": true,
  "colorbar_label": "string or Not specified",
  "colorbar_values": [],
  "axis_alignment_confidence": "high | medium | low",
  "manual_review": false,
  "review_reason": ""
}}
""".strip()


def ai_extract_core_matrix_fixed_shape(
    core_image_path,
    model_name,
    out_dir,
    safe_name,
    tool_row_count,
    tool_column_count
):
    out_dir = Path(out_dir)
    raw_path = out_dir / "ai_raw" / f"{safe_name}_core_matrix_raw.txt"
    json_path = out_dir / "ai_json" / f"{safe_name}_core_matrix.json"
    if CACHE_EXISTING and json_path.exists():
        parsed = load_json(json_path)
        raw = raw_path.read_text(encoding="utf-8") if raw_path.exists() else ""
        print("Loaded cached core matrix JSON:", json_path)
    else:
        prompt = build_core_matrix_prompt_fixed_shape(
            tool_row_count=tool_row_count,
            tool_column_count=tool_column_count
        )
        parsed, raw = call_ai_json_with_image(
            image_path=core_image_path,
            prompt=prompt,
            model_name=model_name
        )
        ensure_dir(raw_path.parent)
        raw_path.write_text(str(raw), encoding="utf-8")
        save_json(parsed, json_path)
        print("Saved new core matrix JSON:", json_path)
    return {
        "parsed": parsed,
        "raw": raw,
        "raw_path": raw_path,
        "json_path": json_path
    }


def ai_extract_context_axis_fixed_shape(
    context_image_path,
    model_name,
    out_dir,
    safe_name,
    tool_row_count,
    tool_column_count
):
    out_dir = Path(out_dir)
    raw_path = out_dir / "ai_raw" / f"{safe_name}_context_axis_raw.txt"
    json_path = out_dir / "ai_json" / f"{safe_name}_context_axis.json"
    if CACHE_EXISTING and json_path.exists():
        parsed = load_json(json_path)
        raw = raw_path.read_text(encoding="utf-8") if raw_path.exists() else ""
        print("Loaded cached context axis JSON:", json_path)
    else:
        prompt = build_context_axis_prompt_fixed_shape(
            tool_row_count=tool_row_count,
            tool_column_count=tool_column_count
        )
        parsed, raw = call_ai_json_with_image(
            image_path=context_image_path,
            prompt=prompt,
            model_name=model_name
        )
        ensure_dir(raw_path.parent)
        raw_path.write_text(str(raw), encoding="utf-8")
        save_json(parsed, json_path)
        print("Saved new context axis JSON:", json_path)
    return {
        "parsed": parsed,
        "raw": raw,
        "raw_path": raw_path,
        "json_path": json_path
    }


def core_matrix_json_to_records(core_result):
    parsed = core_result["parsed"]
    records = parsed.get("records", [])
    matrix = parsed.get("matrix", [])
    if records:
        df = pd.DataFrame(records)
    else:
        built = []
        for r, row in enumerate(matrix):
            for c, val in enumerate(row):
                built.append({
                    "row_index": r,
                    "col_index": c,
                    "value_text": "Not specified" if val is None else str(val),
                    "value_numeric": val,
                    "manual_review": val is None,
                    "review_reason": "" if val is not None else "AI returned null."
                })
        df = pd.DataFrame(built)
    required_cols = [
        "row_index",
        "col_index",
        "value_text",
        "value_numeric",
        "manual_review",
        "review_reason"
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = "Not specified"
    df["row_index"] = pd.to_numeric(df["row_index"], errors="coerce")
    df["col_index"] = pd.to_numeric(df["col_index"], errors="coerce")
    df["value_numeric"] = pd.to_numeric(df["value_numeric"], errors="coerce")
    df = df.dropna(subset=["row_index", "col_index"]).copy()
    df["row_index"] = df["row_index"].astype(int)
    df["col_index"] = df["col_index"].astype(int)
    df = df.sort_values(["row_index", "col_index"]).reset_index(drop=True)
    return df


def force_core_records_to_tool_shape(core_records_df, tool_row_count, tool_column_count):
    df = core_records_df.copy()
    df["row_index"] = pd.to_numeric(df["row_index"], errors="coerce")
    df["col_index"] = pd.to_numeric(df["col_index"], errors="coerce")
    df = df.dropna(subset=["row_index", "col_index"]).copy()
    df["row_index"] = df["row_index"].astype(int)
    df["col_index"] = df["col_index"].astype(int)
    df = df[
        (df["row_index"] < tool_row_count) &
        (df["col_index"] < tool_column_count)
    ].copy()
    existing = set(zip(df["row_index"], df["col_index"]))
    missing_rows = []
    for r in range(tool_row_count):
        for c in range(tool_column_count):
            if (r, c) not in existing:
                missing_rows.append({
                    "row_index": r,
                    "col_index": c,
                    "value_text": "Not specified",
                    "value_numeric": np.nan,
                    "manual_review": True,
                    "review_reason": "Missing cell after forcing AI output to tool-detected matrix shape."
                })
    if missing_rows:
        df = pd.concat([df, pd.DataFrame(missing_rows)], ignore_index=True)
    df = df.sort_values(["row_index", "col_index"]).reset_index(drop=True)
    return df


def build_core_ai_ordered_matrix_by_index(core_records_df):
    if core_records_df.empty:
        return pd.DataFrame()
    df = core_records_df.copy()
    df["row_index"] = pd.to_numeric(df["row_index"], errors="coerce")
    df["col_index"] = pd.to_numeric(df["col_index"], errors="coerce")
    df["value_numeric"] = pd.to_numeric(df["value_numeric"], errors="coerce")
    df = df.dropna(subset=["row_index", "col_index"]).copy()
    df["row_index"] = df["row_index"].astype(int)
    df["col_index"] = df["col_index"].astype(int)
    matrix_df = df.pivot_table(
        index="row_index",
        columns="col_index",
        values="value_numeric",
        aggfunc="first"
    )
    matrix_df = matrix_df.sort_index().sort_index(axis=1)
    matrix_df.columns = [f"Col_Index_{int(c)}" for c in matrix_df.columns]
    matrix_df = matrix_df.reset_index().rename(columns={"row_index": "Row_Index"})
    return matrix_df


def force_axis_values_to_tool_shape(axis_json, tool_row_count, tool_column_count):
    axis_json = dict(axis_json)
    x_values = axis_json.get("x_axis_values_left_to_right", [])
    y_values = axis_json.get("y_axis_values_top_to_bottom", [])
    if not isinstance(x_values, list):
        x_values = []
    if not isinstance(y_values, list):
        y_values = []
    x_values = [clean_text(v) for v in x_values]
    y_values = [clean_text(v) for v in y_values]
    x_original_count = len(x_values)
    y_original_count = len(y_values)
    if len(x_values) < tool_column_count:
        x_values = x_values + ["Not specified"] * (tool_column_count - len(x_values))
    if len(x_values) > tool_column_count:
        x_values = x_values[:tool_column_count]
    if len(y_values) < tool_row_count:
        y_values = y_values + ["Not specified"] * (tool_row_count - len(y_values))
    if len(y_values) > tool_row_count:
        y_values = y_values[:tool_row_count]
    axis_json["x_axis_values_left_to_right"] = x_values
    axis_json["y_axis_values_top_to_bottom"] = y_values
    axis_json["axis_original_x_label_count"] = x_original_count
    axis_json["axis_original_y_label_count"] = y_original_count
    axis_json["tool_shape_enforced"] = True
    if x_original_count != tool_column_count or y_original_count != tool_row_count:
        axis_json["manual_review"] = True
        old_reason = clean_text(axis_json.get("review_reason", ""))
        shape_reason = (
            f"Axis label count was normalized to tool shape. "
            f"Original x labels: {x_original_count}, tool columns: {tool_column_count}; "
            f"Original y labels: {y_original_count}, tool rows: {tool_row_count}."
        )
        if old_reason == "Not specified":
            axis_json["review_reason"] = shape_reason
        else:
            axis_json["review_reason"] = old_reason + " " + shape_reason
    return axis_json


def get_axis_value(values, idx):
    if not isinstance(values, list):
        return "Not specified"
    if idx < 0 or idx >= len(values):
        return "Not specified"
    return clean_text(values[idx])


def infer_epitope_id(x_value, y_value, x_role, y_role):
    x_role = clean_text(x_role).lower()
    y_role = clean_text(y_role).lower()
    if x_role == "epitope":
        return clean_text(x_value)
    if y_role == "epitope":
        return clean_text(y_value)
    return "Not specified"


def infer_timepoint(x_value, y_value, x_role, y_role):
    x_role = clean_text(x_role).lower()
    y_role = clean_text(y_role).lower()
    if x_role == "timepoint":
        return clean_text(x_value)
    if y_role == "timepoint":
        return clean_text(y_value)
    return "Not specified"


def infer_condition(x_value, y_value, x_role, y_role):
    x_role = clean_text(x_role).lower()
    y_role = clean_text(y_role).lower()
    if x_role in ["condition", "treatment", "sample", "dose", "route"]:
        return clean_text(x_value)
    if y_role in ["condition", "treatment", "sample", "dose", "route"]:
        return clean_text(y_value)
    return "Not specified"


def merge_core_records_with_axis(core_records_df, axis_json, meta, core_result, axis_result, shape_info):
    tool_row_count = shape_info["tool_row_count"]
    tool_column_count = shape_info["tool_column_count"]
    axis_json = force_axis_values_to_tool_shape(
        axis_json=axis_json,
        tool_row_count=tool_row_count,
        tool_column_count=tool_column_count
    )
    x_values = axis_json.get("x_axis_values_left_to_right", [])
    y_values = axis_json.get("y_axis_values_top_to_bottom", [])
    x_role = clean_text(axis_json.get("x_axis_role", "not_specified"))
    y_role = clean_text(axis_json.get("y_axis_role", "not_specified"))
    stage3c_metadata = collect_stage3c_metadata_for_stage4(meta)
    rows = []
    for _, r in core_records_df.iterrows():
        row_index = int(r["row_index"])
        col_index = int(r["col_index"])
        x_value = get_axis_value(x_values, col_index)
        y_value = get_axis_value(y_values, row_index)
        epitope_id = infer_epitope_id(x_value, y_value, x_role, y_role)
        timepoint = infer_timepoint(x_value, y_value, x_role, y_role)
        condition = infer_condition(x_value, y_value, x_role, y_role)
        cell_manual = str(r.get("manual_review", False)).lower() in ["true", "1", "yes"]
        axis_manual = bool(axis_json.get("manual_review", False))
        review_reason_parts = []
        if cell_manual:
            review_reason_parts.append(clean_text(r.get("review_reason", "Core matrix cell marked manual review.")))
        if axis_manual:
            review_reason_parts.append(clean_text(axis_json.get("review_reason", "Axis extraction marked manual review.")))
        if x_value == "Not specified":
            review_reason_parts.append("X-axis label missing for this col_index.")
        if y_value == "Not specified":
            review_reason_parts.append("Y-axis label missing for this row_index.")
        manual_review = cell_manual or axis_manual or x_value == "Not specified" or y_value == "Not specified"
        value_numeric = safe_float(r.get("value_numeric", np.nan))
        value_text = clean_text(r.get("value_text", "Not specified"))
        rec = {
            "Source_Document": clean_text(meta.get("source_doc", "Not specified")),
            "Source_Section": clean_text(meta.get("source_section", "main")),
            "Figure_Unit_ID": clean_text(meta.get("figure_unit_id", "Not specified")),
            "Candidate_ID": clean_text(meta.get("candidate_id", "Not specified")),
            "Panel_ID": clean_text(meta.get("panel_id", axis_json.get("panel_id", "Not specified"))),
            "Chart_Type": clean_text(axis_json.get("chart_type", "Not specified")),
            "Heatmap_Subtype": clean_text(axis_json.get("heatmap_subtype", "Not specified")),
            "Assay_Type": clean_text(axis_json.get("assay_label", "Not specified")),
            "Matrix_Title": clean_text(axis_json.get("matrix_title", "Not specified")),
            "X_Axis_Title": clean_text(axis_json.get("x_axis_title", "Not specified")),
            "Y_Axis_Title": clean_text(axis_json.get("y_axis_title", "Not specified")),
            "X_Axis_Role": x_role,
            "Y_Axis_Role": y_role,
            "X_Value": x_value,
            "Y_Value": y_value,
            "X_Axis_Value": x_value,
            "Y_Axis_Value": y_value,
            "Epitope_ID": epitope_id,
            "Timepoint": timepoint,
            "Condition": condition,
            "Treatment": condition,
            "Value_Text": value_text,
            "Value_Numeric": value_numeric,
            "Value_Type": "printed_numeric",
            "ELISpot_Value": value_numeric,
            "ELISpot_Value_Text": value_text,
            "ELISpot_Unit": clean_text(meta.get("Legend_ELISpot_Unit", "Not specified")),
            "Row_Index": row_index,
            "Col_Index": col_index,
            "Tool_Row_Count": tool_row_count,
            "Tool_Column_Count": tool_column_count,
            "Tool_Grid_Overlay_Path": clean_text(shape_info.get("tool_grid_overlay_path", "Not generated")),
            "Source_Image_Path": clean_text(meta.get("selected_context_path", "Not specified")),
            "Core_Image_Path": clean_text(meta.get("core_crop_path", "Not specified")),
            "Core_AI_JSON_Path": str(core_result["json_path"]),
            "Context_AI_JSON_Path": str(axis_result["json_path"]),
            "Extraction_Method": "tool_shape_fixed_core_ai_matrix_context_ai_axis",
            "Manual_Review": manual_review,
            "Review_Reason": " ".join([x for x in review_reason_parts if x and x != "Not specified"])
        }
        rec.update(stage3c_metadata)
        rows.append(rec)
    return pd.DataFrame(rows), axis_json


def process_one_stage4_heatmap_generic(meta, out_root, model_name="qwen3-vl-plus", debug_display=True):
    out_root = Path(out_root)
    figure_unit_id = clean_text(meta.get("figure_unit_id", "figure"))
    candidate_id = clean_text(meta.get("candidate_id", "0"))
    panel_id = clean_text(meta.get("panel_id", "Not specified"))
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{figure_unit_id}_candidate_{candidate_id}_panel_{panel_id}")
    core_image_path = Path(clean_text(meta["core_crop_path"]))
    context_image_path = Path(clean_text(meta["selected_context_path"]))
    if not core_image_path.exists():
        raise FileNotFoundError(f"Core image not found: {core_image_path}")
    if not context_image_path.exists():
        raise FileNotFoundError(f"Context image not found: {context_image_path}")
    print("\n" + "=" * 120)
    print("Stage 4 processing:", safe_name)
    print("Core:", core_image_path)
    print("Context:", context_image_path)
    if debug_display:
        print("\nCore image:")
        display(Image.open(core_image_path))
        print("\nContext image:")
        display(Image.open(context_image_path))
    shape_info = detect_core_grid_shape(
        core_image_path=core_image_path,
        out_dir=out_root,
        safe_name=safe_name,
        scale=4,
        debug=debug_display
    )
    tool_row_count = shape_info["tool_row_count"]
    tool_column_count = shape_info["tool_column_count"]
    if tool_row_count <= 0 or tool_column_count <= 0:
        raise ValueError(f"Invalid tool-detected shape: rows={tool_row_count}, cols={tool_column_count}")
    print("Tool-fixed matrix shape:")
    print("Rows:", tool_row_count)
    print("Columns:", tool_column_count)
    core_result = ai_extract_core_matrix_fixed_shape(
        core_image_path=core_image_path,
        model_name=model_name,
        out_dir=out_root,
        safe_name=safe_name,
        tool_row_count=tool_row_count,
        tool_column_count=tool_column_count
    )
    core_records_df = core_matrix_json_to_records(core_result)
    core_records_df = force_core_records_to_tool_shape(
        core_records_df=core_records_df,
        tool_row_count=tool_row_count,
        tool_column_count=tool_column_count
    )
    core_ai_matrix_df = build_core_ai_ordered_matrix_by_index(core_records_df)
    axis_result = ai_extract_context_axis_fixed_shape(
        context_image_path=context_image_path,
        model_name=model_name,
        out_dir=out_root,
        safe_name=safe_name,
        tool_row_count=tool_row_count,
        tool_column_count=tool_column_count
    )
    records_df, normalized_axis_json = merge_core_records_with_axis(
        core_records_df=core_records_df,
        axis_json=axis_result["parsed"],
        meta=meta,
        core_result=core_result,
        axis_result=axis_result,
        shape_info=shape_info
    )
    axis_normalized_json_path = out_root / "ai_json" / f"{safe_name}_context_axis_normalized_to_tool_shape.json"
    save_json(normalized_axis_json, axis_normalized_json_path)
    core_ai_matrix_df.insert(0, "Figure_Unit_ID", figure_unit_id)
    core_ai_matrix_df.insert(1, "Candidate_ID", candidate_id)
    core_ai_matrix_df.insert(2, "Panel_ID", clean_text(meta.get("panel_id", "Not specified")))
    summary = {
        "Figure_Unit_ID": figure_unit_id,
        "Candidate_ID": candidate_id,
        "Panel_ID": clean_text(meta.get("panel_id", normalized_axis_json.get("panel_id", "Not specified"))),
        "Target_Panel_ID": clean_text(meta.get("target_panel_id", "Not specified")),
        "Figure_Number": clean_text(meta.get("figure_number", "Not specified")),
        "Assay_Type": clean_text(normalized_axis_json.get("assay_label", "Not specified")),
        "Chart_Type": clean_text(normalized_axis_json.get("chart_type", "Not specified")),
        "Heatmap_Subtype": clean_text(normalized_axis_json.get("heatmap_subtype", "Not specified")),
        "Legend_Readout": clean_text(meta.get("Legend_Readout", "Not specified")),
        "Legend_Sample_or_Tissue": clean_text(meta.get("Legend_Sample_or_Tissue", "Not specified")),
        "Legend_Treatment": clean_text(meta.get("Legend_Treatment", "Not specified")),
        "Legend_Dose": clean_text(meta.get("Legend_Dose", "Not specified")),
        "Legend_Route": clean_text(meta.get("Legend_Route", "Not specified")),
        "Legend_N_Mice": clean_text(meta.get("Legend_N_Mice", "Not specified")),
        "Legend_Mouse_Tumor_Model": clean_text(meta.get("Legend_Mouse_Tumor_Model", "Not specified")),
        "Legend_Mouse_Strain": clean_text(meta.get("Legend_Mouse_Strain", "Not specified")),
        "Legend_Vaccine_or_Antigen_Set": clean_text(meta.get("Legend_Vaccine_or_Antigen_Set", "Not specified")),
        "Legend_ELISpot_Unit": clean_text(meta.get("Legend_ELISpot_Unit", "Not specified")),
        "Panel_Range_Marker": clean_text(meta.get("panel_range_marker", "Not specified")),
        "Panel_Range_Legend_Context": clean_text(meta.get("panel_range_legend_context", "Not specified")),
        "Panel_Specific_Legend_Context": clean_text(meta.get("panel_specific_legend_context", "Not specified")),
        "Combined_Panel_Legend_Context": clean_text(meta.get("combined_panel_legend_context", "Not specified")),
        "Tool_Row_Count": tool_row_count,
        "Tool_Column_Count": tool_column_count,
        "Record_Count": int(len(records_df)),
        "Manual_Review_Count": int(records_df["Manual_Review"].astype(str).str.lower().isin(["true", "1", "yes"]).sum()),
        "Core_Image_Path": str(core_image_path),
        "Context_Image_Path": str(context_image_path),
        "Tool_Grid_Overlay_Path": clean_text(shape_info.get("tool_grid_overlay_path", "Not generated")),
        "Core_AI_JSON_Path": str(core_result["json_path"]),
        "Context_AI_JSON_Path": str(axis_result["json_path"]),
        "Context_AI_Normalized_JSON_Path": str(axis_normalized_json_path),
        "Axis_X_Values": json.dumps(normalized_axis_json.get("x_axis_values_left_to_right", []), ensure_ascii=False),
        "Axis_Y_Values": json.dumps(normalized_axis_json.get("y_axis_values_top_to_bottom", []), ensure_ascii=False),
        "Axis_Alignment_Confidence": clean_text(normalized_axis_json.get("axis_alignment_confidence", "Not specified")),
        "Axis_Manual_Review": bool(normalized_axis_json.get("manual_review", False)),
        "Axis_Review_Reason": clean_text(normalized_axis_json.get("review_reason", ""))
    }
    if debug_display:
        print("\nCore AI ordered matrix by index:")
        display(core_ai_matrix_df)
        print("\nFinal records head:")
        display(records_df.head(30))
    return {
        "records_df": records_df,
        "core_ai_matrix_df": core_ai_matrix_df,
        "summary": summary
    }


def run_stage4_generic_all_selected(
    stage3_selected_manifest_path,
    out_root="./stage4_final_generic_outputs_after_stage3b3c",
    model_name="qwen3-vl-plus",
    figure_unit_filter=None,
    limit=None,
    debug_display=True
):
    out_root = Path(out_root)
    for d in [
        out_root,
        out_root / "ai_raw",
        out_root / "ai_json",
        out_root / "tool_grid_overlays"
    ]:
        ensure_dir(d)
    stage3_df = read_table_robust(stage3_selected_manifest_path)
    stage3_df = ensure_stage3c_passthrough_columns(stage3_df)
    required_cols = ["core_crop_path", "selected_context_path"]
    for col in required_cols:
        if col not in stage3_df.columns:
            raise KeyError(f"Stage 4 input manifest must contain column: {col}")
    stage3_df = stage3_df[
        stage3_df["selected_context_path"].astype(str).ne("Not selected")
    ].copy()
    if figure_unit_filter is not None:
        if isinstance(figure_unit_filter, str):
            figure_unit_filter = [figure_unit_filter]
        if "figure_unit_id" not in stage3_df.columns:
            raise KeyError("figure_unit_filter requires column: figure_unit_id")
        stage3_df = stage3_df[
            stage3_df["figure_unit_id"].astype(str).isin(figure_unit_filter)
        ].copy()
    if limit is not None:
        stage3_df = stage3_df.head(limit).copy()
    print("Stage 4 selected heatmaps to process:", len(stage3_df))
    preview_cols = [
        "figure_unit_id",
        "candidate_id",
        "panel_id",
        "core_crop_path",
        "selected_context_path",
        "Legend_Readout",
        "Legend_Treatment",
        "Legend_Vaccine_or_Antigen_Set",
        "panel_specific_legend_context"
    ]
    display(stage3_df[[c for c in preview_cols if c in stage3_df.columns]])
    all_records = []
    all_core_ai_matrices = []
    summaries = []
    failed = []
    for idx, row in stage3_df.iterrows():
        meta = row.to_dict()
        try:
            one = process_one_stage4_heatmap_generic(
                meta=meta,
                out_root=out_root,
                model_name=model_name,
                debug_display=debug_display
            )
            all_records.append(one["records_df"])
            all_core_ai_matrices.append(one["core_ai_matrix_df"])
            summaries.append(one["summary"])
        except Exception as e:
            failed.append({
                "row_index_in_manifest": int(idx),
                "figure_unit_id": clean_text(meta.get("figure_unit_id", "Not specified")),
                "candidate_id": clean_text(meta.get("candidate_id", "Not specified")),
                "panel_id": clean_text(meta.get("panel_id", "Not specified")),
                "error": str(e)
            })
            print("FAILED:", failed[-1])
    if all_records:
        all_records_df = pd.concat(all_records, ignore_index=True)
    else:
        all_records_df = pd.DataFrame()
    if all_core_ai_matrices:
        all_core_ai_matrices_df = pd.concat(all_core_ai_matrices, ignore_index=True)
    else:
        all_core_ai_matrices_df = pd.DataFrame()
    summary_df = pd.DataFrame(summaries)
    failed_df = pd.DataFrame(failed)
    all_records_csv = out_root / "Stage4_Cell_Records.csv"
    all_records_xlsx = out_root / "Stage4_Cell_Records.xlsx"
    core_ai_matrix_csv = out_root / "Stage4_Core_AI_Matrix_By_Index.csv"
    core_ai_matrix_xlsx = out_root / "Stage4_Core_AI_Matrix_By_Index.xlsx"
    summary_csv = out_root / "Stage4_Panel_Summary.csv"
    summary_xlsx = out_root / "Stage4_Panel_Summary.xlsx"
    failed_csv = out_root / "Stage4_Failed_Items.csv"
    failed_xlsx = out_root / "Stage4_Failed_Items.xlsx"
    combined_xlsx = out_root / "Stage4_Combined_Output.xlsx"
    qc_json = out_root / "Stage4_QC_Summary.json"
    all_records_df.to_csv(all_records_csv, index=False, encoding="utf-8-sig")
    all_records_df.to_excel(all_records_xlsx, index=False)
    all_core_ai_matrices_df.to_csv(core_ai_matrix_csv, index=False, encoding="utf-8-sig")
    all_core_ai_matrices_df.to_excel(core_ai_matrix_xlsx, index=False)
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    summary_df.to_excel(summary_xlsx, index=False)
    failed_df.to_csv(failed_csv, index=False, encoding="utf-8-sig")
    failed_df.to_excel(failed_xlsx, index=False)
    qc = {
        "source_stage3b3c_manifest": str(stage3_selected_manifest_path),
        "model_name": model_name,
        "cache_existing": CACHE_EXISTING,
        "figure_unit_filter": figure_unit_filter,
        "selected_heatmap_count": int(len(stage3_df)),
        "processed_success_count": int(len(summary_df)),
        "failed_count": int(len(failed_df)),
        "total_cell_records": int(len(all_records_df)),
        "manual_review_count": int(
            all_records_df["Manual_Review"].astype(str).str.lower().isin(["true", "1", "yes"]).sum()
        ) if not all_records_df.empty and "Manual_Review" in all_records_df.columns else 0,
        "records_by_figure": all_records_df["Figure_Unit_ID"].value_counts(dropna=False).to_dict()
            if not all_records_df.empty and "Figure_Unit_ID" in all_records_df.columns else {},
        "records_by_panel": all_records_df["Panel_ID"].value_counts(dropna=False).to_dict()
            if not all_records_df.empty and "Panel_ID" in all_records_df.columns else {},
        "outputs": {
            "cell_records_csv": str(all_records_csv),
            "cell_records_xlsx": str(all_records_xlsx),
            "core_ai_matrix_csv": str(core_ai_matrix_csv),
            "core_ai_matrix_xlsx": str(core_ai_matrix_xlsx),
            "summary_csv": str(summary_csv),
            "summary_xlsx": str(summary_xlsx),
            "failed_csv": str(failed_csv),
            "failed_xlsx": str(failed_xlsx),
            "combined_xlsx": str(combined_xlsx),
            "qc_json": str(qc_json)
        }
    }
    save_json(qc, qc_json)
    with pd.ExcelWriter(combined_xlsx, engine="openpyxl") as writer:
        all_records_df.to_excel(writer, sheet_name="Stage4_Cell_Records", index=False)
        all_core_ai_matrices_df.to_excel(writer, sheet_name="Core_AI_Matrix_By_Index", index=False)
        summary_df.to_excel(writer, sheet_name="Panel_Summary", index=False)
        failed_df.to_excel(writer, sheet_name="Failed_Items", index=False)
        pd.DataFrame([
            {
                "Metric": k,
                "Value": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
            }
            for k, v in qc.items()
        ]).to_excel(writer, sheet_name="QC_Summary", index=False)
    print("\n" + "=" * 120)
    print("Stage 4 complete.")
    print("Selected heatmaps:", len(stage3_df))
    print("Processed success:", len(summary_df))
    print("Failed:", len(failed_df))
    print("Total cell records:", len(all_records_df))
    print("Cell records:", all_records_xlsx)
    print("Core AI matrix:", core_ai_matrix_xlsx)
    print("Panel summary:", summary_xlsx)
    print("Combined Excel:", combined_xlsx)
    print("QC:", qc_json)
    print("\nCore AI matrix by index:")
    display(all_core_ai_matrices_df)
    print("\nPanel summary:")
    display(summary_df)
    print("\nCell records head:")
    display(all_records_df.head(50))
    print("\nFailed:")
    display(failed_df)
    return {
        "records_df": all_records_df,
        "core_ai_matrix_df": all_core_ai_matrices_df,
        "summary_df": summary_df,
        "failed_df": failed_df,
        "qc": qc,
        "cell_records_csv": all_records_csv,
        "cell_records_xlsx": all_records_xlsx,
        "core_ai_matrix_csv": core_ai_matrix_csv,
        "core_ai_matrix_xlsx": core_ai_matrix_xlsx,
        "summary_csv": summary_csv,
        "summary_xlsx": summary_xlsx,
        "failed_csv": failed_csv,
        "failed_xlsx": failed_xlsx,
        "combined_xlsx": combined_xlsx,
        "qc_json": qc_json
    }


In [23]:
# ============================================================
# Generic Stage 4 shape detector
# Explicit line grid when available; colour-transition boundaries otherwise.
# ============================================================

def _group_peak_indices(indices, max_gap):
    if len(indices) == 0:
        return []
    groups, current = [], [int(indices[0])]
    for idx in indices[1:]:
        idx = int(idx)
        if idx - current[-1] <= max_gap:
            current.append(idx)
        else:
            groups.append(current)
            current = [idx]
    groups.append(current)
    return [int(round(np.mean(g))) for g in groups]


def _regularize_boundaries(peaks, length, min_cells=2):
    peaks = sorted(set(int(x) for x in peaks if 0 < x < length - 1))
    if len(peaks) < min_cells - 1:
        return [], 0.0
    points = [0] + peaks + [length - 1]
    gaps = np.diff(points).astype(float)
    positive = gaps[gaps > max(2, length * 0.003)]
    if len(positive) < min_cells:
        return [], 0.0
    median_gap = float(np.median(positive))
    if median_gap <= 0:
        return [], 0.0
    kept = [0]
    for p in points[1:-1]:
        if p - kept[-1] >= median_gap * 0.42:
            kept.append(p)
    if length - 1 - kept[-1] >= median_gap * 0.42:
        kept.append(length - 1)
    else:
        kept[-1] = length - 1
    gaps = np.diff(kept).astype(float)
    if len(gaps) < min_cells:
        return [], 0.0
    cv = float(np.std(gaps) / max(np.mean(gaps), 1e-6))
    confidence = float(max(0.0, min(1.0, 1.0 - cv)))
    return kept, confidence


def _color_transition_boundaries(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    lab = cv2.GaussianBlur(lab, (5, 5), 0)
    # Robust mean colour discontinuity across most of the opposite dimension.
    dx = np.linalg.norm(lab[:, 1:, :] - lab[:, :-1, :], axis=2)
    dy = np.linalg.norm(lab[1:, :, :] - lab[:-1, :, :], axis=2)
    x_profile = np.percentile(dx, 65, axis=0)
    y_profile = np.percentile(dy, 65, axis=1)
    x_thr = max(float(np.percentile(x_profile, 82)), float(np.mean(x_profile) + 0.60 * np.std(x_profile)))
    y_thr = max(float(np.percentile(y_profile, 82)), float(np.mean(y_profile) + 0.60 * np.std(y_profile)))
    x_peaks = _group_peak_indices(np.where(x_profile >= x_thr)[0] + 1, max_gap=max(2, rgb.shape[1] // 220))
    y_peaks = _group_peak_indices(np.where(y_profile >= y_thr)[0] + 1, max_gap=max(2, rgb.shape[0] // 220))
    x_lines, x_conf = _regularize_boundaries(x_peaks, rgb.shape[1])
    y_lines, y_conf = _regularize_boundaries(y_peaks, rgb.shape[0])
    return y_lines, x_lines, float(min(x_conf, y_conf)), {"x_threshold": x_thr, "y_threshold": y_thr, "x_peak_count": len(x_peaks), "y_peak_count": len(y_peaks)}


def _line_grid_boundaries(gray):
    h, w = gray.shape
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 31, 12)
    horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (max(18, w // 18), 1)))
    vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(15, h // 12))))
    h_proj, v_proj = horizontal.sum(axis=1), vertical.sum(axis=0)
    h_idx = np.where(h_proj >= max(np.max(h_proj) * 0.32, np.percentile(h_proj, 91)))[0] if np.max(h_proj) > 0 else []
    v_idx = np.where(v_proj >= max(np.max(v_proj) * 0.32, np.percentile(v_proj, 91)))[0] if np.max(v_proj) > 0 else []
    h_lines = merge_close_lines(_group_peak_indices(h_idx, max(2, h // 250)), min_dist=max(3, h // 180))
    v_lines = merge_close_lines(_group_peak_indices(v_idx, max(2, w // 250)), min_dist=max(3, w // 180))
    # Coverage rejects text strokes pretending to be grid lines.
    h_cov = float(np.mean([np.mean(horizontal[y] > 0) for y in h_lines])) if h_lines else 0.0
    v_cov = float(np.mean([np.mean(vertical[:, x] > 0) for x in v_lines])) if v_lines else 0.0
    confidence = float(min(1.0, (h_cov + v_cov) / 0.75))
    # Candidate core crops are intended to be tight. If the outer border is
    # absent or pale, image edges are valid deterministic matrix boundaries.
    if h_lines:
        h_lines[0] = 0 if h_lines[0] <= h * 0.04 else h_lines[0]
        h_lines[-1] = h - 1 if h - 1 - h_lines[-1] <= h * 0.04 else h_lines[-1]
        if h_lines[0] > h * 0.08:
            h_lines.insert(0, 0)
        if h - 1 - h_lines[-1] > h * 0.08:
            h_lines.append(h - 1)
    if v_lines:
        v_lines[0] = 0 if v_lines[0] <= w * 0.04 else v_lines[0]
        v_lines[-1] = w - 1 if w - 1 - v_lines[-1] <= w * 0.04 else v_lines[-1]
        if v_lines[0] > w * 0.08:
            v_lines.insert(0, 0)
        if w - 1 - v_lines[-1] > w * 0.08:
            v_lines.append(w - 1)
    return h_lines, v_lines, confidence, {"horizontal_coverage": h_cov, "vertical_coverage": v_cov}


def detect_core_grid_shape(core_image_path, out_dir=None, safe_name="core", scale=4, debug=True):
    core_image_path = Path(core_image_path)
    original = Image.open(core_image_path).convert("RGB")
    img = original.resize((original.width * scale, original.height * scale), Image.Resampling.LANCZOS)
    rgb = np.array(img)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    line_h, line_v, line_conf, line_meta = _line_grid_boundaries(gray)
    color_h, color_v, color_conf, color_meta = _color_transition_boundaries(rgb)
    if len(line_h) >= 3 and len(line_v) >= 3 and line_conf >= color_conf:
        horizontal_lines, vertical_lines = line_h, line_v
        method, confidence, evidence = "explicit_grid_lines", line_conf, line_meta
    else:
        horizontal_lines, vertical_lines = color_h, color_v
        method, confidence, evidence = "color_transition_boundaries", color_conf, color_meta
    rows = max(0, len(horizontal_lines) - 1)
    cols = max(0, len(vertical_lines) - 1)
    overlay_path = "Not generated"
    if out_dir is not None:
        overlay_dir = Path(out_dir) / "tool_grid_overlays"
        ensure_dir(overlay_dir)
        overlay_path = overlay_dir / f"{safe_name}_tool_grid_overlay.png"
        draw_tool_grid_overlay(gray, horizontal_lines, vertical_lines, overlay_path)
    manual_review = rows <= 0 or cols <= 0 or confidence < GENERIC_HEATMAP_CONFIG.get("shape_min_confidence", 0.45)
    result = {
        "tool_row_count": int(rows),
        "tool_column_count": int(cols),
        "horizontal_lines": [int(x) for x in horizontal_lines],
        "vertical_lines": [int(x) for x in vertical_lines],
        "scale": int(scale),
        "tool_grid_overlay_path": str(overlay_path),
        "shape_detection_method": method,
        "shape_detection_confidence": float(confidence),
        "shape_detection_evidence": evidence,
        "shape_manual_review": bool(manual_review),
        "uses_numeric_confidence_threshold": False,
    }
    if debug:
        print("Shape method:", method)
        print("Tool shape:", rows, "x", cols)
        print("Structural confidence (diagnostic only):", confidence)
        print("Manual review:", manual_review)
        if Path(str(overlay_path)).exists():
            display(Image.open(overlay_path))
    return result


def build_core_matrix_prompt_fixed_shape(tool_row_count, tool_column_count):
    return f"""
You are extracting printed values from a scientific heatmap core crop.
Return ONLY valid JSON and no markdown.

A deterministic visual detector fixed the matrix shape at exactly {tool_row_count} rows and {tool_column_count} columns.
Do not change the shape. Read top-to-bottom and left-to-right with zero-based indices.

Scientific integrity rules:
- Read only numbers or explicit text visibly printed inside a cell.
- Never convert colour, shade, intensity, or colorbar position into a guessed numeric value.
- If a cell is colour-only or unreadable, set value_text to null, value_numeric to null,
  manual_review to true, and review_reason to "No printed numeric value; colour-only cell" or the actual reason.
- Do not copy colorbar tick labels into matrix cells.
- Do not invent missing values.
- The records list must contain exactly {tool_row_count * tool_column_count} records.

Schema:
{{
  "matrix_detected": true,
  "row_count": {tool_row_count},
  "column_count": {tool_column_count},
  "read_direction": "top_to_bottom_left_to_right",
  "manual_review": false,
  "review_reason": "",
  "matrix": [[]],
  "records": [
    {{
      "row_index": 0,
      "col_index": 0,
      "value_text": null,
      "value_numeric": null,
      "manual_review": true,
      "review_reason": "No printed numeric value; colour-only cell"
    }}
  ]
}}
""".strip()


In [27]:

# ============================================================
# Reviewer Comment 3 — formal structure-constrained mechanism
# D_cv(I) -> G, E_vlm(I | G) -> M_hat, V(M_hat, G), P_G(M_hat) -> M_star
# ============================================================
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class StructureConstraint:
    """Deterministic grid constraint G produced before VLM value recognition."""
    rows: int
    columns: int
    horizontal_grid_lines: tuple = ()
    vertical_grid_lines: tuple = ()
    detector: str = "OpenCV deterministic grid-line detection"

    @property
    def expected_record_count(self):
        return int(self.rows) * int(self.columns)

    def to_dict(self):
        data = asdict(self)
        data["expected_record_count"] = self.expected_record_count
        return data


def build_structure_constraint(shape_info):
    """D_cv(I) -> G. Qwen does not determine rows/columns."""
    return StructureConstraint(
        rows=int(shape_info["tool_row_count"]),
        columns=int(shape_info["tool_column_count"]),
        horizontal_grid_lines=tuple(shape_info.get("horizontal_lines") or ()),
        vertical_grid_lines=tuple(shape_info.get("vertical_lines") or ()),
    )


def validate_against_structure_constraint(raw_core_json, raw_axis_json, constraint):
    """V(M_hat,G): validate raw VLM output without normalization."""
    core_valid, core_errors = validate_stage4_core_raw(
        raw_core_json, constraint.rows, constraint.columns
    )
    axis_valid, axis_errors = validate_stage4_axis_raw(
        raw_axis_json, constraint.rows, constraint.columns
    )
    return {
        "raw_core_schema_valid": bool(core_valid),
        "raw_axis_schema_valid": bool(axis_valid),
        "raw_structure_valid": bool(core_valid and axis_valid),
        "core_validation_errors": core_errors,
        "axis_validation_errors": axis_errors,
        "constraint": constraint.to_dict(),
    }


def project_core_output_to_constraint(raw_core_json, constraint):
    """P_G(M_hat) -> M_star using the existing Stage 4 normalization unchanged."""
    raw_records = core_matrix_json_to_records({"parsed": raw_core_json})
    projected_records = force_core_records_to_tool_shape(
        raw_records,
        tool_row_count=constraint.rows,
        tool_column_count=constraint.columns,
    )
    projected_for_validation = projected_records.rename(columns={
        "row_index": "Row_Index", "col_index": "Col_Index"
    })
    usable, errors = validate_stage4_normalized_records(
        projected_for_validation, constraint.rows, constraint.columns
    )
    return projected_records, {
        "post_normalization_usable": bool(usable),
        "post_normalization_errors": errors,
        "raw_validity_is_not_inferred_from_normalized_usability": True,
    }


def assess_structure_constrained_mechanism(raw_core_json, raw_axis_json,
                                           normalized_records_df, constraint):
    """Keep raw-model validity and post-normalization usability separate."""
    raw_report = validate_against_structure_constraint(
        raw_core_json, raw_axis_json, constraint
    )
    normalized_valid, normalized_errors = validate_stage4_normalized_records(
        normalized_records_df, constraint.rows, constraint.columns
    )
    return {
        **raw_report,
        "post_normalization_usable": bool(normalized_valid),
        "post_normalization_errors": normalized_errors,
        "normalization_operator": "force_core_records_to_tool_shape",
        "uses_numeric_confidence_threshold": False,
    }


STRUCTURE_CONSTRAINED_METHOD = {
    "name": "Structure-constrained multimodal extraction",
    "formal_definition": {
        "structure_detection": "G = D_cv(I) = (R, C, L_h, L_v)",
        "constrained_recognition": "M_hat = E_vlm(I | R, C)",
        "structural_validation": "V(M_hat,G) = V_shape AND V_record AND V_index AND V_label",
        "projection": "M_star = P_G(M_hat)",
        "non_equivalence": "V(M_hat,G) is recorded separately from Usable(P_G(M_hat))",
    },
    "deterministic_structure_source": "OpenCV grid-line detection",
    "multimodal_content_reader": "Qwen",
    "uses_numeric_confidence_threshold": False,
}

REVIEWER3_METHOD_DIR = Path(RUN_ROOT) / "reviewer3_structure_constrained_method"
REVIEWER3_METHOD_DIR.mkdir(parents=True, exist_ok=True)
(REVIEWER3_METHOD_DIR / "structure_constraint_definition.json").write_text(
    json.dumps(STRUCTURE_CONSTRAINED_METHOD, ensure_ascii=False, indent=2),
    encoding="utf-8"
)


573

In [25]:

# Reviewer Comment 6 — audit Stage 4 without changing extraction or normalization logic.
_call_ai_json_with_image_unversioned = call_ai_json_with_image

def _infer_stage4_task_and_shape(prompt):
    text = str(prompt)
    shape_match = re.search(r"exactly\s+(\d+)\s+rows?\s+and\s+(\d+)\s+columns?", text, flags=re.I)
    rows = int(shape_match.group(1)) if shape_match else None
    cols = int(shape_match.group(2)) if shape_match else None
    if '"matrix"' in text and '"records"' in text:
        return "stage4_core_matrix_fixed_shape", rows, cols
    return "stage4_context_axis_fixed_shape", rows, cols

def call_ai_json_with_image(image_path, prompt, model_name="qwen3-vl-plus"):
    task_name, rows, cols = _infer_stage4_task_and_shape(prompt)
    image_url = image_to_data_url(image_path)
    def invoke_once():
        client = build_client()
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]}],
            temperature=AI_AUDIT_CONFIG["temperature"],
            top_p=AI_AUDIT_CONFIG["top_p"],
        )
        return response.choices[0].message.content or ""
    parsed, raw, _audit = run_audited_json_inference(
        invoke_once, prompt=prompt, task_name=task_name, stage="Stage 4", model=model_name,
        context={"paper": PAPER_CONFIG.get("paper_id"), "figure": Path(image_path).stem, "panel": None},
        rows=rows, cols=cols,
    )
    if isinstance(parsed, dict):
        parsed["prompt_version"] = PROMPT_VERSIONS[task_name]
        parsed["uses_numeric_confidence_threshold"] = False
        confidence = parsed.get("confidence") or parsed.get("axis_alignment_confidence")
        if confidence is not None:
            parsed["model_self_reported_confidence"] = confidence
    return parsed, raw

_process_one_stage4_original = process_one_stage4_heatmap_generic
def process_one_stage4_heatmap_generic(meta, out_root, model_name="qwen3-vl-plus", debug_display=True):
    result = _process_one_stage4_original(meta, out_root, model_name=model_name, debug_display=debug_display)
    if not result:
        return result
    records_df = result["records_df"]
    matrix_df = result["core_ai_matrix_df"]
    summary = result["summary"]
    rows = int(summary.get("Tool_Row_Count") or (records_df["Tool_Row_Count"].iloc[0] if len(records_df) else 0))
    cols = int(summary.get("Tool_Column_Count") or (records_df["Tool_Column_Count"].iloc[0] if len(records_df) else 0))
    _write_prompt_manifest_entry(
        "stage4_core_matrix_fixed_shape", PROMPT_VERSIONS["stage4_core_matrix_fixed_shape"],
        build_core_matrix_prompt_fixed_shape(rows, cols), model_name,
        AI_TASK_SCHEMAS["stage4_core_matrix_fixed_shape"]
    )
    _write_prompt_manifest_entry(
        "stage4_context_axis_fixed_shape", PROMPT_VERSIONS["stage4_context_axis_fixed_shape"],
        build_context_axis_prompt_fixed_shape(rows, cols), model_name,
        AI_TASK_SCHEMAS["stage4_context_axis_fixed_shape"]
    )

    core_json_path = Path(summary.get("Core_AI_JSON_Path", ""))
    axis_json_path = Path(summary.get("Context_AI_JSON_Path", ""))
    try:
        raw_core = json.loads(core_json_path.read_text(encoding="utf-8"))
    except Exception as exc:
        raw_core = {}
        raw_core_load_error = str(exc)
    else:
        raw_core_load_error = ""
    try:
        raw_axis = json.loads(axis_json_path.read_text(encoding="utf-8"))
    except Exception as exc:
        raw_axis = {}
        raw_axis_load_error = str(exc)
    else:
        raw_axis_load_error = ""

    core_raw_valid, core_raw_errors = validate_stage4_core_raw(raw_core, rows, cols)
    axis_raw_valid, axis_raw_errors = validate_stage4_axis_raw(raw_axis, rows, cols)
    if raw_core_load_error:
        core_raw_errors.append(f"raw JSON load failure: {raw_core_load_error}")
    if raw_axis_load_error:
        axis_raw_errors.append(f"raw JSON load failure: {raw_axis_load_error}")
    normalized_core_usable, normalized_core_errors = validate_stage4_normalized_records(records_df, rows, cols)
    normalized_axis = {
        "x_axis_values_left_to_right": records_df.sort_values("Col_Index").drop_duplicates("Col_Index")["X_Value"].tolist(),
        "y_axis_values_top_to_bottom": records_df.sort_values("Row_Index").drop_duplicates("Row_Index")["Y_Value"].tolist(),
    }
    normalized_axis_usable, normalized_axis_errors = validate_stage4_normalized_axis(normalized_axis, rows, cols)
    structure_constraint = StructureConstraint(rows=rows, columns=cols)
    mechanism_assessment = assess_structure_constrained_mechanism(
        raw_core, raw_axis, records_df, structure_constraint
    )

    safe_name = Path(core_json_path).stem.replace("_core_matrix", "") if str(core_json_path) else "stage4"
    validation_dir = Path(out_root) / "validation"
    validation_dir.mkdir(parents=True, exist_ok=True)
    raw_validation_path = validation_dir / f"{safe_name}_raw_ai_validation.json"
    normalized_validation_path = validation_dir / f"{safe_name}_post_normalization_validation.json"
    mechanism_assessment_path = validation_dir / f"{safe_name}_structure_constraint_assessment.json"
    raw_validation = {
        "raw_core_ai_schema_valid": bool(core_raw_valid),
        "raw_core_validation_errors": core_raw_errors,
        "raw_axis_ai_schema_valid": bool(axis_raw_valid),
        "raw_axis_validation_errors": axis_raw_errors,
        "normalization_success_is_not_raw_ai_success": True,
    }
    normalized_validation = {
        "normalized_core_usable": bool(normalized_core_usable),
        "normalized_core_validation_errors": normalized_core_errors,
        "normalized_axis_usable": bool(normalized_axis_usable),
        "normalized_axis_validation_errors": normalized_axis_errors,
        "normalization_function": "force_core_records_to_tool_shape / force_axis_values_to_tool_shape",
    }
    raw_validation_path.write_text(json.dumps(raw_validation, ensure_ascii=False, indent=2), encoding="utf-8")
    normalized_validation_path.write_text(json.dumps(normalized_validation, ensure_ascii=False, indent=2), encoding="utf-8")
    mechanism_assessment_path.write_text(
        json.dumps(mechanism_assessment, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    records_df["Structure_Constraint_Rows"] = rows
    records_df["Structure_Constraint_Columns"] = cols
    records_df["Structure_Constraint_Expected_Cells"] = structure_constraint.expected_record_count
    records_df["Raw_Core_AI_Schema_Valid"] = bool(core_raw_valid)
    records_df["Raw_Axis_AI_Schema_Valid"] = bool(axis_raw_valid)
    records_df["Post_Normalization_Core_Usable"] = bool(normalized_core_usable)
    records_df["Post_Normalization_Axis_Usable"] = bool(normalized_axis_usable)
    records_df["Uses_Numeric_Confidence_Threshold"] = False
    records_df["Stage4_Core_Prompt_Version"] = PROMPT_VERSIONS["stage4_core_matrix_fixed_shape"]
    records_df["Stage4_Axis_Prompt_Version"] = PROMPT_VERSIONS["stage4_context_axis_fixed_shape"]
    if not core_raw_valid or not axis_raw_valid:
        if "Manual_Review" in records_df.columns:
            records_df["Manual_Review"] = True
        reason = "Raw AI schema invalid; normalized output is tracked separately."
        if "Review_Reason" in records_df.columns:
            records_df["Review_Reason"] = records_df["Review_Reason"].fillna("").astype(str).map(
                lambda x: (x + "; " + reason).strip("; ")
            )
    summary.update({
        "Raw_Core_AI_Schema_Valid": bool(core_raw_valid),
        "Raw_Axis_AI_Schema_Valid": bool(axis_raw_valid),
        "Post_Normalization_Core_Usable": bool(normalized_core_usable),
        "Post_Normalization_Axis_Usable": bool(normalized_axis_usable),
        "Raw_AI_Validation_Path": str(raw_validation_path),
        "Post_Normalization_Validation_Path": str(normalized_validation_path),
        "Structure_Constraint_Assessment_Path": str(mechanism_assessment_path),
        "Structure_Constraint_Rows": rows,
        "Structure_Constraint_Columns": cols,
        "Structure_Constraint_Expected_Cells": structure_constraint.expected_record_count,
        "Uses_Numeric_Confidence_Threshold": False,
    })
    result["records_df"] = records_df
    result["core_ai_matrix_df"] = matrix_df
    result["summary"] = summary
    return result


In [ ]:
# ============================================================
# Generic Stage 4C — deterministic cell visual features
# These are image proxies, not reconstructed biological values.
# ============================================================

def extract_cell_visual_features(core_image_path, shape_info):
    img = Image.open(core_image_path).convert("RGB")
    scale = int(shape_info.get("scale", 1))
    rgb = np.array(img.resize((img.width * scale, img.height * scale), Image.Resampling.LANCZOS))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    hs = [int(x) for x in shape_info.get("horizontal_lines", [])]
    vs = [int(x) for x in shape_info.get("vertical_lines", [])]
    rows = []
    for r in range(max(0, len(hs) - 1)):
        for c in range(max(0, len(vs) - 1)):
            y0, y1 = hs[r], hs[r + 1]
            x0, x1 = vs[c], vs[c + 1]
            border = max(1, int(min(y1 - y0, x1 - x0) * 0.10))
            inner = rgb[min(y1, y0 + border):max(y0 + border + 1, y1 - border), min(x1, x0 + border):max(x0 + border + 1, x1 - border)]
            inner_gray = gray[min(y1, y0 + border):max(y0 + border + 1, y1 - border), min(x1, x0 + border):max(x0 + border + 1, x1 - border)]
            if inner.size == 0 or inner_gray.size == 0:
                rows.append({"Row_Index": r, "Col_Index": c, "Cell_Visual_Feature_Available": False})
                continue
            med = np.median(inner.reshape(-1, 3), axis=0)
            p10, p90 = np.percentile(inner_gray, [10, 90])
            rows.append({
                "Row_Index": r,
                "Col_Index": c,
                "Cell_Visual_Feature_Available": True,
                "Cell_Median_R": float(med[0]),
                "Cell_Median_G": float(med[1]),
                "Cell_Median_B": float(med[2]),
                "Cell_Color_Hex_Proxy": "#{:02x}{:02x}{:02x}".format(*[int(round(x)) for x in med]),
                "Cell_Gray_Median_Proxy": float(np.median(inner_gray)),
                "Cell_Visual_Contrast_Proxy": float(p90 - p10),
                "Cell_Width_Px": float((x1 - x0) / max(scale, 1)),
                "Cell_Height_Px": float((y1 - y0) / max(scale, 1)),
                "Cell_Area_Px2": float((x1 - x0) * (y1 - y0) / max(scale * scale, 1)),
                "Cell_Color_Is_Not_Numeric_Value": True,
            })
    return pd.DataFrame(rows)


_generic_visual_process_original = process_one_stage4_heatmap_generic
def process_one_stage4_heatmap_generic(meta, out_root, model_name="qwen3-vl-plus", debug_display=True):
    result = _generic_visual_process_original(meta, out_root, model_name=model_name, debug_display=debug_display)
    if not result or not GENERIC_HEATMAP_CONFIG.get("save_cell_visual_features", True):
        return result
    core_path = Path(clean_text(meta["core_crop_path"]))
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{meta.get('figure_unit_id','figure')}_candidate_{meta.get('candidate_id','0')}_panel_{meta.get('panel_id','Not specified')}")
    shape_info = detect_core_grid_shape(core_path, out_dir=out_root, safe_name=safe_name, scale=4, debug=False)
    visual_df = extract_cell_visual_features(core_path, shape_info)
    records = result["records_df"].merge(visual_df, on=["Row_Index", "Col_Index"], how="left")
    records["Numeric_Value_From_Color_Allowed"] = False
    records["Visual_Features_Are_Proxies"] = True
    result["records_df"] = records
    result["summary"]["Shape_Detection_Method"] = shape_info.get("shape_detection_method")
    result["summary"]["Shape_Detection_Confidence"] = shape_info.get("shape_detection_confidence")
    result["summary"]["Shape_Manual_Review"] = shape_info.get("shape_manual_review")
    result["summary"]["Numeric_Value_From_Color_Allowed"] = False
    return result


## Coordinate-locked Stage 4 extraction

The tool fixes every cell coordinate before Qwen reads the content. Each model call receives one indexed row atlas, so the model cannot rebuild or shift the complete sparse matrix. Row and column indices are assigned by code. The legacy whole-matrix extractor remains available through configuration for reproducibility.


In [ ]:
# Stage 4D: coordinate-locked indexed-row extraction
# Fixes the semantic-alignment gap left by shape-only validation.
# The tool defines cells; Qwen reads one indexed cell row; code assigns coordinates.

from PIL import Image, ImageDraw, ImageFont
import math

GENERIC_HEATMAP_CONFIG.setdefault("stage4_core_extraction_mode", "indexed_row_atlas")
GENERIC_HEATMAP_CONFIG.setdefault("stage4_row_atlas_cell_width", 180)
GENERIC_HEATMAP_CONFIG.setdefault("stage4_row_atlas_cell_height", 100)
GENERIC_HEATMAP_CONFIG.setdefault("stage4_row_atlas_columns", 7)
GENERIC_HEATMAP_CONFIG.setdefault("stage4_cell_inner_padding_fraction", 0.06)

PROMPT_VERSIONS["stage4_core_indexed_row"] = "stage4_core_indexed_row_v1.1"
AI_TASK_SCHEMAS["stage4_core_indexed_row"] = {
    "type": "object",
    "required": ["row_index", "column_count", "cells_by_key", "manual_review", "review_reason"],
    "shape_constraints": [
        "column_count == tool-fixed columns",
        "cells_by_key contains exactly C00..Cnn",
        "row/column coordinates are assigned by code, never by the model",
        "printed_integer has an integer value_numeric",
        "non-numeric token types have value_numeric == null",
    ],
}

_validation_for_task_before_indexed_rows = _validation_for_task

def validate_stage4_indexed_row_raw(parsed, rows, cols):
    # Strict per-row validation; spatial indices must be complete and unique.
    errors = []
    if not isinstance(parsed, dict):
        return False, ["response is not a JSON object"]
    for key in AI_TASK_SCHEMAS["stage4_core_indexed_row"]["required"]:
        if key not in parsed:
            errors.append(f"missing required field: {key}")
    try:
        row_index = int(parsed.get("row_index"))
        if rows is not None and not (0 <= row_index < int(rows)):
            errors.append(f"row_index={row_index} is out of bounds")
    except Exception:
        errors.append("row_index is missing or not an integer")
    try:
        if int(parsed.get("column_count")) != int(cols):
            errors.append(f"column_count={parsed.get('column_count')} != {cols}")
    except Exception:
        errors.append("column_count is missing or not an integer")

    cells = parsed.get("cells_by_key")
    allowed = {"printed_integer", "dash", "missing", "no_printed_number", "unreadable"}
    expected_keys = {f"C{c:02d}" for c in range(int(cols))}
    if not isinstance(cells, dict):
        errors.append("cells_by_key is not an object")
    else:
        actual_keys = set(cells)
        if actual_keys != expected_keys:
            errors.append(f"cells_by_key keys must be exactly C00..C{int(cols)-1:02d}")
        for key in sorted(expected_keys):
            cell = cells.get(key)
            if not isinstance(cell, dict):
                errors.append(f"cell {key} is not an object")
                continue
            token = str(cell.get("token_type", ""))
            if token not in allowed:
                errors.append(f"cell {key} has invalid token_type={token!r}")
            value = cell.get("value_numeric")
            if token == "printed_integer":
                if isinstance(value, bool) or not isinstance(value, (int, float)) or not float(value).is_integer():
                    errors.append(f"cell {key} printed_integer requires an integer value_numeric")
            elif value is not None:
                errors.append(f"cell {key} token_type={token} requires value_numeric=null")
    return len(errors) == 0, errors

def _validation_for_task(task_name, parsed, rows=None, cols=None):
    if task_name == "stage4_core_indexed_row":
        return validate_stage4_indexed_row_raw(parsed, rows, cols)
    return _validation_for_task_before_indexed_rows(task_name, parsed, rows=rows, cols=cols)

def _tool_cell_boundaries(core_image_path, rows, cols, out_dir, safe_name):
    # Resolve tool boundaries once; fall back to equal partitions only when needed.
    image = Image.open(core_image_path).convert("RGB")
    width, height = image.size
    method = "equal_partition_of_tool_fixed_shape"
    try:
        shape = detect_core_grid_shape(core_image_path, out_dir=out_dir, safe_name=safe_name, scale=4, debug=False)
        scale = max(1, int(shape.get("scale", 1)))
        xs = [int(round(v / scale)) for v in shape.get("vertical_lines", [])]
        ys = [int(round(v / scale)) for v in shape.get("horizontal_lines", [])]
        xs = sorted(set(max(0, min(width, x)) for x in xs))
        ys = sorted(set(max(0, min(height, y)) for y in ys))
        if len(xs) == int(cols) + 1 and len(ys) == int(rows) + 1:
            method = "detected_grid_boundaries"
        else:
            xs = [round(i * width / int(cols)) for i in range(int(cols) + 1)]
            ys = [round(i * height / int(rows)) for i in range(int(rows) + 1)]
    except Exception:
        xs = [round(i * width / int(cols)) for i in range(int(cols) + 1)]
        ys = [round(i * height / int(rows)) for i in range(int(rows) + 1)]
    return image, xs, ys, method

def _cell_visual_token_evidence_proxy(cell_rgb):
    # Image-only glyph evidence proxy. It never supplies a numeric value.
    arr = np.asarray(cell_rgb.convert("RGB"))
    if arr.size == 0 or min(arr.shape[:2]) < 3:
        return "unavailable", 0
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    pad = max(1, int(min(h, w) * 0.10))
    inner = gray[pad:max(pad + 1, h - pad), pad:max(pad + 1, w - pad)]
    if inner.size < 9:
        return "unavailable", 0
    # Local contrast components can indicate glyphs, but are not treated as proof.
    med = float(np.median(inner))
    diff = np.abs(inner.astype(np.float32) - med)
    mask = (diff >= max(22.0, float(np.percentile(diff, 85)))).astype(np.uint8) * 255
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    plausible = 0
    dash_like = False
    for k in range(1, n):
        x, y, cw, ch, area = stats[k]
        if 2 <= area <= max(8, int(inner.size * 0.28)) and ch >= 2 and cw >= 1:
            plausible += 1
            if cw >= 2.2 * ch:
                dash_like = True
    if dash_like and plausible == 1:
        return "dash_like_proxy", plausible
    if plausible:
        return "possible_glyph_proxy", plausible
    return "no_glyph_detected_proxy", 0

def _group_active_projection_indices(indices, max_gap):
    groups = []
    for value in [int(x) for x in indices]:
        if not groups or value - groups[-1][-1] > int(max_gap):
            groups.append([value])
        else:
            groups[-1].append(value)
    return groups

def preflight_core_crop_completeness(core_image_path, expected_rows, expected_cols):
    # Printed glyph-band proxy. It validates crop coverage; it never supplies values.
    rgb = np.asarray(Image.open(core_image_path).convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    height, width = gray.shape
    mask = gray < 100
    row_projection = mask.sum(axis=1)
    row_active = np.where(row_projection >= max(3, int(width * 0.004)))[0]
    row_groups = _group_active_projection_indices(row_active, max_gap=3)
    row_centres = [
        int(round(np.average(group, weights=np.maximum(row_projection[group], 1))))
        for group in row_groups
    ]
    col_projection = mask.sum(axis=0)
    col_active = np.where(col_projection >= max(2, int(height * 0.010)))[0]
    col_groups = _group_active_projection_indices(col_active, max_gap=max(3, int(width / max(expected_cols, 1) * 0.30)))
    col_centres = [
        int(round(np.average(group, weights=np.maximum(col_projection[group], 1))))
        for group in col_groups
    ]
    # Ignore crop-border components, which are not text columns.
    col_centres = [x for x in col_centres if 2 < x < width - 2]
    row_intervals = np.diff(row_centres).astype(float) if len(row_centres) > 1 else np.array([])
    col_intervals = np.diff(col_centres).astype(float) if len(col_centres) > 1 else np.array([])
    row_reliable = bool(len(row_centres) >= 2 and np.median(row_intervals) > 0 and np.std(row_intervals) / np.median(row_intervals) < 0.35)
    col_reliable = bool(len(col_centres) >= 2 and np.median(col_intervals) > 0 and np.std(col_intervals) / np.median(col_intervals) < 0.35)
    errors = []
    if row_reliable and len(row_centres) != int(expected_rows):
        errors.append(f"glyph-band row count={len(row_centres)} != expected rows={expected_rows}")
    if col_reliable and len(col_centres) != int(expected_cols):
        errors.append(f"glyph-band column count={len(col_centres)} != expected columns={expected_cols}")
    return {
        "check_name": "printed_glyph_band_crop_completeness_proxy",
        "expected_rows": int(expected_rows), "expected_columns": int(expected_cols),
        "detected_row_glyph_bands": len(row_centres),
        "detected_column_glyph_bands": len(col_centres),
        "row_band_centres_px": row_centres, "column_band_centres_px": col_centres,
        "row_count_reliable": row_reliable, "column_count_reliable": col_reliable,
        "preflight_pass": not errors,
        "validation_errors": errors,
        "manual_review": bool(errors or not (row_reliable and col_reliable)),
        "manual_review_reason": "; ".join(errors) if errors else ("glyph-band completeness unavailable; inspect crop manually" if not (row_reliable and col_reliable) else ""),
        "does_not_supply_numeric_values": True,
    }

def build_indexed_row_atlas(core_image_path, row_index, rows, cols, out_path, out_dir=None, safe_name="core"):
    # Create one row atlas. Labels are outside cell pixels and never enter the crop.
    image, xs, ys, boundary_method = _tool_cell_boundaries(
        core_image_path, rows, cols, out_dir or Path(out_path).parent, safe_name
    )
    tile_w = int(GENERIC_HEATMAP_CONFIG["stage4_row_atlas_cell_width"])
    tile_h = int(GENERIC_HEATMAP_CONFIG["stage4_row_atlas_cell_height"])
    label_h = 30
    atlas_cols = min(int(cols), int(GENERIC_HEATMAP_CONFIG["stage4_row_atlas_columns"]))
    atlas_rows = int(math.ceil(int(cols) / atlas_cols))
    slot_h = tile_h + label_h
    atlas = Image.new("RGB", (tile_w * atlas_cols, slot_h * atlas_rows), "white")
    draw = ImageDraw.Draw(atlas)
    proxies = []
    for c in range(int(cols)):
        x0, x1 = xs[c], xs[c + 1]
        y0, y1 = ys[int(row_index)], ys[int(row_index) + 1]
        frac = float(GENERIC_HEATMAP_CONFIG["stage4_cell_inner_padding_fraction"])
        px = max(0, int((x1 - x0) * frac))
        py = max(0, int((y1 - y0) * frac))
        crop = image.crop((min(x1 - 1, x0 + px), min(y1 - 1, y0 + py), max(x0 + px + 1, x1 - px), max(y0 + py + 1, y1 - py)))
        evidence, component_count = _cell_visual_token_evidence_proxy(crop)
        proxies.append({
            "row_index": int(row_index), "col_index": c,
            "visual_token_evidence_proxy": evidence,
            "visual_component_count_proxy": int(component_count),
        })
        atlas_r, atlas_c = divmod(c, atlas_cols)
        left, top = atlas_c * tile_w, atlas_r * slot_h
        fitted = crop.resize((tile_w - 12, tile_h - 12), Image.Resampling.LANCZOS)
        atlas.paste(fitted, (left + 6, top + label_h + 6))
        draw.rectangle((left, top + label_h, left + tile_w - 1, top + label_h + tile_h - 1), outline=(40, 40, 40), width=2)
        draw.text((left + 8, top + 6), f"CELL C{c:02d}", fill=(0, 0, 0))
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    atlas.save(out_path)
    return {"atlas_path": str(out_path), "boundary_method": boundary_method, "cell_proxies": proxies}

def build_indexed_row_prompt(row_index, rows, cols):
    return f'''You are reading exactly one tool-segmented row from a scientific heatmap.
The fixed full matrix has exactly {rows} rows and {cols} columns. This image is row {row_index}.
The code has cropped every cell separately into a compact atlas. Each tile has a large label CELL C00 through CELL C{cols-1:02d}.

For every key, inspect only the tile carrying that exact CELL label.
Do not move a value to another column. Do not reconstruct the full matrix. Do not infer a number from colour.

Use token_type:
- "printed_integer": a complete integer is visibly printed in that cell; copy it to value_numeric.
- "dash": a dash is visibly printed; value_numeric must be null.
- "missing": the publisher marks the cell as missing/not tested; value_numeric must be null.
- "no_printed_number": no numeric text is printed; value_numeric must be null.
- "unreadable": text may be present but cannot be read reliably; value_numeric must be null and manual_review true.

Return JSON only:
{{
  "row_index": {row_index},
  "column_count": {cols},
  "cells_by_key": {{
    "C00": {{"token_type": "printed_integer|dash|missing|no_printed_number|unreadable", "value_numeric": null, "value_text": "", "manual_review": false, "review_reason": ""}}
  }},
  "manual_review": false,
  "review_reason": "",
  "confidence": "high|medium|low"
}}

cells_by_key must contain exactly the keys C00 through C{cols-1:02d}, once each. Do not return col_index; code assigns each coordinate from the key. Confidence is only a model self-report; no numeric confidence threshold is used.'''

def _call_qwen_indexed_row(atlas_path, prompt, model_name, row_index, rows, cols, context):
    image_url = image_to_data_url(atlas_path)
    def invoke_once():
        client = build_client()
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]}],
            temperature=AI_AUDIT_CONFIG["temperature"],
            top_p=AI_AUDIT_CONFIG["top_p"],
        )
        return response.choices[0].message.content or ""
    parsed, raw, audit = run_audited_json_inference(
        invoke_once,
        prompt=prompt,
        task_name="stage4_core_indexed_row",
        stage="Stage 4 coordinate-locked row extraction",
        model=model_name,
        context={**context, "panel": f"{context.get('panel', '')}:row_{row_index}"},
        rows=rows,
        cols=cols,
    )
    return parsed, raw, audit

def ai_extract_core_matrix_indexed_rows(core_image_path, model_name, out_dir, safe_name, tool_row_count, tool_column_count):
    # Extract every coordinate exactly once; the model never controls row placement.
    out_dir = Path(out_dir)
    atlas_dir = out_dir / "indexed_row_atlases" / safe_name
    row_json_dir = out_dir / "ai_json" / f"{safe_name}_indexed_rows"
    row_raw_dir = out_dir / "ai_raw" / f"{safe_name}_indexed_rows"
    for path in [atlas_dir, row_json_dir, row_raw_dir]:
        path.mkdir(parents=True, exist_ok=True)
    aggregate_json_path = out_dir / "ai_json" / f"{safe_name}_core_matrix_indexed_rows.json"
    aggregate_raw_path = out_dir / "ai_raw" / f"{safe_name}_core_matrix_indexed_rows.txt"
    preflight = preflight_core_crop_completeness(core_image_path, tool_row_count, tool_column_count)
    preflight_path = out_dir / "validation" / f"{safe_name}_core_crop_preflight.json"
    preflight_path.parent.mkdir(parents=True, exist_ok=True)
    save_json(preflight, preflight_path)
    if preflight["validation_errors"]:
        raise ValueError(
            "Core crop failed completeness preflight; Qwen was not called: "
            + "; ".join(preflight["validation_errors"])
        )
    if CACHE_EXISTING and aggregate_json_path.exists():
        parsed = load_json(aggregate_json_path)
        raw = aggregate_raw_path.read_text(encoding="utf-8") if aggregate_raw_path.exists() else ""
        print("Loaded cached coordinate-locked core matrix:", aggregate_json_path)
        return {"parsed": parsed, "raw": raw, "raw_path": aggregate_raw_path, "json_path": aggregate_json_path}

    all_records, matrix, row_audits, proxy_rows, raw_parts = [], [], [], [], []
    boundary_methods = set()
    paper_id = PAPER_CONFIG.get("paper_id", ACTIVE_PAPER)
    for r in range(int(tool_row_count)):
        atlas_path = atlas_dir / f"row_{r:03d}.png"
        atlas_meta = build_indexed_row_atlas(
            core_image_path, r, tool_row_count, tool_column_count, atlas_path,
            out_dir=out_dir, safe_name=f"{safe_name}_indexed"
        )
        boundary_methods.add(atlas_meta["boundary_method"])
        proxy_by_col = {int(x["col_index"]): x for x in atlas_meta["cell_proxies"]}
        proxy_rows.extend(atlas_meta["cell_proxies"])
        prompt = build_indexed_row_prompt(r, tool_row_count, tool_column_count)
        parsed_row, raw_row, audit = _call_qwen_indexed_row(
            atlas_path, prompt, model_name, r, tool_row_count, tool_column_count,
            {"paper": paper_id, "figure": safe_name, "panel": safe_name},
        )
        (row_raw_dir / f"row_{r:03d}.txt").write_text(str(raw_row), encoding="utf-8")
        save_json(parsed_row, row_json_dir / f"row_{r:03d}.json")
        raw_parts.append(f"===== row {r} =====\n{raw_row}")
        row_valid, row_errors = validate_stage4_indexed_row_raw(parsed_row, tool_row_count, tool_column_count)
        cells_by_key = parsed_row.get("cells_by_key", {}) if isinstance(parsed_row, dict) else {}
        matrix_row = []
        for c in range(int(tool_column_count)):
            cell = dict(cells_by_key.get(f"C{c:02d}", {})) if isinstance(cells_by_key, dict) else {}
            token = str(cell.get("token_type") or "unreadable")
            value = cell.get("value_numeric") if token == "printed_integer" else None
            if isinstance(value, float) and value.is_integer():
                value = int(value)
            proxy = proxy_by_col.get(c, {})
            mismatch = proxy.get("visual_token_evidence_proxy") == "possible_glyph_proxy" and token in {"missing", "no_printed_number"}
            review = bool(cell.get("manual_review", False) or not row_valid or token == "unreadable" or mismatch)
            reasons = []
            if cell.get("review_reason"):
                reasons.append(str(cell["review_reason"]))
            if not row_valid:
                reasons.append("raw indexed-row schema invalid: " + "; ".join(row_errors))
            if mismatch:
                reasons.append("visual glyph-evidence proxy disagrees with empty AI token; inspect manually")
            record = {
                "row_index": r, "col_index": c,
                "value_text": str(cell.get("value_text") or ("" if value is None else value)),
                "value_numeric": value,
                "token_type": token,
                "manual_review": review,
                "review_reason": "; ".join(reasons),
                "coordinate_assignment": "tool_assigned",
                "visual_token_evidence_proxy": proxy.get("visual_token_evidence_proxy", "unavailable"),
                "visual_component_count_proxy": proxy.get("visual_component_count_proxy"),
            }
            all_records.append(record)
            matrix_row.append(value)
        matrix.append(matrix_row)
        row_audits.append({
            "row_index": r, "schema_valid": bool(row_valid), "validation_errors": row_errors,
            "atlas_path": str(atlas_path), "raw_response_path": str(row_raw_dir / f"row_{r:03d}.txt"),
            "parsed_json_path": str(row_json_dir / f"row_{r:03d}.json"),
            "retry_count": audit.get("retry_count"), "manual_review": audit.get("manual_review"),
        })

    parsed = {
        "row_count": int(tool_row_count), "column_count": int(tool_column_count),
        "matrix": matrix, "records": all_records,
        "extraction_mode": "coordinate_locked_indexed_rows",
        "coordinate_assignment": "tool_assigned",
        "boundary_method": sorted(boundary_methods),
        "core_crop_completeness_preflight": preflight,
        "core_crop_preflight_path": str(preflight_path),
        "row_audits": row_audits,
        "manual_review": any(x["manual_review"] for x in all_records),
        "review_reason": "Inspect cells flagged by row schema or visual-evidence proxy." if any(x["manual_review"] for x in all_records) else "",
        "model_self_reported_confidence_only": True,
        "uses_numeric_confidence_threshold": False,
        "prompt_version": PROMPT_VERSIONS["stage4_core_indexed_row"],
    }
    aggregate_raw_path.write_text("\n\n".join(raw_parts), encoding="utf-8")
    save_json(parsed, aggregate_json_path)
    save_json({"rows": row_audits, "cell_visual_evidence_proxies": proxy_rows}, out_dir / "validation" / f"{safe_name}_indexed_row_validation.json")
    print("Saved coordinate-locked core matrix:", aggregate_json_path)
    return {"parsed": parsed, "raw": "\n\n".join(raw_parts), "raw_path": aggregate_raw_path, "json_path": aggregate_json_path}

_ai_extract_core_matrix_legacy = ai_extract_core_matrix_fixed_shape

def ai_extract_core_matrix_fixed_shape(core_image_path, model_name, out_dir, safe_name, tool_row_count, tool_column_count):
    mode = GENERIC_HEATMAP_CONFIG.get("stage4_core_extraction_mode", "indexed_row_atlas")
    if mode == "legacy_whole_matrix":
        return _ai_extract_core_matrix_legacy(
            core_image_path, model_name, out_dir, safe_name, tool_row_count, tool_column_count
        )
    if mode != "indexed_row_atlas":
        raise ValueError(f"Unknown stage4_core_extraction_mode: {mode}")
    return ai_extract_core_matrix_indexed_rows(
        core_image_path, model_name, out_dir, safe_name, tool_row_count, tool_column_count
    )

print("Stage 4 core extraction mode:", GENERIC_HEATMAP_CONFIG["stage4_core_extraction_mode"])
print("Coordinate policy: tool-assigned row/column; Qwen reads cell content only.")


In [ ]:
STAGE4_INPUT_PATH = stage3b3c_result["clean_handoff_xlsx"]
stage4_result = run_stage4_generic_all_selected(
    stage3_selected_manifest_path=STAGE4_INPUT_PATH,
    out_root=str(RUN_ROOT / "stage4_visual"),
    model_name="qwen3-vl-plus",
    figure_unit_filter=PAPER_CONFIG.get("figure_unit_filter"),
    limit=None,
    debug_display=True,
)


In [ ]:
# ============================================================
# STAGE 4B: optional publisher Source Data fallback
# Use source tables only when the figure does not print every value.
# The original sheet and row are retained for provenance; no value is
# inferred from colour intensity.
# ============================================================

def _normalise_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def _read_all_tabular_sheets(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return {"csv": pd.read_csv(path)}
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=None)
    raise ValueError(f"Source data must be CSV/XLSX, got: {path}")


def _find_column(df, aliases):
    norm = {_normalise_name(c): c for c in df.columns}
    for alias in aliases:
        a = _normalise_name(alias)
        if a in norm:
            return norm[a]
    for alias in aliases:
        a = _normalise_name(alias)
        for n, original in norm.items():
            if a and (a in n or n in a):
                return original
    return None


def run_stage4_source_data_fallback(source_data_path, out_root):
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    if not source_data_path:
        return None
    source_data_path = Path(source_data_path)
    if not source_data_path.exists():
        raise FileNotFoundError(source_data_path)

    sheets = _read_all_tabular_sheets(source_data_path)
    records, review = [], []
    value_aliases = [
        "elispot value", "ifng elispot", "ifn-g elispot", "spot forming units",
        "sfu", "response", "background subtracted", "magnitude"
    ]
    id_aliases = ["peptide id", "peptide identification", "epitope id", "neoantigen id", "peptide"]
    patient_aliases = ["patient", "patient id", "subject", "participant"]
    pool_aliases = ["peptide pool", "pool", "vaccine pool"]
    time_aliases = ["timepoint", "time point", "week", "visit", "day"]

    for sheet_name, raw in sheets.items():
        if raw is None or raw.empty:
            continue
        df = raw.copy()
        value_col = _find_column(df, value_aliases)
        if value_col is None:
            review.append({
                "Source_Sheet": sheet_name,
                "Reason": "No unambiguous ELISpot numeric column detected; sheet preserved without guessing",
                "Columns": " | ".join(map(str, df.columns)),
            })
            df.to_csv(out_root / f"raw_{re.sub(r'[^A-Za-z0-9]+','_',sheet_name)}.csv", index=False, encoding="utf-8-sig")
            continue

        id_col = _find_column(df, id_aliases)
        patient_col = _find_column(df, patient_aliases)
        pool_col = _find_column(df, pool_aliases)
        time_col = _find_column(df, time_aliases)
        for row_index, row in df.iterrows():
            value = pd.to_numeric(row.get(value_col), errors="coerce")
            if pd.isna(value):
                continue
            records.append({
                "Figure_Unit_ID": "publisher_source_data",
                "Panel_ID": "Not specified",
                "X_Axis_Value": clean_text(row.get(time_col)) if time_col else "Not specified",
                "Y_Axis_Value": clean_text(row.get(id_col)) if id_col else "Not specified",
                "Epitope_ID": clean_text(row.get(id_col)) if id_col else "Not specified",
                "Timepoint": clean_text(row.get(time_col)) if time_col else "Not specified",
                "Condition": clean_text(row.get(pool_col)) if pool_col else "Not specified",
                "Patient_ID": clean_text(row.get(patient_col)) if patient_col else "Not specified",
                "ELISpot_Value": float(value),
                "Value_Source": "publisher_source_data",
                "Source_Data_File": str(source_data_path),
                "Source_Data_Sheet": sheet_name,
                "Source_Data_Row": int(row_index) + 2,
                "Extraction_Method": "direct_tabular_value",
                "Manual_Review": False,
                "Review_Reason": "",
            })

    records_df = pd.DataFrame(records)
    review_df = pd.DataFrame(review)
    records_path = out_root / "Stage4_Source_Data_Cell_Records.xlsx"
    records_df.to_excel(records_path, index=False)
    records_df.to_csv(out_root / "Stage4_Source_Data_Cell_Records.csv", index=False, encoding="utf-8-sig")
    review_df.to_csv(out_root / "Stage4_Source_Data_Review.csv", index=False, encoding="utf-8-sig")
    qc = {
        "source_data": str(source_data_path),
        "sheet_count": len(sheets),
        "extracted_numeric_records": len(records_df),
        "review_sheet_count": len(review_df),
        "policy": "Direct table values only; no colour-intensity inference",
    }
    save_json(qc, out_root / "Stage4_Source_Data_QC.json")
    print("Stage 4B source-data records:", len(records_df))
    print("Saved:", records_path)
    return {"records_df": records_df, "records_xlsx": records_path, "review_df": review_df, "qc": qc}

stage4_source_data_result = run_stage4_source_data_fallback(
    source_data_path=SOURCE_DATA_PATH,
    out_root=RUN_ROOT / "stage4_source_data",
) if SOURCE_DATA_PATH else None


In [22]:

# ============================================================
# Reviewer Comment 7 — cell-level validation and failure-boundary analysis
# No Qwen call is made here. No ground truth is inferred or fabricated.
# If verified manual ground truth is absent, only a prefilled template is created.
# ============================================================
import math

COMMENT7_OUT_DIR = PROJECT_ROOT / "reviewer_comment_7_outputs"
COMMENT7_OUT_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_GT_TEMPLATE_PATH = COMMENT7_OUT_DIR / "manual_cell_validation_template.csv"
GROUND_TRUTH_STATUS_PATH = COMMENT7_OUT_DIR / "ground_truth_status.json"
IMAGE_PROXY_METRICS_PATH = COMMENT7_OUT_DIR / "image_proxy_metrics.csv"
MIN_GROUP_N = 5

def locate_stage4_cell_records():
    candidates = [
        Path(RUN_ROOT) / "stage4_final_generic_outputs_after_stage3b3c" / "Stage4_Cell_Records.xlsx",
        Path(RUN_ROOT) / "stage4_final_generic_outputs_after_stage3b3c" / "Stage4_Cell_Records.csv",
        PROJECT_ROOT / "stage4_final_generic_outputs_after_stage3b3c" / "Stage4_Cell_Records.xlsx",
        PROJECT_ROOT / "stage4_final_generic_outputs_after_stage3b3c" / "Stage4_Cell_Records.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("No cached Stage4_Cell_Records.xlsx/csv was found.")

def load_stage4_cell_records(path=None):
    path = Path(path or locate_stage4_cell_records())
    return pd.read_excel(path) if path.suffix.lower() in {".xlsx", ".xls"} else pd.read_csv(path)

def _first_existing_column(df, names, default=None):
    for name in names:
        if name in df.columns:
            return df[name]
    return pd.Series([default] * len(df), index=df.index)

def _canonical_prediction_table(records_df):
    out = pd.DataFrame(index=records_df.index)
    out["paper_id"] = PAPER_CONFIG.get("paper_id", ACTIVE_PAPER)
    out["source_document"] = _first_existing_column(
        records_df, ["Source_Document", "source_document", "source_doc"], PAPER_CONFIG.get("main_pdf", "")
    )
    out["figure_unit_id"] = _first_existing_column(records_df, ["Figure_Unit_ID", "figure_unit_id"])
    out["figure"] = _first_existing_column(records_df, ["Figure_Number", "figure_number", "Figure_Unit_ID"])
    out["panel_id"] = _first_existing_column(records_df, ["Panel_ID", "panel_id"], "unknown")
    out["row_index"] = pd.to_numeric(_first_existing_column(records_df, ["Row_Index", "row_index"]), errors="coerce")
    out["column_index"] = pd.to_numeric(_first_existing_column(records_df, ["Col_Index", "Column_Index", "column_index"]), errors="coerce")
    out["predicted_value"] = _first_existing_column(records_df, ["ELISpot_Value", "Predicted_Value", "predicted_value"])
    out["x_value"] = _first_existing_column(records_df, ["X_Value", "x_value"])
    out["y_value"] = _first_existing_column(records_df, ["Y_Value", "y_value"])
    out["x_axis_role"] = _first_existing_column(records_df, ["X_Axis_Role", "x_axis_role"])
    out["y_axis_role"] = _first_existing_column(records_df, ["Y_Axis_Role", "y_axis_role"])
    out["core_image_path"] = _first_existing_column(records_df, ["Core_Image_Path", "core_image_path"])
    out["tool_row_count"] = pd.to_numeric(_first_existing_column(records_df, ["Tool_Row_Count", "tool_row_count"]), errors="coerce")
    out["tool_column_count"] = pd.to_numeric(_first_existing_column(records_df, ["Tool_Column_Count", "tool_column_count"]), errors="coerce")
    return out

def _detect_grid_lines_for_validation(image_path, expected_rows, expected_cols):
    """Use existing Stage 4 detector when available; otherwise return unavailable."""
    try:
        info = detect_core_grid_shape(image_path, out_dir=None, safe_name="comment7", scale=4, debug=False)
        h = list(info.get("horizontal_lines") or [])
        v = list(info.get("vertical_lines") or [])
        scale = float(info.get("scale") or 4)
        if len(h) != int(expected_rows) + 1 or len(v) != int(expected_cols) + 1:
            return None, "grid_line_count_does_not_match_tool_fixed_shape"
        return {"horizontal": h, "vertical": v, "scale": scale}, "available"
    except Exception as exc:
        return None, f"unavailable: {type(exc).__name__}: {exc}"

def _cell_image_proxies(gray, x0, y0, x1, y1):
    x0, y0, x1, y1 = map(int, [x0, y0, x1, y1])
    width, height = max(0, x1 - x0), max(0, y1 - y0)
    inset = max(1, int(round(min(width, height) * 0.08)))
    inner = gray[y0 + inset:y1 - inset, x0 + inset:x1 - inset]
    if inner.size == 0:
        return np.nan, "unavailable", np.nan, "unavailable"
    contrast = float(np.percentile(inner, 90) - np.percentile(inner, 10))
    contrast_status = "available"
    try:
        blur = cv2.GaussianBlur(inner, (3, 3), 0)
        _, bw = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        n_labels, _, stats, _ = cv2.connectedComponentsWithStats(bw, connectivity=8)
        heights = []
        for idx in range(1, n_labels):
            w, h, area = int(stats[idx, cv2.CC_STAT_WIDTH]), int(stats[idx, cv2.CC_STAT_HEIGHT]), int(stats[idx, cv2.CC_STAT_AREA])
            if 2 <= area <= inner.size * 0.25 and 1 <= h <= inner.shape[0] * 0.8 and 1 <= w <= inner.shape[1] * 0.8:
                heights.append(h)
        if heights:
            text_proxy, text_status = float(np.median(heights)), "available"
        else:
            text_proxy, text_status = np.nan, "unavailable_no_reliable_components"
    except Exception as exc:
        text_proxy, text_status = np.nan, f"unavailable: {type(exc).__name__}"
    return text_proxy, text_status, contrast, contrast_status

def compute_cell_image_proxy_metrics(prediction_df):
    rows = []
    key_cols = ["paper_id", "figure_unit_id", "panel_id", "row_index", "column_index"]
    group_cols = ["paper_id", "figure_unit_id", "panel_id", "core_image_path", "tool_row_count", "tool_column_count"]
    for keys, group in prediction_df.groupby(group_cols, dropna=False):
        paper, figure_unit, panel, image_path, n_rows, n_cols = keys
        base = {"paper_id": paper, "figure_unit_id": figure_unit, "panel_id": panel}
        try:
            n_rows, n_cols = int(n_rows), int(n_cols)
            image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if image is None:
                raise FileNotFoundError(str(image_path))
            grid, grid_status = _detect_grid_lines_for_validation(image_path, n_rows, n_cols)
            if grid is None:
                for _, pred in group.iterrows():
                    rows.append({**{k: pred[k] for k in key_cols}, "cell_width_px": np.nan, "cell_height_px": np.nan,
                                 "cell_area_px2": np.nan, "mean_cell_area_px2": np.nan, "matrix_cell_density": np.nan,
                                 "text_height_proxy_px": np.nan, "text_height_proxy_status": "unavailable",
                                 "visual_contrast_proxy": np.nan, "visual_contrast_proxy_status": "unavailable",
                                 "grid_metric_status": grid_status})
                continue
            h_lines = np.array(grid["horizontal"], dtype=float) / grid["scale"]
            v_lines = np.array(grid["vertical"], dtype=float) / grid["scale"]
            cell_areas = [max(0, v_lines[c+1]-v_lines[c]) * max(0, h_lines[r+1]-h_lines[r])
                          for r in range(n_rows) for c in range(n_cols)]
            mean_area = float(np.mean(cell_areas)) if cell_areas else np.nan
            matrix_area = max(0, v_lines[-1]-v_lines[0]) * max(0, h_lines[-1]-h_lines[0])
            density = float((n_rows*n_cols) / matrix_area) if matrix_area > 0 else np.nan
            for _, pred in group.iterrows():
                r, c = int(pred["row_index"]), int(pred["column_index"])
                if not (0 <= r < n_rows and 0 <= c < n_cols):
                    continue
                x0, x1, y0, y1 = v_lines[c], v_lines[c+1], h_lines[r], h_lines[r+1]
                text_h, text_status, contrast, contrast_status = _cell_image_proxies(image, x0, y0, x1, y1)
                rows.append({**{k: pred[k] for k in key_cols},
                             "cell_width_px": float(x1-x0), "cell_height_px": float(y1-y0),
                             "cell_area_px2": float((x1-x0)*(y1-y0)), "mean_cell_area_px2": mean_area,
                             "matrix_cell_density": density, "text_height_proxy_px": text_h,
                             "text_height_proxy_status": text_status, "visual_contrast_proxy": contrast,
                             "visual_contrast_proxy_status": contrast_status, "grid_metric_status": "available"})
        except Exception as exc:
            for _, pred in group.iterrows():
                rows.append({**{k: pred[k] for k in key_cols}, "cell_width_px": np.nan, "cell_height_px": np.nan,
                             "cell_area_px2": np.nan, "mean_cell_area_px2": np.nan, "matrix_cell_density": np.nan,
                             "text_height_proxy_px": np.nan, "text_height_proxy_status": "unavailable",
                             "visual_contrast_proxy": np.nan, "visual_contrast_proxy_status": "unavailable",
                             "grid_metric_status": f"unavailable: {type(exc).__name__}: {exc}"})
    return pd.DataFrame(rows)

def create_manual_cell_validation_template(prediction_df, proxy_df, path=MANUAL_GT_TEMPLATE_PATH):
    keys = ["paper_id", "figure_unit_id", "panel_id", "row_index", "column_index"]
    template = prediction_df.merge(proxy_df, on=keys, how="left") if len(proxy_df) else prediction_df.copy()
    template["ground_truth_value"] = ""  # intentionally blank; must be entered by a human
    template["ground_truth_source"] = ""
    template["validated_by"] = ""
    template["validation_notes"] = ""
    template.to_csv(path, index=False, encoding="utf-8-sig")
    return template

def find_verified_ground_truth(search_root=PROJECT_ROOT):
    """Accept only explicit ground_truth_value columns with at least one nonblank value."""
    candidates = [MANUAL_GT_TEMPLATE_PATH]
    candidates += [p for p in Path(search_root).rglob("*.csv") if p != MANUAL_GT_TEMPLATE_PATH]
    candidates += list(Path(search_root).rglob("*.xlsx"))
    for path in candidates:
        try:
            header = pd.read_excel(path, nrows=5) if path.suffix.lower() in {".xlsx", ".xls"} else pd.read_csv(path, nrows=5)
        except Exception:
            continue
        gt_col = next((c for c in header.columns if str(c).strip().lower() in {"ground_truth_value", "ground_truth"}), None)
        if not gt_col:
            continue
        try:
            frame = pd.read_excel(path) if path.suffix.lower() in {".xlsx", ".xls"} else pd.read_csv(path)
        except Exception:
            continue
        numeric_gt = pd.to_numeric(frame[gt_col], errors="coerce")
        if numeric_gt.notna().any():
            frame = frame.rename(columns={gt_col: "ground_truth_value"})
            return path, frame
    return None, None

def _safe_qcut(series, requested=4):
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    if valid.nunique() < 2:
        return pd.Series(pd.NA, index=series.index, dtype="object")
    try:
        codes = pd.qcut(numeric, q=min(requested, valid.nunique()), labels=False, duplicates="drop")
    except Exception:
        return pd.Series(pd.NA, index=series.index, dtype="object")
    actual = int(codes.dropna().max()) + 1 if codes.notna().any() else 0
    names = ["low", "mid-low", "mid-high", "high"]
    if actual == 2:
        labels = ["low", "high"]
    elif actual == 3:
        labels = ["low", "mid", "high"]
    else:
        labels = names[:actual]
    return codes.map({i: labels[i] for i in range(actual)}).astype("object")

def _error_type(pred, gt):
    if pd.isna(pred): return "prediction_missing"
    if pd.isna(gt): return "ground_truth_missing"
    return "exact" if float(pred) == float(gt) else "numeric_mismatch"

def _group_error_table(df, group_col):
    table = df.groupby(group_col, dropna=False).agg(n=("exact_match", "size"), errors=("exact_match", lambda s: int((~s).sum()))).reset_index()
    table["error_rate"] = table["errors"] / table["n"]
    table["insufficient_sample"] = table["n"] < MIN_GROUP_N
    return table

def _parse_timepoint(value):
    text = str(value or "").strip().lower()
    match = re.fullmatch(r"\s*([-+]?\d+(?:\.\d+)?)\s*(h|hr|hrs|hour|hours|d|day|days|w|wk|week|weeks|m|month|months)?\s*", text)
    if not match:
        return np.nan
    number = float(match.group(1)); unit = match.group(2) or ""
    factor = {"": 1, "h": 1, "hr": 1, "hrs": 1, "hour": 1, "hours": 1,
              "d": 24, "day": 24, "days": 24, "w": 168, "wk": 168, "week": 168, "weeks": 168,
              "m": 24*30, "month": 24*30, "months": 24*30}.get(unit)
    return number * factor if factor is not None else np.nan

def add_transition_analysis(df):
    out = df.copy()
    out["delta_value"] = np.nan
    out["transition"] = pd.NA
    applicable_any = False
    for (figure, panel), panel_df in out.groupby(["figure_unit_id", "panel_id"], dropna=False):
        x_role = str(panel_df["x_axis_role"].dropna().iloc[0]).lower() if panel_df["x_axis_role"].notna().any() else ""
        y_role = str(panel_df["y_axis_role"].dropna().iloc[0]).lower() if panel_df["y_axis_role"].notna().any() else ""
        if x_role == "timepoint":
            time_index, time_label, other_index = "column_index", "x_value", "row_index"
        elif y_role == "timepoint":
            time_index, time_label, other_index = "row_index", "y_value", "column_index"
        else:
            continue
        times = panel_df[[time_index, time_label]].drop_duplicates(time_index).copy()
        times["parsed_time"] = times[time_label].map(_parse_timepoint)
        if times["parsed_time"].isna().any() or times["parsed_time"].duplicated().any():
            continue
        applicable_any = True
        order = times.sort_values("parsed_time")[time_index].tolist()
        for _, sequence_df in panel_df.groupby(other_index):
            idx_by_time = {int(out.loc[i, time_index]): i for i in sequence_df.index}
            prev_gt = None
            for time_position in order:
                idx = idx_by_time.get(int(time_position))
                if idx is None: continue
                gt = out.loc[idx, "ground_truth_value"]
                if prev_gt is not None and pd.notna(gt) and pd.notna(prev_gt):
                    delta = float(gt) - float(prev_gt)
                    out.loc[idx, "delta_value"] = delta
                    out.loc[idx, "transition"] = "rising" if delta > 0 else ("falling" if delta < 0 else "unchanged")
                prev_gt = gt
    return out, applicable_any

def adjacent_column_error_analysis(df):
    rows = []
    for (figure, panel), panel_df in df.groupby(["figure_unit_id", "panel_id"], dropna=False):
        counts = {}
        gt_lookup = {(int(r.row_index), int(r.column_index)): r.ground_truth_value for r in panel_df.itertuples()}
        for offset in [0, -1, 1]:
            matched = 0; compared = 0
            for rec in panel_df.itertuples():
                gt = gt_lookup.get((int(rec.row_index), int(rec.column_index) + offset))
                if pd.notna(rec.predicted_value) and pd.notna(gt):
                    compared += 1
                    matched += int(float(rec.predicted_value) == float(gt))
            counts[offset] = (matched, compared)
        best = max(counts, key=lambda k: counts[k][0])
        shift = "adjacent_column_shift_left" if best == -1 and counts[-1][0] > counts[0][0] else (
                "adjacent_column_shift_right" if best == 1 and counts[1][0] > counts[0][0] else "other")
        for offset in [0, -1, 1]:
            rows.append({"figure_unit_id": figure, "panel_id": panel, "offset": offset,
                         "matches": counts[offset][0], "comparisons": counts[offset][1], "shift_classification": shift})
    return pd.DataFrame(rows)

def _plot_error_rate(table, x_col, title, out_path):
    if table.empty: return
    ax = table.plot(kind="bar", x=x_col, y="error_rate", legend=False, figsize=(7, 4), color="#4472C4")
    ax.set_title(title); ax.set_ylabel("Error rate"); ax.set_ylim(0, 1)
    plt.tight_layout(); plt.savefig(out_path, dpi=180); plt.close()

def run_cell_level_error_analysis(ground_truth_df, prediction_df, proxy_df, out_dir=COMMENT7_OUT_DIR):
    keys = ["paper_id", "figure_unit_id", "panel_id", "row_index", "column_index"]
    gt = ground_truth_df.copy()
    for key in keys:
        if key not in gt.columns:
            raise ValueError(f"Ground-truth file is missing required key: {key}")
    gt["ground_truth_value"] = pd.to_numeric(gt["ground_truth_value"], errors="coerce")
    gt = gt[gt["ground_truth_value"].notna()].copy()
    if gt.empty:
        raise ValueError("No real numeric ground_truth_value entries are available.")
    base = prediction_df.merge(proxy_df, on=keys, how="left") if len(proxy_df) else prediction_df.copy()
    keep = keys + ["ground_truth_value"]
    merged = base.merge(gt[keep].drop_duplicates(keys), on=keys, how="inner")
    merged["predicted_value"] = pd.to_numeric(merged["predicted_value"], errors="coerce")
    merged["exact_match"] = merged["predicted_value"].eq(merged["ground_truth_value"])
    merged["absolute_error"] = (merged["predicted_value"] - merged["ground_truth_value"]).abs()
    merged["error_type"] = [_error_type(p, g) for p, g in zip(merged["predicted_value"], merged["ground_truth_value"])]
    merged["value_range"] = _safe_qcut(merged["ground_truth_value"])
    merged["cell_size_bin"] = _safe_qcut(merged["cell_area_px2"])
    merged["density_bin"] = _safe_qcut(merged["matrix_cell_density"])
    merged["text_height_proxy_bin"] = _safe_qcut(merged["text_height_proxy_px"])
    merged["visual_contrast_proxy_bin"] = _safe_qcut(merged["visual_contrast_proxy"])
    merged, transition_applicable = add_transition_analysis(merged)
    merged.to_csv(Path(out_dir) / "cell_level_validation.csv", index=False, encoding="utf-8-sig")

    outputs = {
        "error_by_panel.csv": _group_error_table(merged, "panel_id"),
        "error_by_value_range.csv": _group_error_table(merged, "value_range"),
        "error_by_cell_size.csv": pd.concat([
            _group_error_table(merged, "cell_size_bin").assign(metric="cell_area_px2"),
            _group_error_table(merged, "density_bin").assign(metric="matrix_cell_density")
        ], ignore_index=True),
        "error_by_text_height_proxy.csv": _group_error_table(merged, "text_height_proxy_bin"),
        "error_by_visual_contrast.csv": _group_error_table(merged, "visual_contrast_proxy_bin"),
        "error_by_type.csv": _group_error_table(merged, "error_type"),
    }
    if transition_applicable and merged["transition"].notna().any():
        outputs["error_by_transition.csv"] = _group_error_table(merged[merged["transition"].notna()], "transition")
    failures = merged[~merged["exact_match"]].copy()
    failures.to_csv(Path(out_dir) / "failure_cases.csv", index=False, encoding="utf-8-sig")
    adjacent_column_error_analysis(merged).to_csv(Path(out_dir) / "adjacent_column_error_analysis.csv", index=False, encoding="utf-8-sig")
    boundary_parts = []
    for metric in ["cell_size_bin", "density_bin", "text_height_proxy_bin", "visual_contrast_proxy_bin",
                   "value_range", "transition", "panel_id", "row_index", "column_index"]:
        subset = merged if metric != "transition" else merged[merged["transition"].notna()]
        if subset.empty: continue
        part = _group_error_table(subset, metric).rename(columns={metric: "group"})
        part.insert(0, "metric", metric)
        boundary_parts.append(part)
    pd.concat(boundary_parts, ignore_index=True).to_csv(Path(out_dir) / "failure_boundary_summary.csv", index=False, encoding="utf-8-sig")
    for name, table in outputs.items():
        table.to_csv(Path(out_dir) / name, index=False, encoding="utf-8-sig")

    _plot_error_rate(outputs["error_by_value_range.csv"], "value_range", "Error rate by ground-truth value range", Path(out_dir) / "error_rate_by_value_range.png")
    size_plot = outputs["error_by_cell_size.csv"]
    _plot_error_rate(size_plot[size_plot["metric"] == "cell_area_px2"], "cell_size_bin", "Error rate by cell size", Path(out_dir) / "error_rate_by_cell_size.png")
    _plot_error_rate(size_plot[size_plot["metric"] == "matrix_cell_density"], "density_bin", "Error rate by matrix cell density", Path(out_dir) / "error_rate_by_cell_density.png")
    _plot_error_rate(outputs["error_by_visual_contrast.csv"], "visual_contrast_proxy_bin", "Error rate by visual contrast proxy", Path(out_dir) / "error_rate_by_visual_contrast.png")
    type_counts = outputs["error_by_type.csv"]
    ax = type_counts.plot(kind="bar", x="error_type", y="n", legend=False, figsize=(7, 4), color="#ED7D31")
    ax.set_title("Error type distribution"); ax.set_ylabel("Count")
    plt.tight_layout(); plt.savefig(Path(out_dir) / "error_type_distribution.png", dpi=180); plt.close()
    if "error_by_transition.csv" in outputs:
        _plot_error_rate(outputs["error_by_transition.csv"], "transition", "Transition error rate", Path(out_dir) / "transition_error_rate.png")
    return merged, outputs

# Execute from cached Stage 4 records only; this cell never calls Qwen.
stage4_cached_records = load_stage4_cell_records()
comment7_predictions = _canonical_prediction_table(stage4_cached_records)
comment7_proxy_metrics = compute_cell_image_proxy_metrics(comment7_predictions)
comment7_proxy_metrics.to_csv(IMAGE_PROXY_METRICS_PATH, index=False, encoding="utf-8-sig")
ground_truth_path, verified_ground_truth = find_verified_ground_truth()
if verified_ground_truth is None:
    comment7_template = create_manual_cell_validation_template(comment7_predictions, comment7_proxy_metrics)
    status = {
        "ground_truth_available": False,
        "reason": "No file with nonblank numeric ground_truth_value entries was found. Predictions/manual-review flags were not treated as ground truth.",
        "manual_validation_template": str(MANUAL_GT_TEMPLATE_PATH),
        "error_analysis_generated": False,
        "uses_fabricated_ground_truth": False,
    }
    GROUND_TRUTH_STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"No verified cell-level ground truth found. Fill: {MANUAL_GT_TEMPLATE_PATH}")
else:
    cell_level_validation, comment7_outputs = run_cell_level_error_analysis(
        verified_ground_truth, comment7_predictions, comment7_proxy_metrics
    )
    status = {
        "ground_truth_available": True,
        "ground_truth_path": str(ground_truth_path),
        "validated_cell_count": int(len(cell_level_validation)),
        "error_analysis_generated": True,
        "uses_fabricated_ground_truth": False,
    }
    GROUND_TRUTH_STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Cell-level error analysis saved to: {COMMENT7_OUT_DIR}")


Cell-level error analysis saved to: C:\Users\lenovo\Desktop\my_langchain_project\reviewer_comment_7_outputs


In [28]:

# ============================================================
# Reviewer Comment 3 — internal component ablation
# Ground truth is used only for evaluation and never enters a Qwen prompt.
# Existing fixed-shape caches are reused. The unconstrained variant makes
# at most one cached Qwen call per panel when explicitly enabled.
# ============================================================
REVIEWER3_ABLATION_DIR = PROJECT_ROOT / "reviewer3_internal_ablation"
REVIEWER3_ABLATION_DIR.mkdir(parents=True, exist_ok=True)
(REVIEWER3_ABLATION_DIR / "ai_json").mkdir(parents=True, exist_ok=True)
(REVIEWER3_ABLATION_DIR / "ai_raw").mkdir(parents=True, exist_ok=True)

REVIEWER3_UNCONSTRAINED_TASK = "reviewer3_qwen_unconstrained_core"
PROMPT_VERSIONS[REVIEWER3_UNCONSTRAINED_TASK] = "reviewer3_unconstrained_core_v1.0"
AI_TASK_SCHEMAS[REVIEWER3_UNCONSTRAINED_TASK] = {
    "type": "object",
    "required": ["row_count", "column_count", "matrix", "records", "manual_review", "review_reason"],
}

def build_unconstrained_core_prompt():
    """Ablation prompt: no OpenCV-derived row/column counts are supplied."""
    return """
You are extracting all printed numeric values from a scientific heatmap core image.

Return ONLY valid JSON. Do not output markdown.

Infer the number of rows and columns from the image yourself. Read the matrix from
top to bottom and left to right. Use zero-based row_index and col_index. Do not read
colorbar ticks or axis labels as matrix cells. If a printed value is unreadable, use
null and mark the record for manual review. Do not infer a value from color intensity.

Return JSON with this schema:
{
  "row_count": 0,
  "column_count": 0,
  "matrix": [[]],
  "records": [
    {"row_index": 0, "col_index": 0, "value_text": "0",
     "value_numeric": 0, "manual_review": false, "review_reason": ""}
  ],
  "manual_review": false,
  "review_reason": ""
}
""".strip()


def _resolve_project_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else PROJECT_ROOT / path


def _records_from_json_without_projection(parsed):
    """Convert raw records/matrix to coordinates without shape repair or clipping."""
    rows = []
    records = parsed.get("records", []) if isinstance(parsed, dict) else []
    if isinstance(records, list) and records:
        for rec in records:
            if not isinstance(rec, dict):
                continue
            rows.append({
                "row_index": pd.to_numeric(rec.get("row_index"), errors="coerce"),
                "column_index": pd.to_numeric(rec.get("col_index"), errors="coerce"),
                "predicted_value": pd.to_numeric(rec.get("value_numeric"), errors="coerce"),
            })
    else:
        matrix = parsed.get("matrix", []) if isinstance(parsed, dict) else []
        if isinstance(matrix, list):
            for row_index, row in enumerate(matrix):
                if not isinstance(row, list):
                    continue
                for column_index, value in enumerate(row):
                    rows.append({
                        "row_index": row_index,
                        "column_index": column_index,
                        "predicted_value": pd.to_numeric(value, errors="coerce"),
                    })
    return pd.DataFrame(rows, columns=["row_index", "column_index", "predicted_value"])


def run_unconstrained_panel_ablation(panel_meta, allow_qwen_calls=False,
                                     model_name="qwen3-vl-plus"):
    panel = str(panel_meta["Panel_ID"])
    figure = str(panel_meta["Figure_Unit_ID"])
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{figure}_panel_{panel}")
    json_path = REVIEWER3_ABLATION_DIR / "ai_json" / f"{safe_name}_unconstrained.json"
    raw_path = REVIEWER3_ABLATION_DIR / "ai_raw" / f"{safe_name}_unconstrained.txt"
    if json_path.exists():
        return json.loads(json_path.read_text(encoding="utf-8")), "cache"
    if not allow_qwen_calls:
        return None, "pending_qwen_call"
    prompt = build_unconstrained_core_prompt()
    image_path = _resolve_project_path(panel_meta["Core_Image_Path"])
    image_url = image_to_data_url(image_path)
    def invoke_once():
        client = build_client()
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]}],
            temperature=AI_AUDIT_CONFIG["temperature"],
            top_p=AI_AUDIT_CONFIG["top_p"],
        )
        return response.choices[0].message.content or ""
    parsed, raw, _ = run_audited_json_inference(
        invoke_once,
        prompt=prompt,
        task_name=REVIEWER3_UNCONSTRAINED_TASK,
        stage="Reviewer 3 internal ablation",
        model=model_name,
        context={"paper": PAPER_CONFIG.get("paper_id"), "figure": figure, "panel": panel},
    )
    json_path.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
    raw_path.write_text(str(raw), encoding="utf-8")
    return parsed, "new_qwen_call"


def evaluate_ablation_panel(variant, panel_meta, predicted_df, ground_truth_df,
                            raw_json=None, apply_validation_gate=False,
                            inference_source="cache"):
    panel = str(panel_meta["Panel_ID"])
    figure = str(panel_meta["Figure_Unit_ID"])
    rows, cols = int(panel_meta["Tool_Row_Count"]), int(panel_meta["Tool_Column_Count"])
    constraint = StructureConstraint(rows=rows, columns=cols)
    gt = ground_truth_df[
        (ground_truth_df["figure_unit_id"].astype(str) == figure) &
        (ground_truth_df["panel_id"].astype(str) == panel)
    ][["row_index", "column_index", "ground_truth_value"]].copy()
    gt["ground_truth_value"] = pd.to_numeric(gt["ground_truth_value"], errors="coerce")
    pred = predicted_df.copy()
    for col in ["row_index", "column_index", "predicted_value"]:
        pred[col] = pd.to_numeric(pred[col], errors="coerce")
    parse_success = isinstance(raw_json, dict) if raw_json is not None else True
    if raw_json is not None:
        schema_valid, validation_errors = validate_stage4_core_raw(raw_json, rows, cols)
    else:
        schema_valid, validation_errors = True, []
    duplicate_count = int(pred.duplicated(["row_index", "column_index"]).sum()) if len(pred) else 0
    in_bounds = pred[
        pred["row_index"].between(0, rows - 1, inclusive="both") &
        pred["column_index"].between(0, cols - 1, inclusive="both")
    ].copy()
    out_of_bounds_count = int(len(pred) - len(in_bounds))
    if apply_validation_gate and not schema_valid:
        in_bounds = in_bounds.iloc[0:0].copy()
    unique_pred = in_bounds.drop_duplicates(["row_index", "column_index"], keep="first")
    merged = gt.merge(unique_pred, on=["row_index", "column_index"], how="left")
    covered = merged["predicted_value"].notna()
    exact = covered & merged["predicted_value"].eq(merged["ground_truth_value"])
    covered_n = int(covered.sum())
    exact_n = int(exact.sum())
    return {
        "variant": variant,
        "figure_unit_id": figure,
        "panel_id": panel,
        "tool_rows": rows,
        "tool_columns": cols,
        "ground_truth_cells": int(len(gt)),
        "raw_predicted_records": int(len(pred)),
        "unique_in_bounds_records": int(len(unique_pred)),
        "coverage": covered_n / len(gt) if len(gt) else np.nan,
        "exact_matches": exact_n,
        "exact_accuracy_on_covered": exact_n / covered_n if covered_n else np.nan,
        "end_to_end_exact_accuracy": exact_n / len(gt) if len(gt) else np.nan,
        "parse_success": bool(parse_success),
        "raw_schema_valid": bool(schema_valid),
        "validation_gate_applied": bool(apply_validation_gate),
        "validation_errors": json.dumps(validation_errors, ensure_ascii=False),
        "duplicate_index_count": duplicate_count,
        "out_of_bounds_count": out_of_bounds_count,
        "manual_review": bool(not parse_success or not schema_valid),
        "inference_source": inference_source,
    }


def aggregate_ablation_results(panel_results):
    rows = []
    for variant, group in panel_results.groupby("variant", dropna=False):
        gt_n = int(group["ground_truth_cells"].sum())
        covered_n = int(round((group["coverage"] * group["ground_truth_cells"]).sum()))
        exact_n = int(group["exact_matches"].sum())
        rows.append({
            "variant": variant,
            "panels": int(len(group)),
            "ground_truth_cells": gt_n,
            "covered_cells": covered_n,
            "coverage": covered_n / gt_n if gt_n else np.nan,
            "exact_matches": exact_n,
            "exact_accuracy_on_covered": exact_n / covered_n if covered_n else np.nan,
            "end_to_end_exact_accuracy": exact_n / gt_n if gt_n else np.nan,
            "schema_valid_panels": int(group["raw_schema_valid"].sum()),
            "manual_review_panels": int(group["manual_review"].sum()),
            "duplicate_index_count": int(group["duplicate_index_count"].sum()),
            "out_of_bounds_count": int(group["out_of_bounds_count"].sum()),
        })
    return pd.DataFrame(rows)


def run_structural_fault_injection(raw_json, rows, cols):
    """Deterministic stress test; synthetic faults are labelled and never used as real data."""
    import copy
    cases = {"intact": copy.deepcopy(raw_json)}
    missing = copy.deepcopy(raw_json)
    missing["records"] = list(missing.get("records", []))[:-1]
    cases["missing_record"] = missing
    duplicate = copy.deepcopy(raw_json)
    if duplicate.get("records"):
        duplicate["records"] = list(duplicate["records"]) + [copy.deepcopy(duplicate["records"][0])]
    cases["duplicate_record"] = duplicate
    out_of_bounds = copy.deepcopy(raw_json)
    if out_of_bounds.get("records"):
        out_of_bounds["records"][0]["row_index"] = rows
    cases["out_of_bounds_index"] = out_of_bounds
    wrong_count = copy.deepcopy(raw_json)
    wrong_count["row_count"] = rows + 1
    cases["wrong_declared_shape"] = wrong_count
    wrong_matrix = copy.deepcopy(raw_json)
    wrong_matrix["matrix"] = list(wrong_matrix.get("matrix", []))[:-1]
    cases["wrong_matrix_dimension"] = wrong_matrix
    results = []
    for fault, candidate in cases.items():
        valid, errors = validate_stage4_core_raw(candidate, rows, cols)
        try:
            constraint = StructureConstraint(rows=rows, columns=cols)
            _, post = project_core_output_to_constraint(candidate, constraint)
            usable = bool(post["post_normalization_usable"])
        except Exception:
            usable = False
        results.append({
            "fault_case": fault,
            "synthetic_fault_injection": fault != "intact",
            "raw_schema_valid": bool(valid),
            "validator_detected_fault": bool(fault != "intact" and not valid),
            "post_normalization_usable": usable,
            "validation_errors": json.dumps(errors, ensure_ascii=False),
        })
    return pd.DataFrame(results)


def run_reviewer3_internal_ablation(allow_unconstrained_qwen_calls=False,
                                    model_name="qwen3-vl-plus"):
    gt_path = PROJECT_ROOT / "reviewer_comment_7_outputs" / "manual_cell_validation_from_paper.csv"
    if not gt_path.exists():
        raise FileNotFoundError(f"Verified source-figure ground truth is required: {gt_path}")
    ground_truth = pd.read_csv(gt_path)
    panel_summary_path = PROJECT_ROOT / "stage4_final_generic_outputs_after_stage3b3c" / "Stage4_Panel_Summary.xlsx"
    records_path = PROJECT_ROOT / "stage4_final_generic_outputs_after_stage3b3c" / "Stage4_Cell_Records.xlsx"
    panel_summary = pd.read_excel(panel_summary_path)
    normalized_records = pd.read_excel(records_path)
    panel_rows = []
    first_raw = None
    first_shape = None
    for _, meta in panel_summary.iterrows():
        figure, panel = str(meta["Figure_Unit_ID"]), str(meta["Panel_ID"])
        panel_record = normalized_records[
            (normalized_records["Figure_Unit_ID"].astype(str) == figure) &
            (normalized_records["Panel_ID"].astype(str) == panel)
        ].copy()
        panel_record = panel_record.rename(columns={
            "Row_Index": "row_index", "Col_Index": "column_index",
            "ELISpot_Value": "predicted_value",
        })
        raw_path = _resolve_project_path(meta["Core_AI_JSON_Path"])
        fixed_raw = json.loads(raw_path.read_text(encoding="utf-8"))
        fixed_pred = _records_from_json_without_projection(fixed_raw)
        if first_raw is None:
            first_raw = fixed_raw
            first_shape = (int(meta["Tool_Row_Count"]), int(meta["Tool_Column_Count"]))
        panel_rows.append(evaluate_ablation_panel(
            "A1_fixed_shape_prompt_raw", meta, fixed_pred, ground_truth,
            raw_json=fixed_raw, apply_validation_gate=False, inference_source="existing_cache"
        ))
        panel_rows.append(evaluate_ablation_panel(
            "A2_fixed_shape_plus_validation_gate", meta, fixed_pred, ground_truth,
            raw_json=fixed_raw, apply_validation_gate=True, inference_source="existing_cache"
        ))
        panel_rows.append(evaluate_ablation_panel(
            "A3_full_structure_constrained", meta,
            panel_record[["row_index", "column_index", "predicted_value"]], ground_truth,
            raw_json=fixed_raw, apply_validation_gate=False, inference_source="existing_cache_normalized"
        ))
        unconstrained_json, source = run_unconstrained_panel_ablation(
            meta, allow_qwen_calls=allow_unconstrained_qwen_calls, model_name=model_name
        )
        if unconstrained_json is not None:
            unconstrained_pred = _records_from_json_without_projection(unconstrained_json)
            panel_rows.append(evaluate_ablation_panel(
                "A0_qwen_unconstrained_raw", meta, unconstrained_pred, ground_truth,
                raw_json=unconstrained_json, apply_validation_gate=False, inference_source=source
            ))
    panel_results = pd.DataFrame(panel_rows)
    summary = aggregate_ablation_results(panel_results)
    panel_results.to_csv(REVIEWER3_ABLATION_DIR / "ablation_panel_results.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(REVIEWER3_ABLATION_DIR / "ablation_summary.csv", index=False, encoding="utf-8-sig")
    if first_raw is not None:
        fault_results = run_structural_fault_injection(first_raw, *first_shape)
        fault_results.to_csv(REVIEWER3_ABLATION_DIR / "structural_fault_injection.csv", index=False, encoding="utf-8-sig")
    protocol = {
        "variants": {
            "A0_qwen_unconstrained_raw": "Qwen infers shape; no OpenCV shape is supplied in the prompt.",
            "A1_fixed_shape_prompt_raw": "OpenCV shape is supplied; raw Qwen records are evaluated without repair.",
            "A2_fixed_shape_plus_validation_gate": "A1 plus structural validation; invalid panels are routed to manual review.",
            "A3_full_structure_constrained": "Fixed shape, validation, existing projection/normalization, and manual-review metadata.",
        },
        "ground_truth_is_evaluation_only": True,
        "external_baseline_comparison": False,
        "unconstrained_qwen_calls_enabled": bool(allow_unconstrained_qwen_calls),
        "uses_numeric_confidence_threshold": False,
    }
    (REVIEWER3_ABLATION_DIR / "ablation_protocol.json").write_text(
        json.dumps(protocol, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    display(summary)
    return {"panel_results": panel_results, "summary": summary, "protocol": protocol}


# Safe default for Run All: reuse existing caches and do not spend API calls.
# Set to True once to obtain the four-panel unconstrained Qwen ablation; outputs are cached.
RUN_REVIEWER3_UNCONSTRAINED_QWEN_ABLATION = True
reviewer3_ablation_result = run_reviewer3_internal_ablation(
    allow_unconstrained_qwen_calls=RUN_REVIEWER3_UNCONSTRAINED_QWEN_ABLATION
)


,variant,panels,ground_truth_cells,covered_cells,coverage,exact_matches,exact_accuracy_on_covered,end_to_end_exact_accuracy,schema_valid_panels,manual_review_panels,duplicate_index_count,out_of_bounds_count
0,A0_qwen_unconstrained_raw,4,311,307,0.987138,282,0.918567,0.906752,0,4,0,13
1,A1_fixed_shape_prompt_raw,4,311,311,1.000000,310,0.996785,0.996785,4,0,0,0
2,A2_fixed_shape_plus_validation_gate,4,311,311,1.000000,310,0.996785,0.996785,4,0,0,0
3,A3_full_structure_constrained,4,311,311,1.000000,310,0.996785,0.996785,4,0,0,0


In [ ]:
# ============================================================
# STAGE 5 FINAL UNIVERSAL CODE
# Tool-only record-block extraction + separate Vaccine extraction
# + generic missing-Gene repair
#
# Required existing variable:
# - table_page_images
#
# Expected table_page_images example:
# [
#   {"page_number": 17, "image_path": ".../table_page_17.png"},
#   {"page_number": 18, "image_path": ".../table_page_18.png"},
#   {"page_number": 19, "image_path": ".../table_page_19.png"}
# ]
#
# User setting:
# - SUPP_PDF_PATH
#
# Output:
# ./stage5_final_universal_tool_table_extraction/
# ============================================================

import re
import json
from pathlib import Path

try:
    import fitz
except ImportError:
    fitz = None
import pandas as pd
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


# ============================================================
# 1. SETTINGS
# ============================================================

SUPP_PDF_PATH = str(SUPPLEMENTARY_SOURCE_PATH)

OUT_ROOT = RUN_ROOT / "stage5_reference"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

ROW_BLOCK_OUT = OUT_ROOT / "record_block_crops"
ROW_BLOCK_OUT.mkdir(parents=True, exist_ok=True)

GENERIC_REPAIR_OUT = OUT_ROOT / "generic_gene_repair"
GENERIC_REPAIR_OUT.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT_ROOT.resolve())

SUPP_SOURCE_PATH = Path(SUPP_PDF_PATH)


def _stage5_norm(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def _stage5_find_col(df, aliases):
    norm = {_stage5_norm(c): c for c in df.columns}
    for alias in aliases:
        a = _stage5_norm(alias)
        if a in norm:
            return norm[a]
    for alias in aliases:
        a = _stage5_norm(alias)
        for n, original in norm.items():
            if a and (a in n or n in a):
                return original
    return None


def _stage5_read_sheet_with_header(path, sheet_name):
    preview = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=25)
    aliases = {"peptide", "epitope", "sequence", "gene", "mutation", "proteinchange", "hla", "pool"}
    best_row, best_score = 0, -1
    for i, row in preview.iterrows():
        vals = {_stage5_norm(v) for v in row.dropna().tolist()}
        score = sum(any(a in v for a in aliases) for v in vals)
        if score > best_score:
            best_row, best_score = i, score
    return pd.read_excel(path, sheet_name=sheet_name, header=best_row)


def extract_stage5_tabular_reference(path, out_root):
    path, out_root = Path(path), Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".csv":
        sheets = {"csv": pd.read_csv(path)}
    else:
        xls = pd.ExcelFile(path)
        sheets = {name: _stage5_read_sheet_with_header(path, name) for name in xls.sheet_names}

    frames = []
    for sheet_name, df in sheets.items():
        if df is None or df.empty:
            continue
        score_cols = " ".join(map(str, df.columns)).lower()
        if not any(k in score_cols for k in ["peptide", "epitope", "neoantigen", "sequence", "gene"]):
            continue
        out = df.copy()
        out["_Source_Sheet"] = sheet_name
        frames.append(out)
    if not frames:
        raise ValueError(f"No epitope/peptide table found in {path}")
    raw = pd.concat(frames, ignore_index=True, sort=False)

    aliases = {
        "Epitope_ID": ["epitope id", "peptide id", "peptide identification", "neoantigen id", "id"],
        "Gene": ["gene", "gene symbol", "corresponding gene"],
        "Mutation_or_Substitution": ["protein change", "mutation", "substitution", "amino acid change"],
        "Sequence": ["peptide sequence", "sequence", "neoantigen sequence", "long peptide"],
        "MHC_or_HLA": ["binding allele", "hla allele", "hla", "mhc"],
        "MHC_type": ["mhc type", "mhc class", "hla class"],
        "Vaccine_or_Antigen_Set": ["assigned peptide pool", "peptide pool", "pool", "vaccine group", "antigen set"],
        "Patient_ID": ["patient id", "patient", "subject", "participant"],
        "Tumor_Model": ["tumor model", "cancer type", "indication"],
    }
    final = pd.DataFrame(index=raw.index)
    for target, names in aliases.items():
        c = _stage5_find_col(raw, names)
        final[target] = raw[c].map(clean_text) if c else "Not specified"

    if final["Epitope_ID"].eq("Not specified").all():
        final["Epitope_ID"] = [f"ROW_{i+1}" for i in range(len(final))]
        generated_id = True
    else:
        generated_id = False
    missing_mhc = final["MHC_type"].eq("Not specified")
    final.loc[missing_mhc & final["MHC_or_HLA"].str.contains("HLA", case=False, na=False), "MHC_type"] = "MHC-I/II not resolved"
    final["Source_Document"] = str(path)
    final["Source_Table"] = raw["_Source_Sheet"].map(clean_text)
    final["Source_Page"] = "Not applicable"
    final["Extraction_Method"] = "direct_tabular_import"
    final["Manual_Review"] = generated_id
    final["Review_Reason"] = "Generated row ID because no explicit peptide/epitope ID column was found" if generated_id else ""
    final["Stage6_Merge_Key"] = final["Epitope_ID"]

    # Preserve every original field with a Raw_ prefix for auditability.
    for c in raw.columns:
        final[f"Raw_{c}"] = raw[c]

    final_csv = out_root / "Epitope_Reference_Table_FINAL.csv"
    final_xlsx = out_root / "Epitope_Reference_Table_FINAL.xlsx"
    final.to_csv(final_csv, index=False, encoding="utf-8-sig")
    final.to_excel(final_xlsx, index=False)
    qc = {
        "source_document": str(path),
        "input_sheets": list(sheets),
        "final_reference_row_count": len(final),
        "generated_epitope_ids": generated_id,
        "extraction_method": "direct_tabular_import",
        "outputs": {"final_csv": str(final_csv), "final_xlsx": str(final_xlsx)},
    }
    with open(out_root / "stage5_final_qc.json", "w", encoding="utf-8") as f:
        json.dump(qc, f, ensure_ascii=False, indent=2)
    print("Stage 5 tabular rows:", len(final))
    print("Saved:", final_xlsx)
    return final, final_csv, final_xlsx, qc


if SUPP_SOURCE_PATH.suffix.lower() in {".csv", ".xlsx", ".xls"}:
    final_reference_df, final_csv, final_xlsx, stage5_qc = extract_stage5_tabular_reference(
        SUPP_SOURCE_PATH, OUT_ROOT
    )
    qc_path = OUT_ROOT / "stage5_final_qc.json"
    stage5_result = {
        "reference_df": final_reference_df,
        "final_csv": final_csv,
        "final_xlsx": final_xlsx,
        "qc": stage5_qc,
        "qc_path": qc_path,
    }
else:
    if fitz is None:
        raise ImportError("PyMuPDF is required for PDF supplementary sources. Run: %pip install pymupdf")
    if "table_page_images" not in globals():
        configured_pages = PAPER_CONFIG.get("supplementary_table_pages") or []
        if not configured_pages:
            raise ValueError(
                "For a supplementary PDF, set PAPER_CONFIG['supplementary_table_pages'] "
                "to the 1-based page numbers containing the epitope table."
            )
        table_page_dir = OUT_ROOT / "table_pages"
        table_page_dir.mkdir(parents=True, exist_ok=True)
        table_page_images = []
        supp_doc = fitz.open(str(SUPP_SOURCE_PATH))
        for page_number in configured_pages:
            page_index = int(page_number) - 1
            if page_index < 0 or page_index >= len(supp_doc):
                raise IndexError(f"Supplementary page out of range: {page_number}")
            page = supp_doc[page_index]
            pix = page.get_pixmap(matrix=fitz.Matrix(2.5, 2.5), alpha=False)
            image_path = table_page_dir / f"table_page_{page_number}.png"
            pix.save(str(image_path))
            table_page_images.append({
                "page_number": int(page_number),
                "image_path": str(image_path),
            })
        supp_doc.close()
    print("Input table page images:")
    for x in table_page_images:
        print(x)
    # ============================================================
    # 2. REGEX + HELPERS
    # ============================================================

    AA_LETTERS = "ACDEFGHIKLMNPQRSTVWY"

    AA_SEQUENCE_RE = re.compile(
        rf"\b[{AA_LETTERS}]{{8,80}}\b"
    )

    AA_SEQUENCE_FULL_RE = re.compile(
        rf"^[{AA_LETTERS}]{{8,80}}$"
    )

    MUTATION_RE = re.compile(
        r"\b[A-Z]\d+[A-Z]\b|"                         # D241N, V600E
        r"\b\d+\.[A-Z]/[A-ZX]{1,3}\b|"                # 839.H/X, 134.T/NX
        r"\b\d+-\d+\.-/[A-Z]\b|"                      # 1991-1992.-/I
        r"\bp\.[A-Za-z]{1,3}\d+[A-Za-z]{1,3}\b|"      # p.G12D
        r"\bc\.\d+[ACGT]>[ACGT]\b|"                   # c.35G>A
        r"\bchr[\w]+:\d+[ACGT]>[ACGT]\b|"
        r"\bfs\b|\bdel\b|\bins\b",
        flags=re.I
    )

    MHC_HLA_RE = re.compile(
        r"\bMHC[-\s]?(I|II|1|2)\b|"
        r"\bclass\s+(I|II)\b|"
        r"\bHLA[-\s]?[A-Z]\*?\d{2}:?\d{2}\b|"
        r"\bHLA[-\s]?[A-Z0-9]+\b|"
        r"\bH[-\s]?2K[bk]?\b|"
        r"\bH[-\s]?2D[bk]?\b|"
        r"\bH2[-\s]?K[bk]?\b|"
        r"\bH2[-\s]?D[bk]?\b|"
        r"\ballele\b|"
        r"\bbinding\s+allele\b",
        flags=re.I
    )

    GENERIC_CONTEXT_LABEL_RE = re.compile(
        r"^("
        r"[A-Za-z]{2,}[-_][A-Za-z0-9]+|"
        r"[A-Za-z0-9]+[-_]?vaccine|"
        r"vaccine[-\s]?[A-Za-z0-9]+|"
        r"patient\s*\d+|"
        r"subject\s*\d+|"
        r"pool\s*\d+|"
        r"antigen\s*set\s*\d+|"
        r"group\s*[A-Za-z0-9]+|"
        r"condition\s*[A-Za-z0-9]+|"
        r"treatment\s*[A-Za-z0-9]+"
        r")$",
        flags=re.I
    )

    HEADER_LIKE_TERMS = {
        "gene", "substitution", "mutation", "sequence",
        "mhc", "mhc_type", "mhc-type", "hla", "hla_type",
        "vaccine", "epitope", "neoepitope", "neoantigen",
        "peptide", "patient", "sample", "model",
        "condition", "treatment", "group"
    }


    def clean_line(x):
        x = str(x)
        x = x.replace("\u00ad", "")
        x = x.replace("\ufeff", "")
        x = x.replace("￾", "")
        x = x.replace("−", "-")
        x = re.sub(r"\s+", " ", x).strip()
        return x


    def normalize_mhc_type(x):
        x = clean_line(x)
        low = x.lower()

        if not x or low in ["not specified", "nan", "none"]:
            return "Not specified"

        if re.search(r"\bmhc[-\s]?ii\b", low) or re.search(r"\bclass\s+ii\b", low):
            return "MHC-II"

        if re.search(r"\bmhc[-\s]?i\b", low) or re.search(r"\bclass\s+i\b", low):
            return "MHC-I"

        if "hla" in low:
            return "HLA"

        return x


    def is_valid_context_label(text):
        text = clean_line(text)
        low = text.lower()
        low_norm = low.replace(" ", "_").replace("-", "_")

        if not text:
            return False

        if low in HEADER_LIKE_TERMS or low_norm in HEADER_LIKE_TERMS:
            return False

        if AA_SEQUENCE_FULL_RE.fullmatch(text):
            return False

        if MUTATION_RE.search(text):
            return False

        if MHC_HLA_RE.search(text):
            return False

        if re.fullmatch(r"\d+", text):
            return False

        if len(text.split()) > 6:
            return False

        return bool(GENERIC_CONTEXT_LABEL_RE.fullmatch(text))


    def is_plausible_gene_token(x):
        x = clean_line(x)

        if not x:
            return False

        low = x.lower()

        if low in [
            "gene", "substitution", "mutation", "sequence",
            "mhc", "mhc-i", "mhc-ii", "vaccine", "not", "specified"
        ]:
            return False

        if re.fullmatch(r"\d+", x):
            return False

        if re.fullmatch(r"MHC[-\s]?(I|II|1|2)", x, flags=re.I):
            return False

        if re.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]{8,80}", x):
            return False

        return bool(re.fullmatch(r"[A-Za-z0-9_.-]{2,30}", x))


    # ============================================================
    # 3. RECONSTRUCT PDF VISUAL LINES
    # ============================================================

    def reconstruct_pdf_visual_lines(pdf_path, page_number, y_tol=3):
        """
        Reconstruct visual text lines from PDF words.
        Page number is 1-based.
        """
        doc = fitz.open(str(pdf_path))
        page = doc[page_number - 1]
        words = page.get_text("words")
        doc.close()

        items = []

        for w in words:
            x0, y0, x1, y1, text, *_ = w
            text = clean_line(text)

            if not text:
                continue

            items.append({
                "x0": float(x0),
                "y0": float(y0),
                "x1": float(x1),
                "y1": float(y1),
                "y_mid": float((y0 + y1) / 2),
                "text": text
            })

        items = sorted(items, key=lambda r: (r["y_mid"], r["x0"]))

        lines = []

        for item in items:
            placed = False

            for line in lines:
                if abs(line["y_mid"] - item["y_mid"]) <= y_tol:
                    line["items"].append(item)
                    line["y_mid"] = sum(x["y_mid"] for x in line["items"]) / len(line["items"])
                    placed = True
                    break

            if not placed:
                lines.append({
                    "y_mid": item["y_mid"],
                    "items": [item]
                })

        records = []

        for line in lines:
            line_items = sorted(line["items"], key=lambda x: x["x0"])
            line_text = clean_line(" ".join(x["text"] for x in line_items))

            records.append({
                "line_text": line_text,
                "x0_min": min(x["x0"] for x in line_items),
                "x1_max": max(x["x1"] for x in line_items),
                "y0_min": min(x["y0"] for x in line_items),
                "y1_max": max(x["y1"] for x in line_items),
                "y_mid": float(line["y_mid"]),
                "word_count": len(line_items)
            })

        return pd.DataFrame(records)


    # ============================================================
    # 4. BUILD RECORD BLOCKS
    # ============================================================

    def is_record_start(line_text):
        return bool(re.match(r"^\d+\b", clean_line(line_text)))


    def is_probable_reference_record_block(text):
        text = clean_line(text)

        if not re.match(r"^\d+\b", text):
            return False

        evidence = 0

        if AA_SEQUENCE_RE.search(text):
            evidence += 1

        if MUTATION_RE.search(text):
            evidence += 1

        if MHC_HLA_RE.search(text):
            evidence += 1

        return evidence >= 2


    def build_record_blocks_from_lines(lines_df):
        """
        Group visual lines into record blocks:
        from current numeric-start line to before next numeric-start line.
        """
        lines_df = lines_df.sort_values("y_mid").reset_index(drop=True)

        start_idx = []
        for i, row in lines_df.iterrows():
            if is_record_start(row["line_text"]):
                start_idx.append(i)

        blocks = []

        for k, s in enumerate(start_idx):
            e = start_idx[k + 1] - 1 if k + 1 < len(start_idx) else len(lines_df) - 1

            block_df = lines_df.iloc[s:e + 1].copy()
            full_text = clean_line(" ".join(block_df["line_text"].tolist()))

            if not is_probable_reference_record_block(full_text):
                continue

            blocks.append({
                "start_line_idx": s,
                "end_line_idx": e,
                "line_count_in_block": len(block_df),
                "block_text_from_pdf_words": full_text,
                "pdf_y0": float(block_df["y0_min"].min()),
                "pdf_y1": float(block_df["y1_max"].max()),
                "pdf_x0": float(block_df["x0_min"].min()),
                "pdf_x1": float(block_df["x1_max"].max())
            })

        return pd.DataFrame(blocks)


    def generate_record_block_crops(pdf_path, table_page_images, out_dir=ROW_BLOCK_OUT, y_tol=3, pad_y=10):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        page_image_map = {
            int(item["page_number"]): Path(item["image_path"])
            for item in table_page_images
        }

        doc = fitz.open(str(pdf_path))
        all_blocks = []

        for page_no, rendered_img_path in sorted(page_image_map.items()):
            if not rendered_img_path.exists():
                raise FileNotFoundError(f"Rendered page image not found: {rendered_img_path}")

            page = doc[page_no - 1]
            pdf_h = float(page.rect.height)

            rendered_img = Image.open(rendered_img_path).convert("RGB")
            img_w, img_h = rendered_img.size
            scale_y = img_h / pdf_h

            lines_df = reconstruct_pdf_visual_lines(pdf_path, page_no, y_tol=y_tol)
            block_df = build_record_blocks_from_lines(lines_df)

            lines_df.to_csv(
                out_dir / f"page_{page_no}_visual_lines.csv",
                index=False,
                encoding="utf-8-sig"
            )

            print(f"\nPage {page_no}")
            print("visual lines:", len(lines_df))
            print("record blocks:", len(block_df))

            for row_idx, r in block_df.iterrows():
                top = max(0, int(r["pdf_y0"] * scale_y) - pad_y)
                bottom = min(img_h, int(r["pdf_y1"] * scale_y) + pad_y)

                crop = rendered_img.crop((0, top, img_w, bottom))
                row_path = out_dir / f"page_{page_no}_block_{row_idx:03d}.png"
                crop.save(row_path)

                all_blocks.append({
                    "page_number": page_no,
                    "row_index_on_page": int(row_idx),
                    "image_path": str(row_path),
                    "source_page_image": str(rendered_img_path),
                    "block_text_from_pdf_words": r["block_text_from_pdf_words"],
                    "pdf_y0": r["pdf_y0"],
                    "pdf_y1": r["pdf_y1"],
                    "crop_top_px": top,
                    "crop_bottom_px": bottom,
                    "crop_height_px": bottom - top,
                    "line_count_in_block": int(r["line_count_in_block"])
                })

        doc.close()

        out_df = pd.DataFrame(all_blocks)

        out_df.to_csv(out_dir / "row_block_manifest.csv", index=False, encoding="utf-8-sig")
        out_df.to_excel(out_dir / "row_block_manifest.xlsx", index=False)

        print("\nTotal record blocks:", len(out_df))

        return out_df


    row_block_df = generate_record_block_crops(
        pdf_path=SUPP_PDF_PATH,
        table_page_images=table_page_images,
        out_dir=ROW_BLOCK_OUT,
        y_tol=3,
        pad_y=10
    )

    display(row_block_df.head(20))


    # ============================================================
    # 5. PARSE RECORD BLOCKS
    # ============================================================

    def parse_reference_record_block_text(block_text):
        text = clean_line(block_text)

        record = {
            "Epitope_ID": "Not specified",
            "Gene": "Not specified",
            "Mutation_or_Substitution": "Not specified",
            "Sequence": "Not specified",
            "MHC_or_HLA": "Not specified",
            "MHC_type": "Not specified",
            "Patient_ID": "Not specified",
            "Tumor_Model": "Not specified",
            "raw_row_text": text,
            "parse_status": "failed",
            "Manual_Review": True,
            "Review_Reason": ""
        }

        id_match = re.match(r"^\s*(\d+)\b", text)
        seq_match = AA_SEQUENCE_RE.search(text)
        mut_match = MUTATION_RE.search(text)
        mhc_match = MHC_HLA_RE.search(text)

        if id_match:
            record["Epitope_ID"] = id_match.group(1)
        else:
            record["Review_Reason"] += " Epitope_ID not detected."

        if seq_match:
            record["Sequence"] = seq_match.group(0)
        else:
            record["Review_Reason"] += " Sequence not detected."

        if mut_match:
            record["Mutation_or_Substitution"] = mut_match.group(0)
        else:
            record["Review_Reason"] += " Mutation/substitution not detected."

        if mhc_match:
            record["MHC_or_HLA"] = mhc_match.group(0)
            record["MHC_type"] = normalize_mhc_type(mhc_match.group(0))
        else:
            record["Review_Reason"] += " MHC/HLA not detected."

        if id_match and mut_match:
            between_id_and_mut = text[id_match.end(): mut_match.start()]
            gene = clean_line(between_id_and_mut)

            gene = re.sub(
                r"\b(Gene|Substitution|Mutation|Sequence|MHC|HLA|Vaccine|Epitope|Neoantigen|Peptide)\b",
                "",
                gene,
                flags=re.I
            )
            gene = clean_line(gene)

            record["Gene"] = gene if gene else "Not specified"

        if record["Gene"] == "Not specified":
            record["Review_Reason"] += " Gene not detected."

        required_ok = (
            record["Epitope_ID"] != "Not specified"
            and record["Gene"] != "Not specified"
            and record["Mutation_or_Substitution"] != "Not specified"
            and record["Sequence"] != "Not specified"
            and record["MHC_type"] != "Not specified"
        )

        if required_ok:
            record["parse_status"] = "parsed"
            record["Manual_Review"] = False
            record["Review_Reason"] = ""

        record["Review_Reason"] = clean_line(record["Review_Reason"])

        return record


    def parse_all_record_blocks_tool_only(row_block_df):
        rows = []

        df = row_block_df.copy()
        df = df.sort_values(["page_number", "row_index_on_page"]).reset_index(drop=True)

        for _, r in df.iterrows():
            parsed = parse_reference_record_block_text(r["block_text_from_pdf_words"])

            parsed.update({
                "Source_Page": int(r["page_number"]),
                "Row_Index_On_Page": int(r["row_index_on_page"]),
                "Source_Image_Path": str(r["image_path"]),
                "Source_Page_Image": str(r["source_page_image"]),
                "block_text_from_pdf_words": r["block_text_from_pdf_words"],
                "line_count_in_block": int(r["line_count_in_block"]),
                "Extraction_Method": "tool_pdf_words_record_block_parse"
            })

            rows.append(parsed)

        return pd.DataFrame(rows)


    tool_block_reference_df = parse_all_record_blocks_tool_only(row_block_df)

    tool_block_reference_df.to_csv(
        OUT_ROOT / "tool_record_block_reference_before_vaccine.csv",
        index=False,
        encoding="utf-8-sig"
    )

    tool_block_reference_df.to_excel(
        OUT_ROOT / "tool_record_block_reference_before_vaccine.xlsx",
        index=False
    )

    display(tool_block_reference_df.head(30))
    print("Parsed rows:", len(tool_block_reference_df))
    print(tool_block_reference_df["parse_status"].value_counts(dropna=False))


    # ============================================================
    # 6. SEPARATE VACCINE / GROUP LABEL EXTRACTION
    # ============================================================

    def find_context_header_positions(pdf_path, page_numbers):
        context_header_terms = [
            "vaccine", "construct", "pool", "patient", "subject",
            "sample", "tumor", "model", "cell", "line", "antigen", "set",
            "group", "condition", "treatment"
        ]

        doc = fitz.open(str(pdf_path))
        hits = []

        for page_no in page_numbers:
            page = doc[page_no - 1]
            words = page.get_text("words")

            for w in words:
                x0, y0, x1, y1, text, *_ = w
                token = clean_line(text).lower()

                for term in context_header_terms:
                    if token == term:
                        hits.append({
                            "page_number": int(page_no),
                            "x0": float(x0),
                            "x1": float(x1),
                            "y0": float(y0),
                            "y1": float(y1),
                            "y_mid": float((y0 + y1) / 2),
                            "text": text,
                            "matched_term": term
                        })

        doc.close()

        header_df = pd.DataFrame(hits)

        if header_df.empty:
            return None, header_df

        selected = header_df.sort_values(
            ["page_number", "x0"],
            ascending=[True, False]
        ).iloc[0].to_dict()

        return selected, header_df


    def extract_vaccine_labels_separately(pdf_path, table_page_images):
        page_numbers = sorted({int(x["page_number"]) for x in table_page_images})

        context_header, context_header_df = find_context_header_positions(
            pdf_path=pdf_path,
            page_numbers=page_numbers
        )

        doc = fitz.open(str(pdf_path))
        labels = []

        if context_header is not None:
            header_x0 = float(context_header["x0"])
            x_min = header_x0 - 180
            x_max = header_x0 + 300
        else:
            x_min = None
            x_max = None

        for page_no in page_numbers:
            page = doc[page_no - 1]
            words = page.get_text("words")

            for w in words:
                x0, y0, x1, y1, text, *_ = w
                text = clean_line(text)

                if not text:
                    continue

                if x_min is not None:
                    if not (x_min <= float(x0) <= x_max):
                        continue

                if is_valid_context_label(text):
                    labels.append({
                        "label": text,
                        "page_number": int(page_no),
                        "x0": float(x0),
                        "y0": float(y0),
                        "x1": float(x1),
                        "y1": float(y1),
                        "y_mid": float((y0 + y1) / 2)
                    })

        doc.close()

        labels_df = pd.DataFrame(labels)

        if labels_df.empty:
            labels_df = pd.DataFrame(
                columns=["label", "page_number", "x0", "y0", "x1", "y1", "y_mid"]
            )
        else:
            labels_df = (
                labels_df
                .drop_duplicates(subset=["label", "page_number", "x0", "y0"])
                .sort_values(["page_number", "y_mid", "x0"])
                .reset_index(drop=True)
            )

        context_header_df.to_csv(
            OUT_ROOT / "context_header_candidates.csv",
            index=False,
            encoding="utf-8-sig"
        )

        labels_df.to_csv(
            OUT_ROOT / "separate_vaccine_labels_detected.csv",
            index=False,
            encoding="utf-8-sig"
        )

        labels_df.to_excel(
            OUT_ROOT / "separate_vaccine_labels_detected.xlsx",
            index=False
        )

        print("Context header selected:")
        print(context_header)
        print("Separate vaccine/group labels detected:", len(labels_df))
        display(labels_df)

        return labels_df, context_header_df, context_header


    separate_vaccine_labels_df, context_header_df, context_header = extract_vaccine_labels_separately(
        pdf_path=SUPP_PDF_PATH,
        table_page_images=table_page_images
    )


    # ============================================================
    # 7. GROUP BY EPITOPE_ID RESET + ASSIGN VACCINE
    # ============================================================

    def add_group_ids_by_epitope_reset(reference_df):
        df = reference_df.copy()
        df = df.sort_values(["Source_Page", "Row_Index_On_Page"]).reset_index(drop=True)

        group_ids = []
        current_group = 0
        prev_id = None

        for eid in df["Epitope_ID"].astype(str):
            m = re.search(r"\d+", eid)
            eid_int = int(m.group(0)) if m else None

            if prev_id is not None and eid_int is not None:
                if eid_int <= prev_id:
                    current_group += 1

            group_ids.append(current_group)

            if eid_int is not None:
                prev_id = eid_int

        df["_group_id"] = group_ids

        return df


    def assign_vaccine_labels_to_groups(reference_df, labels_df):
        df = reference_df.copy()

        group_ids_sorted = sorted(df["_group_id"].unique())

        if labels_df is None or labels_df.empty:
            label_list = []
        else:
            label_list = (
                labels_df
                .sort_values(["page_number", "y_mid", "x0"])
                ["label"]
                .drop_duplicates()
                .tolist()
            )

        df["Vaccine_or_Antigen_Set"] = "Not specified"
        df["Vaccine_Assignment_Method"] = "not_assigned"
        df["Vaccine_Assignment_Confidence"] = "low"

        df["Review_Reason"] = df["Review_Reason"].fillna("").astype(str)

        if len(label_list) == len(group_ids_sorted):
            for gid, label in zip(group_ids_sorted, label_list):
                mask = df["_group_id"].eq(gid)
                df.loc[mask, "Vaccine_or_Antigen_Set"] = label
                df.loc[mask, "Vaccine_Assignment_Method"] = "separate_vaccine_label_by_group_order"
                df.loc[mask, "Vaccine_Assignment_Confidence"] = "medium"
        else:
            df["Manual_Review"] = True
            df["Review_Reason"] = (
                df["Review_Reason"]
                + f" Vaccine label count ({len(label_list)}) does not match row group count ({len(group_ids_sorted)}); not assigned."
            ).str.strip()

        missing = df["Vaccine_or_Antigen_Set"].eq("Not specified")

        df.loc[missing, "Manual_Review"] = True
        df.loc[missing, "Review_Reason"] = (
            df.loc[missing, "Review_Reason"].fillna("").astype(str)
            + " Vaccine/group label not confidently assigned."
        ).str.strip()

        df["Stage6_Merge_Key"] = (
            df["Vaccine_or_Antigen_Set"].astype(str).str.strip()
            + "__Epitope_"
            + df["Epitope_ID"].astype(str).str.strip()
        )

        return df, label_list


    grouped_reference_df = add_group_ids_by_epitope_reset(tool_block_reference_df)

    final_reference_df, vaccine_label_list = assign_vaccine_labels_to_groups(
        reference_df=grouped_reference_df,
        labels_df=separate_vaccine_labels_df
    )

    group_debug_before_repair = (
        final_reference_df
        .groupby("_group_id")
        .agg(
            row_count=("Epitope_ID", "count"),
            first_epitope=("Epitope_ID", "first"),
            last_epitope=("Epitope_ID", "last"),
            assigned_vaccine=("Vaccine_or_Antigen_Set", "first"),
            first_page=("Source_Page", "first"),
            last_page=("Source_Page", "last")
        )
        .reset_index()
    )

    group_debug_before_repair.to_csv(
        OUT_ROOT / "group_debug_before_generic_repair.csv",
        index=False,
        encoding="utf-8-sig"
    )

    display(group_debug_before_repair)


    # ============================================================
    # 8. GENERIC MISSING GENE REPAIR
    # No hard-coded gene values.
    # ============================================================

    def get_pdf_words_page(pdf_path, page_number):
        doc = fitz.open(str(pdf_path))
        page = doc[page_number - 1]
        words = page.get_text("words")
        doc.close()

        rows = []

        for w in words:
            x0, y0, x1, y1, text, *_ = w
            text = clean_line(text)

            if not text:
                continue

            rows.append({
                "x0": float(x0),
                "y0": float(y0),
                "x1": float(x1),
                "y1": float(y1),
                "y_mid": float((y0 + y1) / 2),
                "text": text
            })

        return pd.DataFrame(rows)


    def build_row_meta_lookup(row_block_df):
        meta = row_block_df.copy()
        meta["_Source_Page_num"] = meta["page_number"].astype(int)
        meta["_Row_Index_num"] = meta["row_index_on_page"].astype(int)
        return meta


    def infer_gene_column_x_range(reference_df, row_block_df, pdf_path, buffer=25):
        meta = build_row_meta_lookup(row_block_df)

        df = reference_df.copy()

        df["_Source_Page_num"] = (
            df["Source_Page"].astype(str).str.extract(r"(\d+)")[0].astype(int)
        )
        df["_Row_Index_num"] = (
            df["Row_Index_On_Page"].astype(str).str.extract(r"(\d+)")[0].astype(int)
        )

        df = df.merge(
            meta[[
                "_Source_Page_num",
                "_Row_Index_num",
                "pdf_y0",
                "pdf_y1"
            ]],
            on=["_Source_Page_num", "_Row_Index_num"],
            how="left"
        )

        gene_hits = []

        good_rows = df[
            (~df["Gene"].astype(str).str.lower().isin(["not specified", "nan", "none", ""])) &
            (df["Manual_Review"].astype(str).str.lower().isin(["false", "0", "no"]))
        ].copy()

        page_cache = {}

        for _, r in good_rows.iterrows():
            page_no = int(r["_Source_Page_num"])
            gene = clean_line(r["Gene"])

            if not gene:
                continue

            if page_no not in page_cache:
                page_cache[page_no] = get_pdf_words_page(pdf_path, page_no)

            words_df = page_cache[page_no]

            if pd.isna(r["pdf_y0"]) or pd.isna(r["pdf_y1"]):
                continue

            y0 = float(r["pdf_y0"]) - 4
            y1 = float(r["pdf_y1"]) + 4

            local_words = words_df[
                (words_df["y_mid"] >= y0) &
                (words_df["y_mid"] <= y1)
            ].copy()

            matched = local_words[local_words["text"].astype(str).eq(gene)].copy()

            if not matched.empty:
                for _, m in matched.iterrows():
                    gene_hits.append({
                        "page": page_no,
                        "gene": gene,
                        "x0": float(m["x0"]),
                        "x1": float(m["x1"])
                    })

        gene_hits_df = pd.DataFrame(gene_hits)

        if gene_hits_df.empty:
            raise ValueError(
                "Could not infer Gene column x-range from parsed rows. "
                "Generic missing-Gene repair skipped."
            )

        x0_med = float(gene_hits_df["x0"].median())
        x1_med = float(gene_hits_df["x1"].median())

        x0_q1 = float(gene_hits_df["x0"].quantile(0.10))
        x1_q9 = float(gene_hits_df["x1"].quantile(0.90))

        x_min = min(x0_med, x0_q1) - buffer
        x_max = max(x1_med, x1_q9) + buffer

        gene_hits_df.to_csv(
            GENERIC_REPAIR_OUT / "inferred_gene_column_hits.csv",
            index=False,
            encoding="utf-8-sig"
        )

        gene_hits_df.to_excel(
            GENERIC_REPAIR_OUT / "inferred_gene_column_hits.xlsx",
            index=False
        )

        gene_column_info = {
            "x_min": x_min,
            "x_max": x_max,
            "x0_median": x0_med,
            "x1_median": x1_med,
            "hit_count": int(len(gene_hits_df))
        }

        with open(GENERIC_REPAIR_OUT / "inferred_gene_column_info.json", "w", encoding="utf-8") as f:
            json.dump(gene_column_info, f, ensure_ascii=False, indent=2)

        print("Inferred Gene column:")
        print(gene_column_info)

        return gene_column_info, gene_hits_df


    def generic_repair_missing_gene_from_gene_column(
        reference_df,
        row_block_df,
        pdf_path,
        y_expand=18,
        gene_x_buffer=25
    ):
        """
        Repair missing Gene values from inferred Gene column.

        Safety rules:
        - No hard-coded gene values.
        - Only Gene is repaired.
        - Epitope_ID, mutation, sequence, MHC are not overwritten.
        - If no unique candidate is found, row stays Manual_Review=True.
        """
        df = reference_df.copy()

        meta = build_row_meta_lookup(row_block_df)

        df["_Source_Page_num"] = (
            df["Source_Page"].astype(str).str.extract(r"(\d+)")[0].astype(int)
        )
        df["_Row_Index_num"] = (
            df["Row_Index_On_Page"].astype(str).str.extract(r"(\d+)")[0].astype(int)
        )

        df = df.merge(
            meta[[
                "_Source_Page_num",
                "_Row_Index_num",
                "pdf_y0",
                "pdf_y1"
            ]],
            on=["_Source_Page_num", "_Row_Index_num"],
            how="left"
        )

        try:
            gene_column_info, gene_hits_df = infer_gene_column_x_range(
                reference_df=reference_df,
                row_block_df=row_block_df,
                pdf_path=pdf_path,
                buffer=gene_x_buffer
            )
        except Exception as e:
            print("Generic Gene repair skipped:", e)
            df["Repair_Method"] = "none"
            df["Repair_Notes"] = ""
            return df, pd.DataFrame(), None

        x_min = gene_column_info["x_min"]
        x_max = gene_column_info["x_max"]

        page_cache = {}
        repair_logs = []

        if "Repair_Method" not in df.columns:
            df["Repair_Method"] = "none"

        if "Repair_Notes" not in df.columns:
            df["Repair_Notes"] = ""

        missing_gene_mask = (
            df["Gene"].astype(str).str.lower().isin(["not specified", "nan", "none", ""])
        )

        for idx, r in df[missing_gene_mask].iterrows():
            page_no = int(r["_Source_Page_num"])

            if pd.isna(r["pdf_y0"]) or pd.isna(r["pdf_y1"]):
                repair_logs.append({
                    "row_index": int(idx),
                    "status": "not_repaired",
                    "reason": "Missing pdf_y0/pdf_y1 metadata."
                })
                continue

            if page_no not in page_cache:
                page_cache[page_no] = get_pdf_words_page(pdf_path, page_no)

            words_df = page_cache[page_no]

            y0 = float(r["pdf_y0"]) - y_expand
            y1 = float(r["pdf_y1"]) + y_expand

            local_gene_words = words_df[
                (words_df["y_mid"] >= y0) &
                (words_df["y_mid"] <= y1) &
                (words_df["x0"] >= x_min) &
                (words_df["x0"] <= x_max)
            ].copy()

            local_gene_words = local_gene_words.sort_values(["y_mid", "x0"]).reset_index(drop=True)

            tokens = [
                clean_line(x)
                for x in local_gene_words["text"].tolist()
                if is_plausible_gene_token(x)
            ]

            unique_tokens = []
            for t in tokens:
                if t not in unique_tokens:
                    unique_tokens.append(t)

            candidates = []

            if len(unique_tokens) == 1:
                candidates = unique_tokens

            elif len(unique_tokens) > 1:
                concat_candidate = clean_line("".join(unique_tokens))

                if is_plausible_gene_token(concat_candidate):
                    candidates = [concat_candidate]

            if len(candidates) == 1:
                repaired_gene = candidates[0]

                df.loc[idx, "Gene"] = repaired_gene
                df.loc[idx, "Manual_Review"] = False
                df.loc[idx, "Review_Reason"] = ""
                df.loc[idx, "Repair_Method"] = "generic_gene_column_repair"
                df.loc[idx, "Repair_Notes"] = (
                    "Gene repaired from inferred Gene column using PDF word coordinates. "
                    "No paper-specific value was hard-coded."
                )

                df.loc[idx, "raw_row_text"] = (
                    str(df.loc[idx, "Epitope_ID"]) + " " +
                    str(df.loc[idx, "Gene"]) + " " +
                    str(df.loc[idx, "Mutation_or_Substitution"]) + " " +
                    str(df.loc[idx, "Sequence"]) + " " +
                    str(df.loc[idx, "MHC_type"])
                )

                repair_logs.append({
                    "row_index": int(idx),
                    "status": "repaired",
                    "Source_Page": page_no,
                    "Row_Index_On_Page": int(r["_Row_Index_num"]),
                    "repaired_gene": repaired_gene,
                    "tokens": unique_tokens,
                    "y0": y0,
                    "y1": y1,
                    "x_min": x_min,
                    "x_max": x_max
                })

            else:
                df.loc[idx, "Manual_Review"] = True
                existing_reason = clean_line(df.loc[idx, "Review_Reason"])
                df.loc[idx, "Review_Reason"] = clean_line(
                    existing_reason + " Generic Gene-column repair could not identify a unique gene candidate."
                )

                repair_logs.append({
                    "row_index": int(idx),
                    "status": "not_repaired",
                    "Source_Page": page_no,
                    "Row_Index_On_Page": int(r["_Row_Index_num"]),
                    "tokens": unique_tokens,
                    "candidate_count": len(candidates),
                    "y0": y0,
                    "y1": y1,
                    "x_min": x_min,
                    "x_max": x_max
                })

        df["Stage6_Merge_Key"] = (
            df["Vaccine_or_Antigen_Set"].astype(str).str.strip()
            + "__Epitope_"
            + df["Epitope_ID"].astype(str).str.strip()
        )

        repair_log_df = pd.DataFrame(repair_logs)

        repair_log_df.to_csv(
            GENERIC_REPAIR_OUT / "generic_missing_gene_repair_log.csv",
            index=False,
            encoding="utf-8-sig"
        )

        repair_log_df.to_excel(
            GENERIC_REPAIR_OUT / "generic_missing_gene_repair_log.xlsx",
            index=False
        )

        helper_cols = [
            "_Source_Page_num",
            "_Row_Index_num",
            "pdf_y0",
            "pdf_y1"
        ]

        df_out = df.drop(columns=[c for c in helper_cols if c in df.columns])

        print("Generic Gene repair complete.")
        print("Missing Gene rows before:", int(missing_gene_mask.sum()))
        print(
            "Manual review after:",
            int(df_out["Manual_Review"].astype(str).str.lower().isin(["true", "1", "yes"]).sum())
        )

        display(repair_log_df)

        return df_out, repair_log_df, gene_column_info


    final_reference_df, generic_repair_log_df, gene_column_info = generic_repair_missing_gene_from_gene_column(
        reference_df=final_reference_df,
        row_block_df=row_block_df,
        pdf_path=SUPP_PDF_PATH,
        y_expand=18,
        gene_x_buffer=25
    )


    # ============================================================
    # 9. FINAL FORMAT + SAVE
    # ============================================================

    final_reference_df["Source_Document"] = SUPP_PDF_PATH
    final_reference_df["Source_Table"] = "Tool-parsed epitope reference table from record blocks"

    for col in ["Repair_Method", "Repair_Notes"]:
        if col not in final_reference_df.columns:
            final_reference_df[col] = "none"

    final_cols = [
        "Vaccine_or_Antigen_Set",
        "Epitope_ID",
        "Gene",
        "Mutation_or_Substitution",
        "Sequence",
        "MHC_or_HLA",
        "MHC_type",
        "Patient_ID",
        "Tumor_Model",
        "Source_Document",
        "Source_Table",
        "Source_Page",
        "Row_Index_On_Page",
        "Source_Image_Path",
        "Source_Page_Image",
        "Extraction_Method",
        "Manual_Review",
        "Review_Reason",
        "Repair_Method",
        "Repair_Notes",
        "Vaccine_Assignment_Method",
        "Vaccine_Assignment_Confidence",
        "_group_id",
        "raw_row_text",
        "block_text_from_pdf_words",
        "line_count_in_block",
        "Stage6_Merge_Key"
    ]

    for col in final_cols:
        if col not in final_reference_df.columns:
            final_reference_df[col] = "Not specified"

    final_reference_df = final_reference_df[final_cols].copy()

    final_csv = OUT_ROOT / "Epitope_Reference_Table_FINAL.csv"
    final_xlsx = OUT_ROOT / "Epitope_Reference_Table_FINAL.xlsx"

    final_reference_df.to_csv(final_csv, index=False, encoding="utf-8-sig")
    final_reference_df.to_excel(final_xlsx, index=False)

    group_debug_after_repair = (
        final_reference_df
        .groupby("_group_id")
        .agg(
            row_count=("Epitope_ID", "count"),
            first_epitope=("Epitope_ID", "first"),
            last_epitope=("Epitope_ID", "last"),
            assigned_vaccine=("Vaccine_or_Antigen_Set", "first"),
            first_page=("Source_Page", "first"),
            last_page=("Source_Page", "last")
        )
        .reset_index()
    )

    group_debug_after_repair.to_csv(
        OUT_ROOT / "group_debug_after_generic_repair.csv",
        index=False,
        encoding="utf-8-sig"
    )

    qc = {
        "source_document": SUPP_PDF_PATH,
        "input_record_block_count": int(len(row_block_df)),
        "final_reference_row_count": int(len(final_reference_df)),
        "manual_review_count": int(
            final_reference_df["Manual_Review"]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
            .sum()
        ),
        "row_group_count": int(final_reference_df["_group_id"].nunique()),
        "separate_vaccine_label_count": int(len(separate_vaccine_labels_df)),
        "separate_vaccine_labels": vaccine_label_list,
        "rows_by_vaccine": final_reference_df["Vaccine_or_Antigen_Set"].value_counts(dropna=False).to_dict(),
        "rows_by_mhc_type": final_reference_df["MHC_type"].value_counts(dropna=False).to_dict(),
        "vaccine_assignment_methods": final_reference_df["Vaccine_Assignment_Method"].value_counts(dropna=False).to_dict(),
        "generic_gene_repair": {
            "gene_column_info": gene_column_info,
            "repair_log_rows": generic_repair_log_df.to_dict(orient="records") if generic_repair_log_df is not None and not generic_repair_log_df.empty else []
        },
        "outputs": {
            "final_csv": str(final_csv),
            "final_xlsx": str(final_xlsx),
            "record_block_manifest": str(ROW_BLOCK_OUT / "row_block_manifest.csv"),
            "separate_vaccine_labels_csv": str(OUT_ROOT / "separate_vaccine_labels_detected.csv"),
            "generic_gene_repair_log": str(GENERIC_REPAIR_OUT / "generic_missing_gene_repair_log.csv"),
            "group_debug_before_generic_repair": str(OUT_ROOT / "group_debug_before_generic_repair.csv"),
            "group_debug_after_generic_repair": str(OUT_ROOT / "group_debug_after_generic_repair.csv")
        }
    }

    qc_path = OUT_ROOT / "stage5_final_qc.json"

    with open(qc_path, "w", encoding="utf-8") as f:
        json.dump(qc, f, ensure_ascii=False, indent=2)

    display(final_reference_df.head(40))

    print("\nFinal rows:", len(final_reference_df))
    print("Manual review:", qc["manual_review_count"])

    print("\nRows by Vaccine:")
    print(final_reference_df.groupby("Vaccine_or_Antigen_Set").size())

    print("\nRows by MHC_type:")
    print(final_reference_df.groupby("MHC_type").size())

    print("\nManual review rows:")
    display(final_reference_df[
        final_reference_df["Manual_Review"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    ])

    print("\nSaved final CSV:", final_csv)
    print("Saved final Excel:", final_xlsx)
    print("Saved QC:", qc_path)
    print(json.dumps(qc, ensure_ascii=False, indent=2))

    stage5_result = {
        "reference_df": final_reference_df,
        "final_csv": final_csv,
        "final_xlsx": final_xlsx,
        "qc": qc,
        "qc_path": qc_path,
    }


In [ ]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd


ELISPOT_MAIN_COLUMNS = [
    "Epitope_ID",
    "Epitope_Sequence",
    "Epitope_Gene",
    "Epitope_Mutation",
    "MHC_Class",
    "Vaccine_or_Antigen_Set",
    "X_Axis_Value",
    "Y_Axis_Value",
    "ELISpot_Value",
    "ELISpot_Unit",
    "Readout",
    "Sample_or_Tissue",
    "Treatment",
    "Dose",
    "Route",
    "N_Mice",
    "Epitope_Source_Model",
    "Mouse_Tumor_Model",
    "Mouse_Strain",
    "Mouse_MHC_I_A",
    "Mouse_MHC_I_B",
    "Mouse_MHC_I_C",
    "Article_Source",
    "Data_Source"
]


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def clean_text(x):
    if x is None:
        return "Not specified"
    try:
        if pd.isna(x):
            return "Not specified"
    except Exception:
        pass
    x = str(x)
    x = x.replace("\u00ad", "")
    x = x.replace("\ufeff", "")
    x = x.replace("−", "-")
    x = x.replace("–", "-")
    x = x.replace("—", "-")
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else "Not specified"


def is_not_specified(x):
    return clean_text(x).lower() in ["", "nan", "none", "null", "not specified"]


def first_nonempty(*values):
    for v in values:
        v = clean_text(v)
        if not is_not_specified(v):
            return v
    return "Not specified"


def safe_float(x):
    if x is None:
        return np.nan
    try:
        return float(str(x).replace(",", "").strip())
    except Exception:
        return np.nan


def read_table_robust(path, preferred_sheets=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in [".xlsx", ".xls"]:
        xls = pd.ExcelFile(path)
        sheet_names = xls.sheet_names

        if preferred_sheets:
            lower_map = {s.lower(): s for s in sheet_names}
            for s in preferred_sheets:
                if s.lower() in lower_map:
                    return pd.read_excel(path, sheet_name=lower_map[s.lower()])

        return pd.read_excel(path, sheet_name=sheet_names[0])

    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, list):
            return pd.DataFrame(obj)

        if isinstance(obj, dict):
            for k in ["records", "data", "results", "table"]:
                if isinstance(obj.get(k), list):
                    return pd.DataFrame(obj[k])
            return pd.DataFrame([obj])

    raise ValueError(f"Unsupported file type: {path}")


def read_many_stage4_records(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]

    dfs = []

    for p in paths:
        p = Path(p)

        if not p.exists():
            print("Missing Stage 4 file:", p)
            continue

        df = read_table_robust(
            p,
            preferred_sheets=["Stage4_Cell_Records", "Cell_Records"]
        )

        df["Stage4_Input_File"] = str(p)
        dfs.append(df)

    if not dfs:
        raise FileNotFoundError("No valid Stage 4 cell record files found.")

    return pd.concat(dfs, ignore_index=True)


def save_json(obj, path):
    path = Path(path)
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def normalize_col_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def get_col(df, aliases):
    alias_norm = [normalize_col_name(a) for a in aliases]
    col_map = {normalize_col_name(c): c for c in df.columns}

    for a in alias_norm:
        if a in col_map:
            return col_map[a]

    return None


def normalize_epitope_id(x):
    x = clean_text(x)

    if is_not_specified(x):
        return "Not specified"

    m = re.search(
        r"(?:epitope|neoepitope|neoantigen|peptide|antigen|epi)[\s_\-#:]*(\d+)",
        x,
        flags=re.I
    )

    if m:
        return str(int(m.group(1)))

    m = re.search(r"^#?\s*(\d+)\s*$", x)

    if m:
        return str(int(m.group(1)))

    m = re.search(r"^[Ee][Pp]?[Ii]?[\s_\-]*(\d+)$", x)

    if m:
        return str(int(m.group(1)))

    return x


def standardize_stage4_records(df):
    df = df.copy()

    alias_map = {
        "Stage4_Row_ID": ["Stage4_Row_ID"],
        "Figure_Unit_ID": ["Figure_Unit_ID", "figure_unit_id"],
        "Candidate_ID": ["Candidate_ID", "candidate_id"],
        "Panel_ID": ["Panel_ID", "panel_id"],
        "Source_Document": ["Source_Document", "source_doc"],
        "Source_Section": ["Source_Section", "source_section"],
        "X_Axis_Value": ["X_Axis_Value", "X_Value"],
        "Y_Axis_Value": ["Y_Axis_Value", "Y_Value"],
        "X_Axis_Role": ["X_Axis_Role", "x_axis_role"],
        "Y_Axis_Role": ["Y_Axis_Role", "y_axis_role"],
        "Row_Index": ["Row_Index", "row_index"],
        "Col_Index": ["Col_Index", "col_index"],
        "Epitope_ID": ["Epitope_ID"],
        "ELISpot_Value": ["ELISpot_Value", "Value_Numeric", "value_numeric"],
        "ELISpot_Value_Text": ["ELISpot_Value_Text", "Value_Text", "value_text"],
        "ELISpot_Unit": ["ELISpot_Unit", "Legend_ELISpot_Unit"],
        "Readout": ["Legend_Readout", "Assay_Type", "Readout"],
        "Sample_or_Tissue": ["Legend_Sample_or_Tissue", "Sample_or_Tissue"],
        "Treatment": ["Treatment", "Legend_Treatment"],
        "Legend_Treatment": ["Legend_Treatment"],
        "Legend_Dose": ["Legend_Dose"],
        "Legend_Route": ["Legend_Route"],
        "Legend_N_Mice": ["Legend_N_Mice"],
        "Legend_Mouse_Tumor_Model": ["Legend_Mouse_Tumor_Model"],
        "Legend_Mouse_Strain": ["Legend_Mouse_Strain"],
        "Legend_Vaccine_or_Antigen_Set": ["Legend_Vaccine_or_Antigen_Set"],
        "Legend_ELISpot_Unit": ["Legend_ELISpot_Unit"],
        "Manual_Review": ["Manual_Review"],
        "Review_Reason": ["Review_Reason"],
        "Source_Image_Path": ["Source_Image_Path", "selected_context_path"],
        "Core_Image_Path": ["Core_Image_Path", "core_crop_path"],
        "Data_Source": ["Extraction_Method", "Data_Source"],
        "Stage4_Input_File": ["Stage4_Input_File"]
    }

    out = pd.DataFrame(index=df.index)

    for std_col, aliases in alias_map.items():
        src = get_col(df, aliases)
        if src is None:
            out[std_col] = "Not specified"
        else:
            out[std_col] = df[src]

    for c in out.columns:
        if c not in ["ELISpot_Value", "Row_Index", "Col_Index"]:
            out[c] = out[c].apply(clean_text)

    out["ELISpot_Value"] = pd.to_numeric(out["ELISpot_Value"], errors="coerce")
    out["Row_Index"] = pd.to_numeric(out["Row_Index"], errors="coerce")
    out["Col_Index"] = pd.to_numeric(out["Col_Index"], errors="coerce")

    if out["Stage4_Row_ID"].astype(str).eq("Not specified").all():
        out["Stage4_Row_ID"] = [f"Stage4_Cell_{i + 1}" for i in range(len(out))]

    return out


def guess_reference_id_column(df):
    for c in df.columns:
        cn = normalize_col_name(c)
        if cn in ["epitopeid", "id", "no", "number", "index"]:
            return c

    for c in df.columns:
        s = df[c].dropna().astype(str).str.strip()
        if len(s) == 0:
            continue
        numeric_like = s.str.match(r"^\d+$").mean()
        if numeric_like >= 0.8:
            vals = pd.to_numeric(s[s.str.match(r"^\d+$")], errors="coerce").dropna()
            if len(vals) >= 5:
                return c

    return None


def guess_gene_column(df):
    for c in df.columns:
        cn = normalize_col_name(c)
        if cn == "gene" or "genename" in cn:
            return c
    return None


def guess_sequence_column(df):
    for c in df.columns:
        cn = normalize_col_name(c)
        if "sequence" in cn and "aa" in cn:
            return c

    for c in df.columns:
        cn = normalize_col_name(c)
        if "sequence" in cn or "peptide" in cn:
            return c

    for c in df.columns:
        s = df[c].dropna().astype(str).str.strip()
        if len(s) == 0:
            continue
        aa_like = s.str.match(r"^[ACDEFGHIKLMNPQRSTVWY]{8,40}$", case=False).mean()
        if aa_like >= 0.5:
            return c

    return None


def guess_mutation_column(df):
    for c in df.columns:
        cn = normalize_col_name(c)
        if (
            ("mutation" in cn or "substitution" in cn or "variant" in cn or "aminoacidchange" in cn or "aachange" in cn)
            and "sequence" not in cn
            and "gene" not in cn
            and "mhc" not in cn
            and "vaccine" not in cn
        ):
            return c

    for c in df.columns:
        s = df[c].dropna().astype(str).str.strip()
        if len(s) == 0:
            continue
        mutation_like = s.str.match(r"^[A-Z][0-9]+[A-Z*]$", case=False).mean()
        if mutation_like >= 0.5:
            return c

    return None


def guess_mhc_column(df):
    for c in df.columns:
        cn = normalize_col_name(c)
        if "mhc" in cn or "hla" in cn:
            return c
    return None


def guess_vaccine_column(df):
    for c in df.columns:
        cn = normalize_col_name(c)
        if "vaccine" in cn or "antigenset" in cn or "group" in cn:
            return c
    return None


def fill_if_all_missing(out, out_col, df, guessed_col):
    if guessed_col is None:
        return out

    if out_col not in out.columns:
        out[out_col] = "Not specified"

    if out[out_col].apply(is_not_specified).all():
        out[out_col] = df[guessed_col].apply(clean_text)

    return out


def standardize_reference_table(df):
    df = df.copy()

    alias_map = {
        "Epitope_ID": [
            "Epitope_ID",
            "epitope_id",
            "ID",
            "No",
            "Number",
            "Index",
            "Neoepitope_ID",
            "Neoantigen_ID"
        ],
        "Epitope_Sequence": [
            "Epitope_Sequence",
            "Sequence",
            "Peptide_Sequence",
            "27AA sequence used for immunization",
            "27AA_sequence_used_for_immunization",
            "27AA sequence",
            "27AA_sequence",
            "27AA",
            "27AAsequenceusedforimmunization"
        ],
        "Epitope_Gene": [
            "Epitope_Gene",
            "Gene",
            "gene",
            "Gene_Name",
            "GeneSymbol",
            "Gene_Symbol"
        ],
        "Epitope_Mutation": [
            "Epitope_Mutation",
            "Epitope_Substitution",
            "Mutation",
            "Substitution",
            "Mutation/Substitution",
            "Mutation_Substitution",
            "Substitution/Mutation",
            "Substitution_Mutation",
            "Amino acid substitution",
            "Amino_acid_substitution",
            "AA substitution",
            "AA_substitution",
            "Variant",
            "Amino acid change",
            "Amino_acid_change",
            "AA change",
            "AA_change",
            "mutation",
            "substitution"
        ],
        "MHC_Class": [
            "MHC_Class",
            "MHC",
            "MHC_type",
            "MHC Type",
            "MHC-type",
            "MHC class",
            "MHC_class"
        ],
        "Vaccine_or_Antigen_Set": [
            "Vaccine_or_Antigen_Set",
            "Vaccine",
            "Antigen_Set",
            "Vaccine_Group",
            "Antigen Group",
            "Antigen_Group"
        ],
        "Reference_Manual_Review": [
            "Manual_Review",
            "Reference_Manual_Review"
        ],
        "Reference_Source": [
            "Reference_Source",
            "Source",
            "source_file"
        ]
    }

    out = pd.DataFrame(index=df.index)

    for std_col, aliases in alias_map.items():
        src = get_col(df, aliases)
        if src is None:
            out[std_col] = "Not specified"
        else:
            out[std_col] = df[src]

    for c in out.columns:
        out[c] = out[c].apply(clean_text)

    out = fill_if_all_missing(out, "Epitope_ID", df, guess_reference_id_column(df))
    out = fill_if_all_missing(out, "Epitope_Gene", df, guess_gene_column(df))
    out = fill_if_all_missing(out, "Epitope_Sequence", df, guess_sequence_column(df))
    out = fill_if_all_missing(out, "Epitope_Mutation", df, guess_mutation_column(df))
    out = fill_if_all_missing(out, "MHC_Class", df, guess_mhc_column(df))
    out = fill_if_all_missing(out, "Vaccine_or_Antigen_Set", df, guess_vaccine_column(df))

    out["Epitope_ID_Normalized"] = out["Epitope_ID"].apply(normalize_epitope_id)

    return out


def derive_epitope_id_from_stage4(row):
    direct = normalize_epitope_id(row.get("Epitope_ID", "Not specified"))

    if direct != "Not specified":
        return direct

    x_role = clean_text(row.get("X_Axis_Role", "")).lower()
    y_role = clean_text(row.get("Y_Axis_Role", "")).lower()

    x_val = clean_text(row.get("X_Axis_Value", "Not specified"))
    y_val = clean_text(row.get("Y_Axis_Value", "Not specified"))

    if x_role == "epitope":
        return normalize_epitope_id(x_val)

    if y_role == "epitope":
        return normalize_epitope_id(y_val)

    candidates = []

    for v in [x_val, y_val]:
        if re.search(r"(epitope|neoepitope|neoantigen|peptide|antigen|epi)", v, flags=re.I):
            candidates.append(normalize_epitope_id(v))

    candidates = [c for c in candidates if c != "Not specified"]

    if len(candidates) == 1:
        return candidates[0]

    x_norm = normalize_epitope_id(x_val)
    y_norm = normalize_epitope_id(y_val)

    if x_norm != "Not specified" and y_norm == "Not specified":
        return x_norm

    if y_norm != "Not specified" and x_norm == "Not specified":
        return y_norm

    return "Not specified"


def label_has_route(x):
    x = clean_text(x).lower()

    if is_not_specified(x):
        return False

    patterns = [
        r"\bs\.c\.?\b",
        r"\bi\.m\.?\b",
        r"\bi\.d\.?\b",
        r"\bi\.v\.?\b",
        r"\bsubcutaneous\b",
        r"\bintramuscular\b",
        r"\bintradermal\b",
        r"\bintravenous\b",
        r"\bintranasal\b",
        r"\bintraperitoneal\b",
        r"\boral\b",
        r"\bp\.o\.?\b",
        r"\bi\.p\.?\b"
    ]

    return any(re.search(p, x, flags=re.I) for p in patterns)


def label_has_dose(x):
    x = clean_text(x)

    if is_not_specified(x):
        return False

    if re.search(
        r"\b\d+(\.\d+)?\s*(ng|μg|ug|µg|mcg|mg|g|ml|mL|uL|µL|μL|IU|pfu|cfu|cells|copies|mol|mM|µM|μM|uM|nM)\b",
        x,
        flags=re.I
    ):
        return True

    if re.search(r"\b(low|medium|high)\s+dose\b", x, flags=re.I):
        return True

    if re.search(r"\bdose\b", x, flags=re.I):
        return True

    return False


def infer_route_from_axis(row):
    x_role = clean_text(row.get("X_Axis_Role", "")).lower()
    y_role = clean_text(row.get("Y_Axis_Role", "")).lower()
    x_val = clean_text(row.get("X_Axis_Value", "Not specified"))
    y_val = clean_text(row.get("Y_Axis_Value", "Not specified"))

    if x_role == "route" and not is_not_specified(x_val):
        return x_val

    if y_role == "route" and not is_not_specified(y_val):
        return y_val

    if label_has_route(x_val):
        return x_val

    if label_has_route(y_val):
        return y_val

    return "Not specified"


def infer_dose_from_axis(row):
    x_role = clean_text(row.get("X_Axis_Role", "")).lower()
    y_role = clean_text(row.get("Y_Axis_Role", "")).lower()
    x_val = clean_text(row.get("X_Axis_Value", "Not specified"))
    y_val = clean_text(row.get("Y_Axis_Value", "Not specified"))

    if x_role == "dose" and not is_not_specified(x_val):
        return x_val

    if y_role == "dose" and not is_not_specified(y_val):
        return y_val

    if label_has_dose(x_val):
        return x_val

    if label_has_dose(y_val):
        return y_val

    return "Not specified"


def infer_treatment_from_axis(row):
    x_role = clean_text(row.get("X_Axis_Role", "")).lower()
    y_role = clean_text(row.get("Y_Axis_Role", "")).lower()
    x_val = clean_text(row.get("X_Axis_Value", "Not specified"))
    y_val = clean_text(row.get("Y_Axis_Value", "Not specified"))

    if x_role in ["condition", "treatment", "sample"] and not is_not_specified(x_val):
        return x_val

    if y_role in ["condition", "treatment", "sample"] and not is_not_specified(y_val):
        return y_val

    return "Not specified"


def extract_mhc_alleles(x):
    x = clean_text(x)

    if is_not_specified(x):
        return []

    patterns = [
        r"H-2[A-Za-z0-9\-\*:/]+",
        r"H2-[A-Za-z0-9\-\*:/]+",
        r"HLA-[A-Za-z0-9\-\*:/]+"
    ]

    out = []

    for p in patterns:
        out.extend(re.findall(p, x, flags=re.I))

    seen = []

    for a in out:
        a = clean_text(a)
        if a not in seen:
            seen.append(a)

    return seen


def build_article_source(row):
    source_doc = clean_text(row.get("Source_Document", "Not specified"))
    source_section = clean_text(row.get("Source_Section", "Not specified"))
    figure_unit_id = clean_text(row.get("Figure_Unit_ID", "Not specified"))
    panel_id = clean_text(row.get("Panel_ID", "Not specified"))
    stage4_file = clean_text(row.get("Stage4_Input_File", "Not specified"))

    parts = []

    for p in [source_doc, source_section, figure_unit_id, panel_id, stage4_file]:
        if not is_not_specified(p):
            parts.append(p)

    if not parts:
        return "Not specified"

    return " | ".join(parts)


def choose_reference_match(epitope_id_norm, vaccine_hint, ref_df):
    epitope_id_norm = clean_text(epitope_id_norm)
    vaccine_hint = clean_text(vaccine_hint)

    if epitope_id_norm == "Not specified":
        return None, "No Epitope_ID available for reference matching."

    subset = ref_df[
        ref_df["Epitope_ID_Normalized"].astype(str).eq(str(epitope_id_norm))
    ].copy()

    if subset.empty:
        return None, "No matching Epitope_ID in reference table."

    if len(subset) == 1:
        return subset.iloc[0].to_dict(), ""

    if not is_not_specified(vaccine_hint):
        sub2 = subset[
            subset["Vaccine_or_Antigen_Set"].astype(str).str.lower().eq(vaccine_hint.lower())
        ].copy()

        if len(sub2) == 1:
            return sub2.iloc[0].to_dict(), ""

        if len(sub2) > 1:
            return sub2.iloc[0].to_dict(), "Multiple reference rows matched Epitope_ID and vaccine; first row used."

    return subset.iloc[0].to_dict(), "Multiple reference rows matched Epitope_ID; first row used."

def build_final_elispot_rows(stage4_df, ref_df):
    rows = []
    audit_rows = []

    for idx, r in stage4_df.iterrows():
        epitope_id_norm = derive_epitope_id_from_stage4(r)

        vaccine_hint = first_nonempty(
            r.get("Legend_Vaccine_or_Antigen_Set", "Not specified"),
            r.get("Legend_Treatment", "Not specified")
        )

        ref_row, ref_note = choose_reference_match(
            epitope_id_norm=epitope_id_norm,
            vaccine_hint=vaccine_hint,
            ref_df=ref_df
        )

        if ref_row is None:
            ref_row = {}

        route_axis = infer_route_from_axis(r)
        dose_axis = infer_dose_from_axis(r)
        treatment_axis = infer_treatment_from_axis(r)

        mhc_class = clean_text(ref_row.get("MHC_Class", "Not specified"))
        mhc_alleles = extract_mhc_alleles(mhc_class)

        final_row = {
            "Epitope_ID": first_nonempty(
                ref_row.get("Epitope_ID", "Not specified"),
                epitope_id_norm
            ),
            "Epitope_Sequence": clean_text(ref_row.get("Epitope_Sequence", "Not specified")),
            "Epitope_Gene": clean_text(ref_row.get("Epitope_Gene", "Not specified")),
            "Epitope_Mutation": clean_text(ref_row.get("Epitope_Mutation", "Not specified")),
            "MHC_Class": mhc_class,
            "Vaccine_or_Antigen_Set": first_nonempty(
                ref_row.get("Vaccine_or_Antigen_Set", "Not specified"),
                r.get("Legend_Vaccine_or_Antigen_Set", "Not specified")
            ),
            "X_Axis_Value": clean_text(r.get("X_Axis_Value", "Not specified")),
            "Y_Axis_Value": clean_text(r.get("Y_Axis_Value", "Not specified")),
            "ELISpot_Value": safe_float(r.get("ELISpot_Value", np.nan)),
            "ELISpot_Unit": first_nonempty(
                r.get("ELISpot_Unit", "Not specified"),
                r.get("Legend_ELISpot_Unit", "Not specified")
            ),
            "Readout": first_nonempty(
                r.get("Readout", "Not specified"),
                r.get("Assay_Type", "Not specified")
            ),
            "Sample_or_Tissue": clean_text(r.get("Sample_or_Tissue", "Not specified")),
            "Treatment": first_nonempty(
                treatment_axis,
                r.get("Legend_Treatment", "Not specified"),
                r.get("Treatment", "Not specified")
            ),
            "Dose": first_nonempty(
                dose_axis,
                r.get("Legend_Dose", "Not specified")
            ),
            "Route": first_nonempty(
                route_axis,
                r.get("Legend_Route", "Not specified")
            ),
            "N_Mice": clean_text(r.get("Legend_N_Mice", "Not specified")),
            "Epitope_Source_Model": "Stage5_reference_table" if ref_row else "Stage4_heatmap_axis",
            "Mouse_Tumor_Model": clean_text(r.get("Legend_Mouse_Tumor_Model", "Not specified")),
            "Mouse_Strain": clean_text(r.get("Legend_Mouse_Strain", "Not specified")),
            "Mouse_MHC_I_A": mhc_alleles[0] if len(mhc_alleles) > 0 else "Not specified",
            "Mouse_MHC_I_B": mhc_alleles[1] if len(mhc_alleles) > 1 else "Not specified",
            "Mouse_MHC_I_C": mhc_alleles[2] if len(mhc_alleles) > 2 else "Not specified",
            "Article_Source": build_article_source(r),
            "Data_Source": "Stage4_heatmap_cell_records + Stage3C_legend_context + Stage5_reference_table"
        }

        for c in ELISPOT_MAIN_COLUMNS:
            if c not in final_row:
                final_row[c] = "Not specified"

        rows.append(final_row)

        audit_rows.append({
            "Stage4_Row_Index": idx,
            "Stage4_Row_ID": clean_text(r.get("Stage4_Row_ID", "Not specified")),
            "Stage4_Input_File": clean_text(r.get("Stage4_Input_File", "Not specified")),
            "Figure_Unit_ID": clean_text(r.get("Figure_Unit_ID", "Not specified")),
            "Candidate_ID": clean_text(r.get("Candidate_ID", "Not specified")),
            "Panel_ID": clean_text(r.get("Panel_ID", "Not specified")),
            "Row_Index": clean_text(r.get("Row_Index", "Not specified")),
            "Col_Index": clean_text(r.get("Col_Index", "Not specified")),
            "Derived_Epitope_ID_Normalized": epitope_id_norm,
            "Reference_Matched": ref_row != {},
            "Reference_Match_Note": clean_text(ref_note),
            "X_Axis_Role": clean_text(r.get("X_Axis_Role", "Not specified")),
            "Y_Axis_Role": clean_text(r.get("Y_Axis_Role", "Not specified")),
            "X_Axis_Value": clean_text(r.get("X_Axis_Value", "Not specified")),
            "Y_Axis_Value": clean_text(r.get("Y_Axis_Value", "Not specified")),
            "Legend_Treatment": clean_text(r.get("Legend_Treatment", "Not specified")),
            "Legend_Dose": clean_text(r.get("Legend_Dose", "Not specified")),
            "Legend_Route": clean_text(r.get("Legend_Route", "Not specified")),
            "Final_Epitope_ID": final_row["Epitope_ID"],
            "Final_Epitope_Mutation": final_row["Epitope_Mutation"],
            "Final_Treatment": final_row["Treatment"],
            "Final_Dose": final_row["Dose"],
            "Final_Route": final_row["Route"],
            "Final_Vaccine_or_Antigen_Set": final_row["Vaccine_or_Antigen_Set"]
        })

    final_df = pd.DataFrame(rows)

    for c in ELISPOT_MAIN_COLUMNS:
        if c not in final_df.columns:
            final_df[c] = "Not specified"

    final_df = final_df[ELISPOT_MAIN_COLUMNS].copy()
    audit_df = pd.DataFrame(audit_rows)

    return final_df, audit_df


def build_missing_summary(df):
    rows = []

    for c in df.columns:
        missing_count = df[c].apply(
            lambda x: is_not_specified(x) or (isinstance(x, float) and pd.isna(x))
        ).sum()

        rows.append({
            "Column": c,
            "Missing_or_Not_Specified_Count": int(missing_count),
            "Total_Count": int(len(df)),
            "Missing_Rate": float(missing_count / len(df)) if len(df) else 0.0
        })

    return pd.DataFrame(rows)


def build_duplicate_check(df):
    key_cols = [
        "Epitope_ID",
        "X_Axis_Value",
        "Y_Axis_Value",
        "ELISpot_Value",
        "Article_Source"
    ]

    existing = [c for c in key_cols if c in df.columns]

    if not existing or df.empty:
        return pd.DataFrame()

    dup_mask = df.duplicated(subset=existing, keep=False)

    return df[dup_mask].copy()


def make_qc_summary(final_df, stage4_df, ref_df, audit_df, failed_df):
    missing_summary = build_missing_summary(final_df)
    duplicate_df = build_duplicate_check(final_df)

    if not audit_df.empty and "Reference_Matched" in audit_df.columns:
        matched_count = int(audit_df["Reference_Matched"].astype(bool).sum())
        unmatched_count = int((~audit_df["Reference_Matched"].astype(bool)).sum())
    else:
        matched_count = 0
        unmatched_count = 0

    qc = {
        "stage4_input_records": int(len(stage4_df)),
        "reference_table_rows": int(len(ref_df)),
        "final_elispot_rows": int(len(final_df)),
        "unique_epitope_ids_final": int(final_df["Epitope_ID"].nunique(dropna=False)) if not final_df.empty else 0,
        "reference_matched_rows": matched_count,
        "reference_unmatched_rows": unmatched_count,
        "failed_rows": int(len(failed_df)),
        "duplicate_rows_by_main_key": int(len(duplicate_df)),
        "epitope_mutation_not_specified_count": int(final_df["Epitope_Mutation"].apply(is_not_specified).sum()) if not final_df.empty else 0,
        "records_by_article_source": final_df["Article_Source"].value_counts(dropna=False).to_dict()
            if not final_df.empty and "Article_Source" in final_df.columns else {},
        "records_by_vaccine_or_antigen_set": final_df["Vaccine_or_Antigen_Set"].value_counts(dropna=False).to_dict()
            if not final_df.empty and "Vaccine_or_Antigen_Set" in final_df.columns else {},
        "missing_by_column": {
            r["Column"]: int(r["Missing_or_Not_Specified_Count"])
            for _, r in missing_summary.iterrows()
        }
    }

    return qc, missing_summary, duplicate_df


def find_reference_table_auto(root="stage5_reference_table_outputs_final"):
    root = Path(root)

    if not root.exists():
        return None

    files = list(root.rglob("*.xlsx")) + list(root.rglob("*.csv"))

    if not files:
        return None

    scored = []

    for f in files:
        name = f.name.lower()
        score = 0

        if "table" in name:
            score += 3

        if "s1" in name:
            score += 3

        if "neo" in name or "epitope" in name:
            score += 4

        if "final" in name:
            score += 5

        if "reference" in name:
            score += 2

        if "failed" in name or "qc" in name or "summary" in name:
            score -= 10

        scored.append((score, f))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    if scored and scored[0][0] > 0:
        return str(scored[0][1])

    return str(files[0])


def run_stage6_final_elispot_merge(
    stage4_cell_records_paths,
    reference_table_path=None,
    out_root="stage6_final_elispot_outputs"
):
    out_root = Path(out_root)
    ensure_dir(out_root)

    if reference_table_path is None:
        reference_table_path = find_reference_table_auto()

    if reference_table_path is None:
        raise FileNotFoundError("Reference table not found. Set reference_table_path manually.")

    stage4_raw = read_many_stage4_records(stage4_cell_records_paths)

    ref_raw = read_table_robust(
        reference_table_path,
        preferred_sheets=[
            "Table_S1_Neoepitopes",
            "Final_Reference_Table",
            "Reference_Table",
            "Table_S1",
            "Neoepitopes"
        ]
    )

    stage4_df = standardize_stage4_records(stage4_raw)
    ref_df = standardize_reference_table(ref_raw)

    final_df, audit_df = build_final_elispot_rows(stage4_df, ref_df)

    failed_df = audit_df[
        audit_df["Derived_Epitope_ID_Normalized"].eq("Not specified")
        | audit_df["Reference_Matched"].eq(False)
    ].copy()

    qc, missing_summary_df, duplicate_df = make_qc_summary(
        final_df=final_df,
        stage4_df=stage4_df,
        ref_df=ref_df,
        audit_df=audit_df,
        failed_df=failed_df
    )

    final_csv = out_root / "ELISpot_Data.csv"
    final_xlsx = out_root / "ELISpot_Data.xlsx"

    ref_csv = out_root / "Table_S1_Neoepitopes_Standardized.csv"
    ref_xlsx = out_root / "Table_S1_Neoepitopes_Standardized.xlsx"

    audit_csv = out_root / "Stage6_Merge_Audit.csv"
    audit_xlsx = out_root / "Stage6_Merge_Audit.xlsx"

    missing_csv = out_root / "Stage6_Missing_Field_Summary.csv"
    duplicate_csv = out_root / "Stage6_Duplicate_Check.csv"
    failed_csv = out_root / "Stage6_Unmatched_or_Review_Items.csv"

    combined_xlsx = out_root / "Stage6_Final_ELISpot_Output.xlsx"
    qc_json = out_root / "Stage6_QC_Summary.json"

    final_df.to_csv(final_csv, index=False, encoding="utf-8-sig")
    final_df.to_excel(final_xlsx, index=False)

    ref_df.to_csv(ref_csv, index=False, encoding="utf-8-sig")
    ref_df.to_excel(ref_xlsx, index=False)

    audit_df.to_csv(audit_csv, index=False, encoding="utf-8-sig")
    audit_df.to_excel(audit_xlsx, index=False)

    missing_summary_df.to_csv(missing_csv, index=False, encoding="utf-8-sig")
    duplicate_df.to_csv(duplicate_csv, index=False, encoding="utf-8-sig")
    failed_df.to_csv(failed_csv, index=False, encoding="utf-8-sig")

    save_json(qc, qc_json)

    with pd.ExcelWriter(combined_xlsx, engine="openpyxl") as writer:
        final_df.to_excel(writer, sheet_name="ELISpot_Data", index=False)
        ref_df.to_excel(writer, sheet_name="Table_S1_Neoepitopes", index=False)
        audit_df.to_excel(writer, sheet_name="Stage6_Merge_Audit", index=False)
        missing_summary_df.to_excel(writer, sheet_name="Missing_Field_Summary", index=False)
        duplicate_df.to_excel(writer, sheet_name="Duplicate_Check", index=False)
        failed_df.to_excel(writer, sheet_name="Unmatched_or_Review", index=False)

        pd.DataFrame([
            {
                "Metric": k,
                "Value": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
            }
            for k, v in qc.items()
        ]).to_excel(writer, sheet_name="QC_Summary", index=False)

    print("=" * 120)
    print("Stage 6 complete.")
    print("Stage 4 inputs:", stage4_cell_records_paths)
    print("Reference input:", reference_table_path)
    print("Final ELISpot rows:", len(final_df))
    print("Reference rows:", len(ref_df))
    print("Reference matched rows:", qc["reference_matched_rows"])
    print("Reference unmatched rows:", qc["reference_unmatched_rows"])
    print("Epitope_Mutation Not specified:", qc["epitope_mutation_not_specified_count"])
    print("Duplicate rows by main key:", qc["duplicate_rows_by_main_key"])
    print("Final Excel:", combined_xlsx)
    print("ELISpot_Data:", final_xlsx)
    print("QC:", qc_json)

    display(final_df.head(50))
    display(ref_df.head(20))
    display(missing_summary_df)
    display(failed_df.head(50))

    return {
        "elispot_data_df": final_df,
        "reference_df": ref_df,
        "audit_df": audit_df,
        "missing_summary_df": missing_summary_df,
        "duplicate_df": duplicate_df,
        "failed_df": failed_df,
        "qc": qc,
        "elispot_data_csv": final_csv,
        "elispot_data_xlsx": final_xlsx,
        "reference_csv": ref_csv,
        "reference_xlsx": ref_xlsx,
        "audit_csv": audit_csv,
        "audit_xlsx": audit_xlsx,
        "missing_csv": missing_csv,
        "duplicate_csv": duplicate_csv,
        "failed_csv": failed_csv,
        "combined_xlsx": combined_xlsx,
        "qc_json": qc_json
    }

if stage4_source_data_result is not None and not stage4_source_data_result["records_df"].empty:
    STAGE4_CELL_RECORDS_PATHS = [str(stage4_source_data_result["records_xlsx"])]
    print("Stage 6 uses publisher Source Data values.")
else:
    STAGE4_CELL_RECORDS_PATHS = [str(stage4_result["cell_records_xlsx"])]
    print("Stage 6 uses values extracted from the figure.")

REFERENCE_TABLE_PATH = str(stage5_result["final_xlsx"])
stage6_result = run_stage6_final_elispot_merge(
    stage4_cell_records_paths=STAGE4_CELL_RECORDS_PATHS,
    reference_table_path=REFERENCE_TABLE_PATH,
    out_root=str(RUN_ROOT / "stage6_final"),
)


## Multi-paper 10-panel heatmap development benchmark

This section evaluates the generalized Stage 4 logic on 10 heatmap panels from 7 papers. The benchmark contains dense printed-number matrices, sparse printed-number matrices, vertically nested blocks, categorical color-only matrices, and continuous color-only matrices.

The routing policy is evidence based. Every numeric-capable panel first receives a coordinate-locked indexed pass. A high observed printed-number fraction selects an additional fixed-shape dense read. Disagreement between those views triggers a local reread of at most seven tool-defined cells. Color-only panels abstain and request publisher source data; the pipeline never converts color intensity into numeric values.

This is a multi-paper development benchmark.

In [ ]:
# Deterministic structure and encoding check for all 10 panels.
from pathlib import Path
import json
import pandas as pd
from heatmap_benchmark_engine import analyze_heatmap_structure

BENCHMARK_PIPELINE_VERSION = "generic_heatmap_benchmark_v1.0"
BENCHMARK_ROOT = Path(PROJECT_ROOT) / "generic_heatmap_benchmark_10"
BENCHMARK_MANIFEST = pd.read_csv(BENCHMARK_ROOT / "benchmark_manifest.csv")

benchmark_structure_rows = []
for item in BENCHMARK_MANIFEST.itertuples(index=False):
    result = analyze_heatmap_structure(item.benchmark_image_path, item.structure_type)
    benchmark_structure_rows.append({
        "benchmark_id": item.benchmark_id,
        "expected_rows": int(item.expected_rows),
        "expected_columns": int(item.expected_columns),
        "detected_rows": result["rows"],
        "detected_columns": result["columns"],
        "shape_exact_match": result["rows"] == int(item.expected_rows) and result["columns"] == int(item.expected_columns),
        "detected_encoding": result["encoding"].get("classification"),
        "glyph_evidence_rate": result["encoding"].get("glyph_evidence_rate"),
        "numeric_values_inferred_from_color": False,
        "structure_diagnostics": json.dumps(result["structure"], ensure_ascii=False),
    })

benchmark_structure = pd.DataFrame(benchmark_structure_rows)
benchmark_structure.to_csv(BENCHMARK_ROOT / "structure_detection_results.csv", index=False, encoding="utf-8-sig")
assert len(benchmark_structure) == 10
assert benchmark_structure["shape_exact_match"].all(), "Review shape failures before Qwen extraction."
display(benchmark_structure[["benchmark_id", "expected_rows", "expected_columns", "detected_rows", "detected_columns", "shape_exact_match", "detected_encoding"]])

In [ ]:
# API execution is opt-in because completed audited responses are already cached.
# Set True only when intentionally rerunning the full benchmark with the Qwen key
# available in this Jupyter kernel.
RUN_QWEN_BENCHMARK = False

if RUN_QWEN_BENCHMARK:
    if not os.getenv("DASHSCOPE_API_KEY"):
        raise RuntimeError("DASHSCOPE_API_KEY is unavailable in this Jupyter kernel")
    exec((Path(PROJECT_ROOT) / "run_heatmap_benchmark_qwen.py").read_text(encoding="utf-8"), globals())
    exec((Path(PROJECT_ROOT) / "run_dense_matrix_refinement.py").read_text(encoding="utf-8"), globals())
    exec((Path(PROJECT_ROOT) / "run_disagreement_refinement.py").read_text(encoding="utf-8"), globals())
else:
    print("Using cached benchmark predictions; no Qwen API calls were made in this cell.")

# Rebuild ground-truth comparisons, tables, figures, and the Results draft.
exec((Path(PROJECT_ROOT) / "build_benchmark_results.py").read_text(encoding="utf-8"), globals())

In [ ]:
benchmark_overall = json.loads((BENCHMARK_ROOT / "benchmark_overall_results.json").read_text(encoding="utf-8"))
benchmark_panels = pd.read_csv(BENCHMARK_ROOT / "benchmark_panel_results.csv")
print(json.dumps(benchmark_overall, ensure_ascii=False, indent=2))
display(benchmark_panels[[
    "benchmark_id", "paper_id", "figure", "panel", "expected_rows", "expected_columns",
    "shape_exact_match", "extraction_outcome", "evaluated_cells", "printed_numeric_cells",
    "numeric_value_accuracy", "correct_abstention"
]])
print("Results draft:", BENCHMARK_ROOT / "results_text_draft.md")
print("Ground truth:", BENCHMARK_ROOT / "benchmark_numeric_ground_truth.csv")
print("Cell validation:", BENCHMARK_ROOT / "benchmark_cell_level_validation.csv")

## Stage 5–6：跨论文补充材料重建（复用 Stage 1–4 缓存）
本节只读取已经保存的 10-panel Stage 4 结果和出版方补充表。不会重新运行 Stage 1–4，不调用 Qwen，也不会根据热图颜色制造数值。颜色图的真实补充表数值保存在 `supplementary_source_value`，与 `stage4_predicted_value` 分开。


In [ ]:
from pathlib import Path
from stage56_cross_paper_pipeline import run_stage56_cross_paper

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'stage56_cross_paper_pipeline.py').exists():
    PROJECT_ROOT = Path(r'C:\Users\lenovo\Desktop\my_langchain_project')

STAGE56_REUSE_STAGE14_CACHE = True
STAGE56_ALLOW_QWEN_API_CALLS = False
stage56_run = run_stage56_cross_paper(PROJECT_ROOT)
stage56_run
